# Generación definitiva del dataset contextual unificado

Este notebook genera el **dataset contextual occurrence-level definitivo** que se utilizará tanto para:

1. **Exp2_prototipos**: cada palabra se agrega posteriormente como una media raw de sus 100 ocurrencias.
2. **Extensión contextual E6b/E6c**: se conservan las 100 ocurrencias individuales de cada palabra.

## Diseño cerrado

Sea \(C\) el pool común:

\[
C = \text{WikiText-103} \cup \text{Wikimedia/Wikipedia 20231101.en}
\]

Una palabra es elegible si pasa el contrato léxico y posee al menos 100 contextos únicos en \(C\). Para **todas** las palabras elegibles se seleccionan exactamente:

\[
n_w = 100
\]

Las ocho `CategoryWords` solo tienen una condición especial: deben estar presentes en el vocabulario final. No reciben más observaciones ni una política de muestreo distinta.

La selección usa una prioridad BLAKE2b determinista sobre `(word, context_window)` y conserva las 100 prioridades menores por palabra. Esto evita sesgo hacia las primeras ocurrencias del corpus.

## Representación contextual

- `bert-base-uncased`;
- promedio de las últimas 4 capas Transformer;
- 768 dimensiones;
- embeddings raw, sin L2;
- ventana ±40 tokens;
- `MAX_SEQ_LEN=128`;
- stopwords conservadas dentro del contexto, pero excluidas como palabras representadas;
- `references`, `external` y `links` excluidos como artefactos estructurales.

## Salidas

Todos los artefactos se guardan en:

`C:\Users\jisaa\Workspace\Tesis\Exp2\Results\contextual_dataset`

El Parquet final queda ordenado por `word`, con row groups de 1.000 filas (10 palabras × 100 ocurrencias), y se genera `word_row_group_index.csv` para cargas rápidas de E6.

In [1]:
# ============================================================
# 00. Dependencias
# ============================================================

import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "torch": "torch",
    "datasets": "datasets",
    "transformers": "transformers",
    "pyarrow": "pyarrow",
    "tqdm": "tqdm",
}

missing = [
    pip_name
    for module_name, pip_name in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(module_name) is None
]

if missing:
    raise RuntimeError(
        "Faltan dependencias del entorno de tesis: "
        f"{missing}. Instálalas en el mismo venv antes de continuar."
    )

# DuckDB se usa para ordenar externamente ~10M candidatos sin cargarlos en RAM.
if importlib.util.find_spec("duckdb") is None:
    print("duckdb no está instalado. Instalando dependencia liviana...")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "duckdb"]
    )

print("Dependencias OK.")

duckdb no está instalado. Instalando dependencia liviana...
Dependencias OK.


In [13]:
# ============================================================
# 01. Imports y configuración CPU/GPU
# ============================================================

import gc
import hashlib
import heapq
import json
import math
import os
import random
import re
import shutil
import time
import unicodedata

from collections import Counter
from pathlib import Path
from typing import List

import duckdb
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import torch

from datasets import load_dataset
from IPython.display import display
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

CPU_LOGICAL_CORES = os.cpu_count() or 8
CPU_THREADS_DATASET = max(1, min(8, CPU_LOGICAL_CORES - 2))

os.environ["OMP_NUM_THREADS"] = str(CPU_THREADS_DATASET)
os.environ["MKL_NUM_THREADS"] = str(CPU_THREADS_DATASET)
os.environ["OPENBLAS_NUM_THREADS"] = str(CPU_THREADS_DATASET)
os.environ["NUMEXPR_NUM_THREADS"] = str(CPU_THREADS_DATASET)
os.environ["RAYON_NUM_THREADS"] = str(CPU_THREADS_DATASET)
os.environ["TOKENIZERS_PARALLELISM"] = "true"

torch.set_num_threads(CPU_THREADS_DATASET)

try:
    torch.set_num_interop_threads(max(1, min(4, CPU_THREADS_DATASET)))
except RuntimeError:
    pass

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if DEVICE.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass

print("CPU_LOGICAL_CORES:", CPU_LOGICAL_CORES)
print("CPU_THREADS_DATASET:", CPU_THREADS_DATASET)
print("DEVICE:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "VRAM total [GB]:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
    )

CPU_LOGICAL_CORES: 20
CPU_THREADS_DATASET: 8
DEVICE: cuda
GPU: NVIDIA GeForce RTX 5060 Laptop GPU
VRAM total [GB]: 7.96


In [14]:
# ============================================================
# 02. Rutas y configuración científica definitiva
# ============================================================

PROJECT_ROOT = Path(r"C:\Users\jisaa\Workspace\Tesis")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

OUT_DIR = PROJECT_ROOT / "Exp2" / "Results" / "contextual_dataset"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Entrada congelada por corpus_audit.ipynb
CORPUS_AUDIT_ROOT = (
    PROJECT_ROOT
    / "Exp2"
    / "Results"
    / "ResultsExp2_prototipos"
    / "00_corpus_audit"
)

AUDITED_ELIGIBLE_VOCAB_PATH = (
    CORPUS_AUDIT_ROOT
    / "common_pool_eligible_vocabulary_min100.csv"
)

# Salidas
FINAL_ELIGIBLE_VOCAB_PATH = OUT_DIR / "eligible_vocabulary_min100.csv"
SELECTED_HASHES_PATH = OUT_DIR / "selected_context_hashes_top100.parquet"
SELECTION_REPORT_PATH = OUT_DIR / "selection_report.json"

UNSORTED_CANDIDATES_PATH = (
    OUT_DIR
    / "_contextual_occurrence_candidates_unsorted.parquet"
)

CANDIDATES_PATH = (
    OUT_DIR
    / "contextual_occurrence_candidates_sorted.parquet"
)

DUCKDB_TEMP_DIR = OUT_DIR / "_duckdb_tmp"

OUT_PARQUET_PATH = (
    OUT_DIR
    / "contextual_occurrences_definitive.parquet"
)

OUT_PARQUET_PARTIAL_PATH = (
    OUT_DIR
    / "_contextual_occurrences_definitive.partial.parquet"
)

META_PATH = OUT_DIR / "contextual_dataset_meta.json"
WORD_COUNTS_PATH = OUT_DIR / "word_counts.csv"
CATEGORY_COVERAGE_PATH = OUT_DIR / "category_words_coverage.csv"
EXP2_METADATA_COVERAGE_PATH = OUT_DIR / "exp2_metadata_words_coverage.csv"
VALIDATION_REPORT_PATH = OUT_DIR / "validation_report.json"
SAMPLE_CONTEXTS_PATH = OUT_DIR / "sample_contexts_by_category_word.csv"
ROW_GROUP_INDEX_PATH = OUT_DIR / "word_row_group_index.csv"
GENERATION_CONFIG_PATH = OUT_DIR / "generation_config.json"

# Diseño científico
MODEL_NAME = "bert-base-uncased"

USE_LAYER_AVG = True
SELECTED_LAYER_IDS = [-4, -3, -2, -1]

L2_NORMALIZE_EMBEDDINGS = False

MAX_SEQ_LEN = 128
WINDOW_TOKENS_EACH_SIDE = 40

MIN_LINE_WORDS = 4
MIN_ROW_WORD_LEN = 3

CONTEXT_KEEP_STOPWORDS = True
REMOVE_STOPWORDS_FROM_CONTEXT_WINDOW = False

OCCURRENCES_PER_WORD = 100

# Valor ya utilizado de forma segura en el generador anterior.
BATCH_SIZE_OCCURRENCES = 64

# 1000 filas = 10 palabras completas.
FINAL_ROW_GROUP_ROWS = 1000

# Row groups del archivo intermedio ordenado.
CANDIDATE_SORT_ROW_GROUP_ROWS = 100_000

CONTEXTUAL_DATASET_SCHEMA_VERSION = (
    "ctx_v3_commonpool_equal100_rawbert"
)

CONFIG_NAME = "ctx_v3_bert_base_commonpool_equal100"

RESET_OUTPUTS = False

# Reutilización de etapas ya terminadas si el kernel se reinicia.
REUSE_SELECTION_MANIFEST = True
REUSE_SORTED_CANDIDATES = True

print("OUT_DIR:", OUT_DIR)
print("AUDITED_ELIGIBLE_VOCAB_PATH:", AUDITED_ELIGIBLE_VOCAB_PATH)
print("OCCURRENCES_PER_WORD:", OCCURRENCES_PER_WORD)
print("MODEL_NAME:", MODEL_NAME)
print("BATCH_SIZE_OCCURRENCES:", BATCH_SIZE_OCCURRENCES)
print("FINAL_ROW_GROUP_ROWS:", FINAL_ROW_GROUP_ROWS)
print("RESET_OUTPUTS:", RESET_OUTPUTS)

OUT_DIR: C:\Users\jisaa\Workspace\Tesis\Exp2\Results\contextual_dataset
AUDITED_ELIGIBLE_VOCAB_PATH: C:\Users\jisaa\Workspace\Tesis\Exp2\Results\ResultsExp2_prototipos\00_corpus_audit\common_pool_eligible_vocabulary_min100.csv
OCCURRENCES_PER_WORD: 100
MODEL_NAME: bert-base-uncased
BATCH_SIZE_OCCURRENCES: 64
FINAL_ROW_GROUP_ROWS: 1000
RESET_OUTPUTS: False


In [15]:
# ============================================================
# 02.b Reset controlado de salidas
# ============================================================

GENERATED_PATHS = [
    FINAL_ELIGIBLE_VOCAB_PATH,
    SELECTED_HASHES_PATH,
    SELECTION_REPORT_PATH,
    UNSORTED_CANDIDATES_PATH,
    CANDIDATES_PATH,
    OUT_PARQUET_PATH,
    OUT_PARQUET_PARTIAL_PATH,
    META_PATH,
    WORD_COUNTS_PATH,
    CATEGORY_COVERAGE_PATH,
    EXP2_METADATA_COVERAGE_PATH,
    VALIDATION_REPORT_PATH,
    SAMPLE_CONTEXTS_PATH,
    ROW_GROUP_INDEX_PATH,
    GENERATION_CONFIG_PATH,
]

if RESET_OUTPUTS:

    print("RESET_OUTPUTS=True: eliminando artefactos previos conocidos.")

    for path in GENERATED_PATHS:
        path = Path(path)

        if path.exists():
            if path.is_dir():
                shutil.rmtree(path)
            else:
                path.unlink()

    if DUCKDB_TEMP_DIR.exists():
        shutil.rmtree(DUCKDB_TEMP_DIR)

else:
    print(
        "RESET_OUTPUTS=False: no se eliminaron artefactos previos. "
        "Las celdas reutilizarán artefactos válidos cuando corresponda."
    )

RESET_OUTPUTS=False: no se eliminaron artefactos previos. Las celdas reutilizarán artefactos válidos cuando corresponda.


In [16]:
# ============================================================
# 03. Helpers de persistencia y bloques copiables
# ============================================================

def json_safe(value):

    if value is None:
        return None

    if isinstance(value, Path):
        return str(value)

    if isinstance(value, (str, bool, int)):
        return value

    if isinstance(value, np.integer):
        return int(value)

    if isinstance(value, np.bool_):
        return bool(value)

    if isinstance(value, (float, np.floating)):
        value = float(value)

        if np.isnan(value):
            return None

        if np.isposinf(value):
            return "Infinity"

        if np.isneginf(value):
            return "-Infinity"

        return value

    if isinstance(value, pd.DataFrame):
        return [
            json_safe(row)
            for row in value.to_dict(orient="records")
        ]

    if isinstance(value, pd.Series):
        return json_safe(value.to_dict())

    if isinstance(value, dict):
        return {
            str(key): json_safe(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple, set, frozenset)):
        return [json_safe(item) for item in value]

    if isinstance(value, np.ndarray):
        return json_safe(value.tolist())

    return str(value)


def save_json(payload, path):

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with open(path, "w", encoding="utf-8") as handle:
        json.dump(
            json_safe(payload),
            handle,
            ensure_ascii=False,
            indent=2,
            sort_keys=True,
        )

    return path


def print_copy_block(title, payload):

    print("\n" + "=" * 90)
    print(f"PARA COPIAR Y PEGAR — {title}")
    print("=" * 90)

    print(
        json.dumps(
            json_safe(payload),
            ensure_ascii=False,
            indent=2,
            sort_keys=True,
        )
    )

    print("=" * 90)
    print(f"FIN — {title}")
    print("=" * 90)

In [17]:
# ============================================================
# 04. Metadata Exp2 heredada
# ============================================================

try:
    from preprocessing import CategoryWords, TargetWords

    category_words = [
        str(word).lower()
        for word in CategoryWords.load()
    ]

    target_extra_words = [
        str(word).lower()
        for word in TargetWords.load()
    ]

except Exception as exc:

    print(
        "No se pudo importar CategoryWords/TargetWords desde preprocessing.py. "
        "Usando fallback."
    )
    print("Error original:", repr(exc))

    category_words = [
        "apple",
        "bicycle",
        "bread",
        "canoe",
        "chair",
        "jellyfish",
        "screwdriver",
        "skunk",
    ]

    target_extra_words = [
        "ant",
        "bus",
        "cheese",
        "crab",
        "curtain",
        "frog",
        "olive",
        "parachute",
        "peanut",
        "pear",
        "raccoon",
        "raft",
        "scooter",
        "sling",
        "stool",
        "wrench",
    ]

target_words = sorted(set(category_words + target_extra_words))

category_set = set(category_words)
target_set = set(target_words)
target_extra_set = set(target_extra_words)

target_roles = {}

for word in target_words:

    if word in category_set and word in target_extra_set:
        target_roles[word] = "category_and_target"

    elif word in category_set:
        target_roles[word] = "category"

    else:
        target_roles[word] = "target"

EXP2_CLASS_LABELS = [
    "apple",
    "bicycle",
    "bread",
    "canoe",
    "chair",
    "jellyfish",
    "screwdriver",
    "skunk",
]

EXP2_CLASS_GROUPS = {
    "apple": ["apple", "olive", "pear"],
    "bicycle": ["bicycle", "parachute", "scooter"],
    "bread": ["bread", "cheese", "peanut"],
    "canoe": ["canoe", "bus", "raft"],
    "chair": ["chair", "curtain", "stool"],
    "jellyfish": ["jellyfish", "ant", "crab"],
    "screwdriver": ["screwdriver", "sling", "wrench"],
    "skunk": ["skunk", "frog", "raccoon"],
}

WORD_TO_TRUE_CLASS = {}
WORD_TO_CLASS_LABEL = {}

for class_id, class_label in enumerate(EXP2_CLASS_LABELS):

    for word in EXP2_CLASS_GROUPS[class_label]:
        WORD_TO_TRUE_CLASS[word] = int(class_id)
        WORD_TO_CLASS_LABEL[word] = str(class_label)


def get_exp2_class_metadata(word):

    word = str(word).lower()

    return {
        "true_class": int(
            WORD_TO_TRUE_CLASS.get(word, -1)
        ),
        "class_label": str(
            WORD_TO_CLASS_LABEL.get(word, "filler")
        ),
        "is_exp2_word": bool(
            word in WORD_TO_TRUE_CLASS
        ),
    }


if len(category_words) != 8:
    raise RuntimeError(
        f"Se esperaban 8 CategoryWords y se obtuvieron {len(category_words)}."
    )

print("CategoryWords:", category_words)
print("TargetWords con metadata Exp2:", len(target_words))

CategoryWords: ['apple', 'bicycle', 'bread', 'canoe', 'chair', 'jellyfish', 'screwdriver', 'skunk']
TargetWords con metadata Exp2: 24


In [30]:
# ============================================================
# 05. Contrato léxico y definición de contexto
# ============================================================

STRICT_WORD_PATTERN = re.compile(r"[a-z]+")

try:
    from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
    STOPWORDS = set(ENGLISH_STOP_WORDS)

except Exception:

    STOPWORDS = {
        "a", "an", "the", "and", "or", "but", "if", "while",
        "of", "to", "in", "on", "for", "with", "as", "by",
        "at", "from", "is", "are", "was", "were", "be", "been",
        "being", "this", "that", "these", "those", "it", "its",
        "he", "she", "they", "them", "we", "you", "i", "me",
        "my", "your", "his", "her", "their",
    }

CUSTOM_BAD_WORDS = {
    "nbsp",
    "amp",
    "lt",
    "gt",
    "ref",
    "br",
    "html",
    "http",
    "https",
    "www",
    "com",
    "org",
    "net",
    "url",
    "jpg",
    "jpeg",
    "png",
    "gif",
    "svg",
    "px",
    "thumb",
    "thumbnail",
    "image",
    "file",
    "align",
    "isbn",
    "utc",
    "doi",
    "redirect",
    "category",
    "template",
    "cite",
    "citation",
    "edit",

    # Artefactos estructurales detectados en la auditoría V100.
    "references",
    "external",
    "links",
}

BAD_WORDS = set(STOPWORDS) | set(CUSTOM_BAD_WORDS)


def normalize_text(text):

    text = unicodedata.normalize("NFKC", str(text))
    text = text.lower()

    text = (
        text
        .replace(" @-@ ", "-")
        .replace(" @.@ ", ".")
        .replace(" @,@ ", ",")
    )

    text = re.sub(r"\s+", " ", text).strip()

    return text


def strict_tokenize(text):

    return STRICT_WORD_PATTERN.findall(
        normalize_text(text)
    )


def is_heading_line(text):

    return str(text).lstrip().startswith("=")


def is_valid_row_word(word):

    word = str(word).lower()

    if len(word) < MIN_ROW_WORD_LEN:
        return False

    if not re.fullmatch(r"[a-z]+", word):
        return False

    if word in BAD_WORDS:
        return False

    return True


def stable_doc_split(
    doc_key,
    train_p=0.70,
    val_p=0.15,
):

    h = hashlib.md5(
        str(doc_key).encode("utf-8")
    ).hexdigest()

    u = int(h[:8], 16) / 0xFFFFFFFF

    if u < train_p:
        return "train"

    if u < train_p + val_p:
        return "validation"

    return "test"


def build_centered_window(
    tokens,
    word_pos,
):

    left = max(
        0,
        int(word_pos) - WINDOW_TOKENS_EACH_SIDE,
    )

    right = min(
        len(tokens),
        int(word_pos) + WINDOW_TOKENS_EACH_SIDE + 1,
    )

    window_tokens = tokens[left:right]
    word_pos_window = int(word_pos) - left

    if REMOVE_STOPWORDS_FROM_CONTEXT_WINDOW:

        cleaned = [
            token
            for token in window_tokens
            if (
                re.fullmatch(r"[a-z]+", token)
                and token not in BAD_WORDS
            )
        ]

        word = tokens[int(word_pos)]

        positions = [
            idx
            for idx, token in enumerate(cleaned)
            if token == word
        ]

        if not positions:
            return [], -1

        window_tokens = cleaned

        word_pos_window = min(
            positions,
            key=lambda idx:
                abs(idx - len(window_tokens) // 2),
        )

    return window_tokens, word_pos_window


def short_context(
    window_tokens,
    word_pos_window,
    radius=10,
):

    left = " ".join(
        window_tokens[
            max(0, word_pos_window - radius):
            word_pos_window
        ]
    )

    right = " ".join(
        window_tokens[
            word_pos_window + 1:
            word_pos_window + 1 + radius
        ]
    )

    return left, right


def context_priority_int(
    word,
    context_window,
):

    payload = (
        str(word)
        + "\0"
        + str(context_window)
    ).encode("utf-8")

    digest = hashlib.blake2b(
        payload,
        digest_size=16,
        person=b"ctx100-v1",
    ).digest()

    return int.from_bytes(
        digest,
        byteorder="big",
        signed=False,
    )


def priority_int_to_bytes(value):

    return int(value).to_bytes(
        16,
        byteorder="big",
        signed=False,
    )


print("Stopwords:", len(STOPWORDS))
print("CUSTOM_BAD_WORDS:", len(CUSTOM_BAD_WORDS))

Stopwords: 318
CUSTOM_BAD_WORDS: 37


In [31]:
# ============================================================
# 06. Cargar y congelar el vocabulario elegible definitivo
# ============================================================
#
# corpus_audit.ipynb ya determinó las palabras con >=100
# contextos únicos en el pool común.
#
# Aquí solo se aplica el contrato léxico definitivo,
# incluyendo references/external/links.
# ============================================================

if not AUDITED_ELIGIBLE_VOCAB_PATH.exists():

    raise FileNotFoundError(
        "No existe el vocabulario auditado requerido:\n"
        f"{AUDITED_ELIGIBLE_VOCAB_PATH}\n\n"
        "Ejecuta primero corpus_audit.ipynb completo."
    )

audited_vocab_df = pd.read_csv(
    AUDITED_ELIGIBLE_VOCAB_PATH,
    keep_default_na=False,
    na_filter=False,
)

if audited_vocab_df["word"].isna().any():
    raise RuntimeError(
        "La columna word contiene valores nulos después de la carga."
    )

if (
    audited_vocab_df["word"]
    .astype(str)
    .str.strip()
    .eq("")
    .any()
):
    raise RuntimeError(
        "La columna word contiene cadenas vacías."
    )

if audited_vocab_df["word"].isna().any():
    raise RuntimeError(
        "La columna word contiene valores nulos después de la carga."
    )

if (
    audited_vocab_df["word"]
    .astype(str)
    .str.strip()
    .eq("")
    .any()
):
    raise RuntimeError(
        "La columna word contiene cadenas vacías."
    )
if "word" not in audited_vocab_df.columns:
    raise RuntimeError(
        "El CSV auditado no contiene columna 'word'."
    )

audited_vocab_df["word"] = (
    audited_vocab_df["word"]
    .astype(str)
    .str.lower()
    .str.strip()
)

audited_vocab_df = (
    audited_vocab_df
    .drop_duplicates(subset=["word"])
    .reset_index(drop=True)
)

lexical_keep_mask = (
    audited_vocab_df["word"]
    .map(is_valid_row_word)
)

excluded_from_audit_df = (
    audited_vocab_df.loc[
        ~lexical_keep_mask
    ]
    .copy()
    .reset_index(drop=True)
)

eligible_vocab_df = (
    audited_vocab_df.loc[
        lexical_keep_mask
    ]
    .copy()
    .sort_values("word")
    .reset_index(drop=True)
)

eligible_words = (
    eligible_vocab_df["word"]
    .tolist()
)

eligible_word_set = set(
    eligible_words
)

missing_category_words = sorted(
    set(category_words) - eligible_word_set
)

if missing_category_words:
    raise RuntimeError(
        "Las siguientes CategoryWords no pertenecen al V100 definitivo: "
        f"{missing_category_words}"
    )

if len(eligible_words) != len(eligible_word_set):
    raise RuntimeError(
        "El vocabulario elegible contiene duplicados."
    )

if any(
    not is_valid_row_word(word)
    for word in eligible_words
):
    raise RuntimeError(
        "Persisten palabras que violan el contrato léxico."
    )

eligible_vocab_df.to_csv(
    FINAL_ELIGIBLE_VOCAB_PATH,
    index=False,
    encoding="utf-8",
)

vocabulary_summary = {
    "audit_source": str(
        AUDITED_ELIGIBLE_VOCAB_PATH
    ),
    "n_words_audited_ge100": int(
        len(audited_vocab_df)
    ),
    "n_words_excluded_by_final_lexical_contract": int(
        len(excluded_from_audit_df)
    ),
    "excluded_words": excluded_from_audit_df[
        "word"
    ].tolist(),
    "n_words_final": int(
        len(eligible_words)
    ),
    "occurrences_per_word": int(
        OCCURRENCES_PER_WORD
    ),
    "expected_total_rows": int(
        len(eligible_words)
        * OCCURRENCES_PER_WORD
    ),
    "all_8_category_words_present": True,
    "category_words": list(category_words),
}

print(
    "Palabras auditadas >=100:",
    len(audited_vocab_df),
)

print(
    "Excluidas por contrato final:",
    excluded_from_audit_df["word"].tolist(),
)

print(
    "Vocabulario final:",
    len(eligible_words),
)

print(
    "Filas esperadas:",
    vocabulary_summary["expected_total_rows"],
)

print_copy_block(
    "VOCABULARIO DEFINITIVO",
    vocabulary_summary,
)

Palabras auditadas >=100: 103033
Excluidas por contrato final: ['external', 'links', 'references']
Vocabulario final: 103030
Filas esperadas: 10303000

PARA COPIAR Y PEGAR — VOCABULARIO DEFINITIVO
{
  "all_8_category_words_present": true,
  "audit_source": "C:\\Users\\jisaa\\Workspace\\Tesis\\Exp2\\Results\\ResultsExp2_prototipos\\00_corpus_audit\\common_pool_eligible_vocabulary_min100.csv",
  "category_words": [
    "apple",
    "bicycle",
    "bread",
    "canoe",
    "chair",
    "jellyfish",
    "screwdriver",
    "skunk"
  ],
  "excluded_words": [
    "external",
    "links",
    "references"
  ],
  "expected_total_rows": 10303000,
  "n_words_audited_ge100": 103033,
  "n_words_excluded_by_final_lexical_contract": 3,
  "n_words_final": 103030,
  "occurrences_per_word": 100
}
FIN — VOCABULARIO DEFINITIVO


In [32]:
print("N palabras finales:", len(eligible_words))

print(
    "Valores no-string:",
    [
        repr(word)
        for word in eligible_words
        if not isinstance(word, str)
    ][:20]
)

print(
    "Tokens NA-like presentes:",
    [
        repr(word)
        for word in eligible_words
        if isinstance(word, str)
        and word.lower() in {
            "nan",
            "null",
            "none",
            "na",
        }
    ]
)

N palabras finales: 103030
Valores no-string: []
Tokens NA-like presentes: ["'nan'", "'null'"]


In [33]:
print("N palabras finales:", len(eligible_words))

print(
    "Valores no-string:",
    [
        repr(word)
        for word in eligible_words
        if not isinstance(word, str)
    ][:20]
)

print(
    "Tokens NA-like presentes:",
    [
        repr(word)
        for word in eligible_words
        if isinstance(word, str)
        and word.lower() in {
            "nan",
            "null",
            "none",
            "na",
        }
    ]
)

N palabras finales: 103030
Valores no-string: []
Tokens NA-like presentes: ["'nan'", "'null'"]


In [9]:
# ============================================================
# 07. Pool común de corpus
# ============================================================

CORPUS_SOURCES = [
    {
        "source_name": "wikitext_103_raw",
        "dataset_name": "wikitext",
        "config": "wikitext-103-raw-v1",
        "splits": ["train", "validation", "test"],
        "text_field": "text",
        "streaming": False,
        "max_records_per_split": None,
    },

    {
        "source_name": "wikimedia_wikipedia_20231101_en_common",
        "dataset_name": "wikimedia/wikipedia",
        "config": "20231101.en",
        "splits": ["train"],
        "text_field": "text",
        "streaming": True,
        "max_records_per_split": 800_000,
    },
]


def load_hf_split(
    dataset_name,
    config,
    split,
    streaming,
):

    if config is None:
        return load_dataset(
            dataset_name,
            split=split,
            streaming=streaming,
        )

    return load_dataset(
        dataset_name,
        config,
        split=split,
        streaming=streaming,
    )


def iter_corpus_texts():

    for source in CORPUS_SOURCES:

        source_name = source["source_name"]
        dataset_name = source["dataset_name"]
        config = source.get("config")
        text_field = source.get("text_field", "text")
        streaming = bool(source.get("streaming", False))
        max_records = source.get("max_records_per_split")

        for split in source["splits"]:

            print(
                "\nCargando fuente:",
                source_name,
                "| split=",
                split,
                "| streaming=",
                streaming,
            )

            try:
                dataset = load_hf_split(
                    dataset_name,
                    config,
                    split,
                    streaming,
                )

            except Exception as exc:

                raise RuntimeError(
                    "No se pudo cargar una fuente que forma parte "
                    "del pool científico definitivo: "
                    f"{source_name}/{split}. "
                    f"Error: {repr(exc)}"
                ) from exc

            for row_idx, row in enumerate(dataset):

                if (
                    max_records is not None
                    and row_idx >= max_records
                ):
                    break

                text = row.get(text_field, "")

                if not isinstance(text, str):
                    continue

                if not text.strip():
                    continue

                yield {
                    "source_name": source_name,
                    "source_split": split,
                    "source_row_idx": int(row_idx),
                    "text": text,
                }


display(
    pd.DataFrame(CORPUS_SOURCES)
)

,source_name,dataset_name,config,splits,text_field,streaming,max_records_per_split
0,wikitext_103_raw,wikitext,wikitext-103-raw-v1,"[train, validation, test]",text,False,NaN
1,wikimedia_wikipedia_20231101_en_common,wikimedia/wikipedia,20231101.en,[train],text,True,800000.0


In [34]:
# ============================================================
# 08. Selección reproducible top-100 por hash
# ============================================================
#
# Para cada palabra:
#   1) cada context_window único recibe una prioridad BLAKE2b;
#   2) se conservan las 100 prioridades menores;
#   3) todas las palabras usan exactamente el mismo procedimiento.
#
# Solo se mantienen hashes en RAM, no millones de textos.
# ============================================================

HASH_MANIFEST_SCHEMA = pa.schema(
    [
        ("word", pa.string()),
        ("priority", pa.binary(16)),
    ]
)


def save_selected_hash_manifest(
    selected_scores_by_word,
    path,
):

    path = Path(path)

    if path.exists():
        path.unlink()

    writer = pq.ParquetWriter(
        path,
        schema=HASH_MANIFEST_SCHEMA,
        compression="zstd",
        compression_level=3,
    )

    try:

        words_chunk = []
        priorities_chunk = []

        for word in sorted(
            selected_scores_by_word
        ):

            scores = sorted(
                selected_scores_by_word[word]
            )

            for score in scores:

                words_chunk.append(word)
                priorities_chunk.append(
                    priority_int_to_bytes(score)
                )

            if len(words_chunk) >= 100_000:

                table = pa.Table.from_arrays(
                    [
                        pa.array(
                            words_chunk,
                            type=pa.string(),
                        ),
                        pa.array(
                            priorities_chunk,
                            type=pa.binary(16),
                        ),
                    ],
                    schema=HASH_MANIFEST_SCHEMA,
                )

                writer.write_table(
                    table,
                    row_group_size=100_000,
                )

                words_chunk.clear()
                priorities_chunk.clear()

        if words_chunk:

            table = pa.Table.from_arrays(
                [
                    pa.array(
                        words_chunk,
                        type=pa.string(),
                    ),
                    pa.array(
                        priorities_chunk,
                        type=pa.binary(16),
                    ),
                ],
                schema=HASH_MANIFEST_SCHEMA,
            )

            writer.write_table(
                table,
                row_group_size=100_000,
            )

    finally:
        writer.close()


def load_selected_hash_manifest(
    path,
    eligible_words,
):

    selected = {
        word: set()
        for word in eligible_words
    }

    parquet_file = pq.ParquetFile(path)

    for batch in tqdm(
        parquet_file.iter_batches(
            batch_size=200_000,
            columns=[
                "word",
                "priority",
            ],
        ),
        desc="Cargando manifest de hashes",
    ):

        words_batch = (
            batch.column(0).to_pylist()
        )

        priorities_batch = (
            batch.column(1).to_pylist()
        )

        for word, priority_bytes in zip(
            words_batch,
            priorities_batch,
        ):

            if word not in selected:
                raise RuntimeError(
                    "El manifest contiene una palabra fuera "
                    f"del vocabulario definitivo: {word}"
                )

            selected[word].add(
                int.from_bytes(
                    priority_bytes,
                    byteorder="big",
                    signed=False,
                )
            )

    bad_counts = {
        word: len(scores)
        for word, scores in selected.items()
        if len(scores) != OCCURRENCES_PER_WORD
    }

    if bad_counts:
        raise RuntimeError(
            "Manifest de selección inválido. "
            f"Ejemplos: {list(bad_counts.items())[:20]}"
        )

    return selected


def compute_top100_context_hashes():

    # ============================================================
    # 1. Estado de selección
    # ============================================================

    heaps = {
        word: []
        for word
        in eligible_words
    }

    selected = {
        word: set()
        for word
        in eligible_words
    }

    source_documents = Counter()
    source_valid_occurrences = Counter()

    n_words_total = int(
        len(
            eligible_words
        )
    )

    n_slots_total = int(
        n_words_total
        * OCCURRENCES_PER_WORD
    )

    # Número total de slots actualmente ocupados.
    # Crece hasta n_words * 100.
    selected_slots_total = 0

    # Número de palabras cuyo heap ya alcanzó 100 candidatos.
    # NO significa selección final: todavía pueden ocurrir
    # reemplazos posteriores por hashes menores.
    heap_full_count = 0

    global_documents_seen = 0

    started = time.perf_counter()


    # ============================================================
    # 2. Recorrer fuente por fuente
    # ============================================================

    for source in CORPUS_SOURCES:

        source_name = source[
            "source_name"
        ]

        dataset_name = source[
            "dataset_name"
        ]

        config = source.get(
            "config"
        )

        text_field = source.get(
            "text_field",
            "text",
        )

        streaming = bool(
            source.get(
                "streaming",
                False,
            )
        )

        max_records = source.get(
            "max_records_per_split"
        )


        for split in source[
            "splits"
        ]:

            # ----------------------------------------------------
            # 2.a. Cargar dataset
            # ----------------------------------------------------

            print(
                "\n"
                + "=" * 90
            )

            print(
                "SELECCIÓN TOP-100"
            )

            print(
                "Fuente:",
                source_name
            )

            print(
                "Split:",
                split
            )

            print(
                "Streaming:",
                streaming
            )

            print(
                "=" * 90
            )


            try:

                dataset = load_hf_split(
                    dataset_name,
                    config,
                    split,
                    streaming,
                )

            except Exception as exc:

                raise RuntimeError(
                    "No se pudo cargar una fuente perteneciente "
                    "al pool científico definitivo: "
                    f"{source_name}/{split}. "
                    f"Error={repr(exc)}"
                ) from exc


            # ----------------------------------------------------
            # 2.b. Determinar total para tqdm
            # ----------------------------------------------------

            if max_records is not None:

                source_total = int(
                    max_records
                )

            elif (
                not streaming
                and hasattr(
                    dataset,
                    "__len__",
                )
            ):

                source_total = int(
                    len(
                        dataset
                    )
                )

            else:

                source_total = None


            # ----------------------------------------------------
            # 2.c. Barra específica para esta fuente/split
            # ----------------------------------------------------

            source_desc = (
                f"{source_name}/{split}"
            )


            pbar = tqdm(
                total=source_total,
                desc=source_desc,
                unit="docs",
                dynamic_ncols=True,
            )


            # ----------------------------------------------------
            # 2.d. Recorrer documentos
            # ----------------------------------------------------

            try:

                for row_idx, row in enumerate(
                    dataset
                ):

                    if (
                        max_records is not None
                        and row_idx
                        >= max_records
                    ):
                        break


                    pbar.update(
                        1
                    )

                    global_documents_seen += 1

                    source_documents[
                        source_name
                    ] += 1


                    # ------------------------------------------------
                    # Texto
                    # ------------------------------------------------

                    text = row.get(
                        text_field,
                        "",
                    )


                    if (
                        not isinstance(
                            text,
                            str,
                        )
                        or not text.strip()
                    ):
                        continue


                    if is_heading_line(
                        text
                    ):
                        continue


                    tokens = strict_tokenize(
                        text
                    )


                    if len(
                        tokens
                    ) < MIN_LINE_WORDS:
                        continue


                    # ------------------------------------------------
                    # Ocurrencias candidatas
                    # ------------------------------------------------

                    for position, word in enumerate(
                        tokens
                    ):

                        if word not in eligible_word_set:
                            continue


                        (
                            window_tokens,
                            word_pos_window,
                        ) = build_centered_window(
                            tokens,
                            position,
                        )


                        if (
                            not window_tokens
                            or word_pos_window < 0
                            or word_pos_window
                            >= len(
                                window_tokens
                            )
                        ):
                            continue


                        if (
                            window_tokens[
                                word_pos_window
                            ]
                            != word
                        ):
                            continue


                        if len(
                            window_tokens
                        ) < MIN_LINE_WORDS:
                            continue


                        context_window = " ".join(
                            window_tokens
                        )


                        score = context_priority_int(
                            word,
                            context_window,
                        )


                        score_set = selected[
                            word
                        ]

                        heap = heaps[
                            word
                        ]


                        # --------------------------------------------
                        # Duplicado actualmente seleccionado
                        # --------------------------------------------

                        if score in score_set:
                            continue


                        source_valid_occurrences[
                            source_name
                        ] += 1


                        # --------------------------------------------
                        # Heap todavía incompleto
                        # --------------------------------------------

                        if len(
                            heap
                        ) < OCCURRENCES_PER_WORD:

                            heapq.heappush(
                                heap,
                                -score,
                            )

                            score_set.add(
                                score
                            )

                            selected_slots_total += 1


                            # La palabra acaba de alcanzar 100.
                            if (
                                len(
                                    heap
                                )
                                == OCCURRENCES_PER_WORD
                            ):

                                heap_full_count += 1


                        # --------------------------------------------
                        # Heap lleno: reemplazar solo si mejora top100
                        # --------------------------------------------

                        else:

                            current_largest_selected = (
                                -heap[
                                    0
                                ]
                            )


                            if (
                                score
                                < current_largest_selected
                            ):

                                old_negative = (
                                    heapq.heapreplace(
                                        heap,
                                        -score,
                                    )
                                )

                                old_score = (
                                    -old_negative
                                )

                                score_set.remove(
                                    old_score
                                )

                                score_set.add(
                                    score
                                )


                    # ------------------------------------------------
                    # Actualizar diagnóstico cada 2.000 documentos
                    # ------------------------------------------------

                    if (
                        pbar.n % 2000
                        == 0
                    ):

                        pbar.set_postfix(
                            {
                                "heap_full":
                                    (
                                        f"{heap_full_count:,}"
                                        f"/{n_words_total:,}"
                                    ),

                                "slots":
                                    (
                                        f"{selected_slots_total:,}"
                                        f"/{n_slots_total:,}"
                                    ),

                                "global_docs":
                                    f"{global_documents_seen:,}",
                            },
                            refresh=False,
                        )


            finally:

                pbar.close()


            # ----------------------------------------------------
            # 2.e. Resumen al terminar cada source/split
            # ----------------------------------------------------

            print(
                "\nFinalizado:",
                source_desc
            )

            print(
                "Documentos procesados en la fuente:",
                source_documents[
                    source_name
                ],
            )

            print(
                "Palabras con heap lleno:",
                f"{heap_full_count:,}/{n_words_total:,}",
            )

            print(
                "Slots ocupados:",
                f"{selected_slots_total:,}/{n_slots_total:,}",
            )


    # ============================================================
    # 3. Validación final
    # ============================================================

    elapsed_seconds = float(
        time.perf_counter()
        - started
    )


    bad_counts = {
        word:
            len(
                scores
            )
        for word, scores
        in selected.items()
        if len(
            scores
        ) != OCCURRENCES_PER_WORD
    }


    if bad_counts:

        raise RuntimeError(
            "El pool común no produjo exactamente "
            f"{OCCURRENCES_PER_WORD} contextos seleccionables "
            "para todas las palabras auditadas. "
            f"N problemáticas={len(bad_counts)}. "
            f"Ejemplos={list(bad_counts.items())[:20]}"
        )


    if heap_full_count != n_words_total:

        raise RuntimeError(
            "Inconsistencia interna: no todas las palabras "
            "terminaron con el heap completo."
        )


    if selected_slots_total != n_slots_total:

        raise RuntimeError(
            "Inconsistencia interna en el número de slots "
            "seleccionados."
        )


    # ============================================================
    # 4. Fingerprint científico de la selección
    # ============================================================

    selection_hasher = hashlib.sha256()


    for word in sorted(
        selected
    ):

        selection_hasher.update(
            word.encode(
                "utf-8"
            )
        )

        selection_hasher.update(
            b"\0"
        )


        for score in sorted(
            selected[
                word
            ]
        ):

            selection_hasher.update(
                priority_int_to_bytes(
                    score
                )
            )


    # ============================================================
    # 5. Reporte
    # ============================================================

    report = {

        "selection_method":
            (
                "blake2b_128_keep_100_smallest_"
                "unique_word_context_hashes"
            ),

        "selection_personalization":
            "ctx100-v1",

        "occurrences_per_word":
            int(
                OCCURRENCES_PER_WORD
            ),

        "n_words":
            int(
                n_words_total
            ),

        "n_selected_contexts":
            int(
                n_slots_total
            ),

        "all_heaps_full":
            bool(
                heap_full_count
                == n_words_total
            ),

        "selection_sha256":
            selection_hasher.hexdigest(),

        "source_documents_scanned":
            source_documents,

        "source_valid_occurrences_seen":
            source_valid_occurrences,

        "global_documents_scanned":
            int(
                global_documents_seen
            ),

        "elapsed_seconds":
            elapsed_seconds,
    }


    return (
        selected,
        report,
    )


if (
    REUSE_SELECTION_MANIFEST
    and SELECTED_HASHES_PATH.exists()
):

    print(
        "Reutilizando manifest existente:",
        SELECTED_HASHES_PATH,
    )

    selected_hashes_by_word = (
        load_selected_hash_manifest(
            SELECTED_HASHES_PATH,
            eligible_words,
        )
    )

    if SELECTION_REPORT_PATH.exists():

        with open(
            SELECTION_REPORT_PATH,
            "r",
            encoding="utf-8",
        ) as handle:

            selection_report = json.load(
                handle
            )

    else:

        selection_report = {
            "selection_method": (
                "loaded_existing_manifest"
            ),
            "occurrences_per_word": (
                OCCURRENCES_PER_WORD
            ),
            "n_words": len(
                eligible_words
            ),
            "n_selected_contexts": (
                len(eligible_words)
                * OCCURRENCES_PER_WORD
            ),
        }

else:

    (
        selected_hashes_by_word,
        selection_report,
    ) = compute_top100_context_hashes()

    save_selected_hash_manifest(
        selected_hashes_by_word,
        SELECTED_HASHES_PATH,
    )

    save_json(
        selection_report,
        SELECTION_REPORT_PATH,
    )


print_copy_block(
    "SELECCIÓN HASH TOP100",
    selection_report,
)

Reutilizando manifest existente: C:\Users\jisaa\Workspace\Tesis\Exp2\Results\contextual_dataset\selected_context_hashes_top100.parquet


Cargando manifest de hashes: 0it [00:00, ?it/s]


PARA COPIAR Y PEGAR — SELECCIÓN HASH TOP100
{
  "all_heaps_full": true,
  "elapsed_seconds": 1619.6036991,
  "global_documents_scanned": 2609468,
  "n_selected_contexts": 10303000,
  "n_words": 103030,
  "occurrences_per_word": 100,
  "selection_method": "blake2b_128_keep_100_smallest_unique_word_context_hashes",
  "selection_personalization": "ctx100-v1",
  "selection_sha256": "5aef564ca77bb2c9e8fd35f923dd3ee583b85b4d7e01990e82d66c6fb3d302dd",
  "source_documents_scanned": {
    "wikimedia_wikipedia_20231101_en_common": 800000,
    "wikitext_103_raw": 1809468
  },
  "source_valid_occurrences_seen": {
    "wikimedia_wikipedia_20231101_en_common": 243906932,
    "wikitext_103_raw": 41931083
  }
}
FIN — SELECCIÓN HASH TOP100


In [35]:
# ============================================================
# 08.b. Corrección final del vocabulario: excluir "nan"
#       SIN repetir el escaneo del corpus
# ============================================================
#
# La palabra literal "nan" quedó preservada correctamente como
# string tras corregir pd.read_csv(), pero no se conservará en
# el vocabulario conceptual definitivo.
#
# La selección top-100 es independiente por palabra, por lo que
# eliminarla ahora es exactamente equivalente a haberla excluido
# antes del escaneo para todas las demás palabras.
# ============================================================

WORD_TO_EXCLUDE_POST_SELECTION = "nan"


# ------------------------------------------------------------
# 1. Incorporar "nan" al contrato léxico definitivo
# ------------------------------------------------------------

CUSTOM_BAD_WORDS.add(
    WORD_TO_EXCLUDE_POST_SELECTION
)

BAD_WORDS = (
    set(STOPWORDS)
    | set(CUSTOM_BAD_WORDS)
)


# ------------------------------------------------------------
# 2. Verificar que efectivamente estaba en la selección
# ------------------------------------------------------------

if (
    WORD_TO_EXCLUDE_POST_SELECTION
    not in selected_hashes_by_word
):
    raise RuntimeError(
        "'nan' no está presente en selected_hashes_by_word; "
        "no se aplicará una corrección silenciosa."
    )


n_nan_contexts_selected = len(
    selected_hashes_by_word[
        WORD_TO_EXCLUDE_POST_SELECTION
    ]
)


if n_nan_contexts_selected != OCCURRENCES_PER_WORD:
    raise RuntimeError(
        "'nan' no contiene exactamente "
        f"{OCCURRENCES_PER_WORD} contextos seleccionados: "
        f"{n_nan_contexts_selected}"
    )


# ------------------------------------------------------------
# 3. Eliminar "nan" de la selección
# ------------------------------------------------------------

selected_hashes_by_word.pop(
    WORD_TO_EXCLUDE_POST_SELECTION
)


# ------------------------------------------------------------
# 4. Eliminar "nan" del vocabulario definitivo
# ------------------------------------------------------------

eligible_vocab_df = (
    eligible_vocab_df.loc[
        eligible_vocab_df["word"]
        != WORD_TO_EXCLUDE_POST_SELECTION
    ]
    .copy()
    .sort_values("word")
    .reset_index(drop=True)
)


eligible_words = (
    eligible_vocab_df["word"]
    .astype(str)
    .tolist()
)


eligible_word_set = set(
    eligible_words
)


# ------------------------------------------------------------
# 5. Validaciones fuertes
# ------------------------------------------------------------

EXPECTED_FINAL_VOCAB_SIZE = 103_029
EXPECTED_FINAL_ROWS = (
    EXPECTED_FINAL_VOCAB_SIZE
    * OCCURRENCES_PER_WORD
)


if len(eligible_words) != EXPECTED_FINAL_VOCAB_SIZE:
    raise RuntimeError(
        "Tamaño inesperado del vocabulario final: "
        f"{len(eligible_words)} != {EXPECTED_FINAL_VOCAB_SIZE}"
    )


if len(selected_hashes_by_word) != EXPECTED_FINAL_VOCAB_SIZE:
    raise RuntimeError(
        "Tamaño inesperado de selected_hashes_by_word: "
        f"{len(selected_hashes_by_word)} "
        f"!= {EXPECTED_FINAL_VOCAB_SIZE}"
    )


if set(selected_hashes_by_word) != eligible_word_set:
    raise RuntimeError(
        "Las palabras de la selección no coinciden exactamente "
        "con el vocabulario definitivo."
    )


bad_selection_counts = {
    word: len(scores)
    for word, scores
    in selected_hashes_by_word.items()
    if len(scores) != OCCURRENCES_PER_WORD
}


if bad_selection_counts:
    raise RuntimeError(
        "Hay palabras con conteo de selección distinto de 100. "
        f"Ejemplos: {list(bad_selection_counts.items())[:20]}"
    )


if WORD_TO_EXCLUDE_POST_SELECTION in eligible_word_set:
    raise RuntimeError(
        "'nan' sigue presente en eligible_word_set."
    )


if any(
    not isinstance(word, str)
    for word
    in eligible_words
):
    raise RuntimeError(
        "Persisten valores no-string en eligible_words."
    )


missing_category_words = sorted(
    set(category_words)
    - eligible_word_set
)


if missing_category_words:
    raise RuntimeError(
        "Se perdió alguna CategoryWord: "
        f"{missing_category_words}"
    )


# ------------------------------------------------------------
# 6. Reescribir vocabulario definitivo
# ------------------------------------------------------------

eligible_vocab_df.to_csv(
    FINAL_ELIGIBLE_VOCAB_PATH,
    index=False,
    encoding="utf-8",
)


# ------------------------------------------------------------
# 7. Reescribir manifest top-100 sin "nan"
# ------------------------------------------------------------

save_selected_hash_manifest(
    selected_hashes_by_word,
    SELECTED_HASHES_PATH,
)


# ------------------------------------------------------------
# 8. Recalcular SHA256 científico de la selección
# ------------------------------------------------------------

selection_hasher = hashlib.sha256()


for word in sorted(
    selected_hashes_by_word
):

    selection_hasher.update(
        word.encode("utf-8")
    )

    selection_hasher.update(
        b"\0"
    )

    for score in sorted(
        selected_hashes_by_word[word]
    ):

        selection_hasher.update(
            priority_int_to_bytes(
                score
            )
        )


final_selection_sha256 = (
    selection_hasher.hexdigest()
)


# ------------------------------------------------------------
# 9. Actualizar reporte de selección
# ------------------------------------------------------------

selection_report[
    "n_words"
] = int(
    len(
        eligible_words
    )
)

selection_report[
    "n_selected_contexts"
] = int(
    len(
        eligible_words
    )
    * OCCURRENCES_PER_WORD
)

selection_report[
    "selection_sha256"
] = final_selection_sha256

selection_report[
    "all_heaps_full"
] = True

selection_report[
    "post_selection_excluded_words"
] = [
    WORD_TO_EXCLUDE_POST_SELECTION
]

selection_report[
    "post_selection_exclusion_reason"
] = (
    "lexical exclusion of literal token 'nan'; "
    "independent per-word top100 selections for all remaining "
    "words are unchanged"
)


save_json(
    selection_report,
    SELECTION_REPORT_PATH,
)


# ------------------------------------------------------------
# 10. Mantener vocabulary_summary coherente para metadata final
# ------------------------------------------------------------

vocabulary_summary[
    "n_words_final"
] = int(
    len(
        eligible_words
    )
)

vocabulary_summary[
    "expected_total_rows"
] = int(
    len(
        eligible_words
    )
    * OCCURRENCES_PER_WORD
)


excluded_words_updated = sorted(
    set(
        vocabulary_summary.get(
            "excluded_words",
            []
        )
    )
    | {
        WORD_TO_EXCLUDE_POST_SELECTION
    }
)


vocabulary_summary[
    "excluded_words"
] = excluded_words_updated

vocabulary_summary[
    "n_words_excluded_by_final_lexical_contract"
] = int(
    len(
        excluded_words_updated
    )
)


# ------------------------------------------------------------
# 11. Resumen
# ------------------------------------------------------------

post_selection_fix_report = {

    "excluded_word":
        WORD_TO_EXCLUDE_POST_SELECTION,

    "removed_selected_contexts":
        int(
            n_nan_contexts_selected
        ),

    "n_words_final":
        int(
            len(
                eligible_words
            )
        ),

    "occurrences_per_word":
        int(
            OCCURRENCES_PER_WORD
        ),

    "n_selected_contexts_final":
        int(
            len(
                eligible_words
            )
            * OCCURRENCES_PER_WORD
        ),

    "expected_final_rows":
        int(
            EXPECTED_FINAL_ROWS
        ),

    "all_8_category_words_present":
        bool(
            set(category_words)
            .issubset(
                eligible_word_set
            )
        ),

    "all_words_have_exactly_100_selected":
        bool(
            not bad_selection_counts
        ),

    "selection_sha256_final":
        final_selection_sha256,

    "null_retained_as_valid_literal_word":
        bool(
            "null"
            in eligible_word_set
        ),
}


print_copy_block(
    "CORRECCIÓN POST-SELECCIÓN NAN",
    post_selection_fix_report,
)


PARA COPIAR Y PEGAR — CORRECCIÓN POST-SELECCIÓN NAN
{
  "all_8_category_words_present": true,
  "all_words_have_exactly_100_selected": true,
  "excluded_word": "nan",
  "expected_final_rows": 10302900,
  "n_selected_contexts_final": 10302900,
  "n_words_final": 103029,
  "null_retained_as_valid_literal_word": true,
  "occurrences_per_word": 100,
  "removed_selected_contexts": 100,
  "selection_sha256_final": "e130061a093ff98c3265cda041a3a86e77c6204c3509e0aa0a56405af87af087"
}
FIN — CORRECCIÓN POST-SELECCIÓN NAN


In [36]:
# ============================================================
# 09. Materializar exactamente los contextos seleccionados
# ============================================================
#
# Se vuelve a recorrer el mismo pool.
# Solo se escriben los (word, context_window) congelados en
# selected_context_hashes_top100.parquet.
# ============================================================

CANDIDATE_SCHEMA = pa.schema(
    [
        ("candidate_id", pa.int64()),
        ("word", pa.string()),
        ("is_target", pa.bool_()),
        ("word_role", pa.string()),
        ("true_class", pa.int32()),
        ("class_label", pa.string()),
        ("is_exp2_word", pa.bool_()),
        ("selection_stage", pa.string()),
        ("sample_priority", pa.binary(16)),
        ("source_name", pa.string()),
        ("source_split", pa.string()),
        ("source_row_idx", pa.int64()),
        ("source_doc_key", pa.string()),
        ("doc_split", pa.string()),
        ("word_pos_original", pa.int32()),
        ("word_pos_window", pa.int32()),
        ("n_tokens_window", pa.int32()),
        ("context_window", pa.string()),
        ("left_context", pa.string()),
        ("right_context", pa.string()),
        ("text", pa.string()),
    ]
)


def records_to_candidate_table(records):

    arrays = []

    for field in CANDIDATE_SCHEMA:

        arrays.append(
            pa.array(
                [
                    row[field.name]
                    for row in records
                ],
                type=field.type,
            )
        )

    return pa.Table.from_arrays(
        arrays,
        schema=CANDIDATE_SCHEMA,
    )


def make_candidate_record(
    *,
    candidate_id,
    word,
    score,
    source_name,
    source_split,
    source_row_idx,
    text,
    tokens,
    word_pos_original,
):

    window_tokens, word_pos_window = (
        build_centered_window(
            tokens,
            word_pos_original,
        )
    )

    if (
        not window_tokens
        or word_pos_window < 0
        or word_pos_window
        >= len(window_tokens)
    ):
        return None

    if window_tokens[
        word_pos_window
    ] != word:
        return None

    if len(window_tokens) < MIN_LINE_WORDS:
        return None

    context_window = " ".join(
        window_tokens
    )

    score_check = context_priority_int(
        word,
        context_window,
    )

    if score_check != score:
        raise RuntimeError(
            "La prioridad recalculada no coincide con "
            "la selección congelada."
        )

    left_context, right_context = (
        short_context(
            window_tokens,
            word_pos_window,
        )
    )

    source_doc_key = (
        f"{source_name}:"
        f"{source_split}:"
        f"{source_row_idx}"
    )

    class_meta = get_exp2_class_metadata(
        word
    )

    return {
        "candidate_id": int(candidate_id),
        "word": str(word),
        "is_target": bool(
            word in target_set
        ),
        "word_role": str(
            target_roles.get(
                word,
                "filler",
            )
        ),
        "true_class": int(
            class_meta["true_class"]
        ),
        "class_label": str(
            class_meta["class_label"]
        ),
        "is_exp2_word": bool(
            class_meta["is_exp2_word"]
        ),
        "selection_stage": (
            "common_pool_hash_top100"
        ),
        "sample_priority": (
            priority_int_to_bytes(score)
        ),
        "source_name": str(
            source_name
        ),
        "source_split": str(
            source_split
        ),
        "source_row_idx": int(
            source_row_idx
        ),
        "source_doc_key": str(
            source_doc_key
        ),
        "doc_split": str(
            stable_doc_split(
                source_doc_key
            )
        ),
        "word_pos_original": int(
            word_pos_original
        ),
        "word_pos_window": int(
            word_pos_window
        ),
        "n_tokens_window": int(
            len(window_tokens)
        ),
        "context_window": (
            context_window
        ),
        "left_context": (
            left_context
        ),
        "right_context": (
            right_context
        ),
        "text": (
            normalize_text(text)[:1000]
        ),
    }


def materialize_selected_candidates(
    selected_hashes_by_word,
):

    if UNSORTED_CANDIDATES_PATH.exists():
        UNSORTED_CANDIDATES_PATH.unlink()

    remaining = {
        word: set(scores)
        for word, scores
        in selected_hashes_by_word.items()
    }

    remaining_total = int(
        len(eligible_words)
        * OCCURRENCES_PER_WORD
    )

    candidate_id = 0
    source_written = Counter()
    buffer = []

    writer = pq.ParquetWriter(
        UNSORTED_CANDIDATES_PATH,
        schema=CANDIDATE_SCHEMA,
        compression="zstd",
        compression_level=3,
    )

    started = time.perf_counter()

    pbar = tqdm(
        total=remaining_total,
        desc="Materializando candidatos top100",
        unit="ctx",
    )

    try:

        for item in iter_corpus_texts():

            if remaining_total == 0:
                break

            text = item["text"]

            if is_heading_line(text):
                continue

            tokens = strict_tokenize(text)

            if len(tokens) < MIN_LINE_WORDS:
                continue

            source_name = item[
                "source_name"
            ]

            source_split = item[
                "source_split"
            ]

            source_row_idx = item[
                "source_row_idx"
            ]

            for position, word in enumerate(
                tokens
            ):

                scores_for_word = (
                    remaining.get(word)
                )

                if not scores_for_word:
                    continue

                window_tokens, word_pos_window = (
                    build_centered_window(
                        tokens,
                        position,
                    )
                )

                if (
                    not window_tokens
                    or word_pos_window < 0
                    or word_pos_window
                    >= len(window_tokens)
                ):
                    continue

                if window_tokens[
                    word_pos_window
                ] != word:
                    continue

                context_window = " ".join(
                    window_tokens
                )

                score = context_priority_int(
                    word,
                    context_window,
                )

                if score not in scores_for_word:
                    continue

                record = make_candidate_record(
                    candidate_id=candidate_id,
                    word=word,
                    score=score,
                    source_name=source_name,
                    source_split=source_split,
                    source_row_idx=source_row_idx,
                    text=text,
                    tokens=tokens,
                    word_pos_original=position,
                )

                if record is None:
                    raise RuntimeError(
                        "Un contexto seleccionado no pudo materializarse."
                    )

                buffer.append(record)

                scores_for_word.remove(
                    score
                )

                candidate_id += 1
                remaining_total -= 1

                source_written[
                    source_name
                ] += 1

                pbar.update(1)

                if len(buffer) >= 50_000:

                    writer.write_table(
                        records_to_candidate_table(
                            buffer
                        ),
                        row_group_size=50_000,
                    )

                    buffer.clear()

            if remaining_total == 0:
                break

        if buffer:

            writer.write_table(
                records_to_candidate_table(
                    buffer
                ),
                row_group_size=50_000,
            )

            buffer.clear()

    finally:

        writer.close()
        pbar.close()

    missing = {
        word: len(scores)
        for word, scores in remaining.items()
        if scores
    }

    if missing:
        raise RuntimeError(
            "No fue posible rematerializar todos los contextos "
            "seleccionados. "
            f"N palabras incompletas={len(missing)}. "
            f"Ejemplos={list(missing.items())[:20]}"
        )

    parquet_file = pq.ParquetFile(
        UNSORTED_CANDIDATES_PATH
    )

    expected_rows = int(
        len(eligible_words)
        * OCCURRENCES_PER_WORD
    )

    if (
        parquet_file.metadata.num_rows
        != expected_rows
    ):
        raise RuntimeError(
            "Número incorrecto de candidatos materializados: "
            f"{parquet_file.metadata.num_rows} != {expected_rows}"
        )

    return {
        "n_candidates": int(
            parquet_file.metadata.num_rows
        ),
        "n_words": int(
            len(eligible_words)
        ),
        "occurrences_per_word": int(
            OCCURRENCES_PER_WORD
        ),
        "source_written_counts": (
            source_written
        ),
        "elapsed_seconds": float(
            time.perf_counter()
            - started
        ),
        "unsorted_candidate_path": str(
            UNSORTED_CANDIDATES_PATH
        ),
    }


def candidate_file_is_structurally_valid(
    path,
):

    path = Path(path)

    if not path.exists():
        return False

    try:

        parquet_file = pq.ParquetFile(
            path
        )

        expected_rows = (
            len(eligible_words)
            * OCCURRENCES_PER_WORD
        )

        return (
            parquet_file.metadata.num_rows
            == expected_rows
            and set(
                CANDIDATE_SCHEMA.names
            ).issubset(
                parquet_file.schema_arrow.names
            )
        )

    except Exception:
        return False


if (
    REUSE_SORTED_CANDIDATES
    and candidate_file_is_structurally_valid(
        CANDIDATES_PATH
    )
):

    print(
        "Candidatos ordenados existentes y estructuralmente válidos. "
        "Se omite rematerialización."
    )

    candidate_materialization_report = {
        "reused_sorted_candidates": True,
        "candidate_path": str(
            CANDIDATES_PATH
        ),
        "n_candidates": int(
            pq.ParquetFile(
                CANDIDATES_PATH
            ).metadata.num_rows
        ),
    }

else:

    candidate_materialization_report = (
        materialize_selected_candidates(
            selected_hashes_by_word
        )
    )


print_copy_block(
    "CANDIDATOS MATERIALIZADOS",
    candidate_materialization_report,
)

Materializando candidatos top100:   0%|          | 0/10302900 [00:00<?, ?ctx/s]


Cargando fuente: wikitext_103_raw | split= train | streaming= False

Cargando fuente: wikitext_103_raw | split= validation | streaming= False

Cargando fuente: wikitext_103_raw | split= test | streaming= False

Cargando fuente: wikimedia_wikipedia_20231101_en_common | split= train | streaming= True


Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/wikimedia/wikipedia/resolve/b04c8d1ceb2f5cd4588862100d08de323dccfbaa/20231101.en/train-00000-of-00041.parquet
Retrying in 1s [Retry 1/5].
'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/wikimedia/wikipedia/resolve/b04c8d1ceb2f5cd4588862100d08de323dccfbaa/20231101.en/train-00000-of-00041.parquet
Retrying in 2s [Retry 2/5].
'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/wikimedia/wikipedia/resolve/b04c8d1ceb2f5cd4588862100d08de323dccfbaa/20231101.en/train-00000-of-00041.parquet
Retrying in 4s [Retry 3/5].
'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/wikimedia/wikipedia/resolve/b04c8d1ceb2f5cd4588862100d08de323dccfbaa/20231101.en/train-00000-of-00041.parquet
Retrying in 8s [Retry 4/5].



PARA COPIAR Y PEGAR — CANDIDATOS MATERIALIZADOS
{
  "elapsed_seconds": 59720.91784939999,
  "n_candidates": 10302900,
  "n_words": 103029,
  "occurrences_per_word": 100,
  "source_written_counts": {
    "wikimedia_wikipedia_20231101_en_common": 8900836,
    "wikitext_103_raw": 1402064
  },
  "unsorted_candidate_path": "C:\\Users\\jisaa\\Workspace\\Tesis\\Exp2\\Results\\contextual_dataset\\_contextual_occurrence_candidates_unsorted.parquet"
}
FIN — CANDIDATOS MATERIALIZADOS


In [37]:
# ============================================================
# 10. Ordenamiento externo de candidatos por palabra
# ============================================================
#
# Orden:
#     word ASC, sample_priority ASC
#
# DuckDB realiza el sort con spill a disco, sin exigir cargar
# el archivo completo en RAM.
# ============================================================

def duckdb_path(path):

    return (
        str(
            Path(path).resolve()
        )
        .replace("\\", "/")
        .replace("'", "''")
    )


def validate_sorted_candidate_counts(path):

    path_sql = duckdb_path(path)

    con = duckdb.connect(
        database=":memory:"
    )

    try:

        audit_df = con.execute(
            f"""
            SELECT
                COUNT(*) AS n_words,
                MIN(n) AS min_count,
                MAX(n) AS max_count,
                SUM(n) AS n_rows
            FROM (
                SELECT
                    word,
                    COUNT(*) AS n
                FROM read_parquet('{path_sql}')
                GROUP BY word
            )
            """
        ).fetchdf()

    finally:

        con.close()

    row = audit_df.iloc[0]

    return {
        "n_words": int(
            row["n_words"]
        ),
        "min_count": int(
            row["min_count"]
        ),
        "max_count": int(
            row["max_count"]
        ),
        "n_rows": int(
            row["n_rows"]
        ),
    }


needs_sort = not candidate_file_is_structurally_valid(
    CANDIDATES_PATH
)

if needs_sort:

    if not UNSORTED_CANDIDATES_PATH.exists():
        raise FileNotFoundError(
            "No existe el Parquet de candidatos sin ordenar."
        )

    if CANDIDATES_PATH.exists():
        CANDIDATES_PATH.unlink()

    DUCKDB_TEMP_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    con = duckdb.connect(
        database=":memory:"
    )

    try:

        con.execute(
            f"PRAGMA temp_directory='{duckdb_path(DUCKDB_TEMP_DIR)}'"
        )

        # Límite conservador para el LOQ con 24 GB RAM.
        con.execute(
            "SET memory_limit='10GB'"
        )

        con.execute(
            "SET preserve_insertion_order=false"
        )

        source_path_sql = duckdb_path(
            UNSORTED_CANDIDATES_PATH
        )

        target_path_sql = duckdb_path(
            CANDIDATES_PATH
        )

        query = f"""
        COPY (
            SELECT *
            FROM read_parquet('{source_path_sql}')
            ORDER BY
                word ASC,
                sample_priority ASC
        )
        TO '{target_path_sql}'
        (
            FORMAT PARQUET,
            COMPRESSION ZSTD,
            ROW_GROUP_SIZE {int(CANDIDATE_SORT_ROW_GROUP_ROWS)}
        )
        """

        print(
            "Ordenando candidatos externamente con DuckDB..."
        )

        started_sort = time.perf_counter()

        con.execute(query)

        sort_elapsed_seconds = float(
            time.perf_counter()
            - started_sort
        )

    finally:
        con.close()

    if not candidate_file_is_structurally_valid(
        CANDIDATES_PATH
    ):
        raise RuntimeError(
            "El archivo de candidatos ordenado no pasó "
            "la validación estructural."
        )

    candidate_count_audit = (
        validate_sorted_candidate_counts(
            CANDIDATES_PATH
        )
    )

    if (
        candidate_count_audit["n_words"]
        != len(eligible_words)
    ):
        raise RuntimeError(
            "Número de palabras incorrecto en candidatos ordenados."
        )

    if (
        candidate_count_audit["min_count"]
        != OCCURRENCES_PER_WORD
        or candidate_count_audit["max_count"]
        != OCCURRENCES_PER_WORD
    ):
        raise RuntimeError(
            "Los candidatos ordenados no tienen exactamente "
            "100 ocurrencias por palabra."
        )

    # El temporal sin ordenar ya no es necesario.
    UNSORTED_CANDIDATES_PATH.unlink()

    if DUCKDB_TEMP_DIR.exists():
        try:
            shutil.rmtree(
                DUCKDB_TEMP_DIR
            )
        except Exception:
            pass

    candidate_sort_report = {
        "sorted_candidate_path": str(
            CANDIDATES_PATH
        ),
        "sort_order": [
            "word ASC",
            "sample_priority ASC",
        ],
        **candidate_count_audit,
        "elapsed_seconds": (
            sort_elapsed_seconds
        ),
    }

else:

    candidate_count_audit = (
        validate_sorted_candidate_counts(
            CANDIDATES_PATH
        )
    )

    if (
        candidate_count_audit["n_words"]
        != len(eligible_words)
        or candidate_count_audit["min_count"]
        != OCCURRENCES_PER_WORD
        or candidate_count_audit["max_count"]
        != OCCURRENCES_PER_WORD
    ):
        raise RuntimeError(
            "El candidate parquet reutilizado no cumple "
            "el contrato exacto de 100 ocurrencias por palabra."
        )

    candidate_sort_report = {
        "reused_existing_sorted_candidates": True,
        "sorted_candidate_path": str(
            CANDIDATES_PATH
        ),
        **candidate_count_audit,
    }


print_copy_block(
    "CANDIDATOS ORDENADOS",
    candidate_sort_report,
)# Liberar el manifest expandido en RAM antes de BERT/DuckDB posteriores.
# Si se necesita reconstruir candidatos, puede recargarse desde
# selected_context_hashes_top100.parquet.
if "selected_hashes_by_word" in globals():
    del selected_hashes_by_word
    gc.collect()

Ordenando candidatos externamente con DuckDB...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


PARA COPIAR Y PEGAR — CANDIDATOS ORDENADOS
{
  "elapsed_seconds": 445.39678100001765,
  "max_count": 100,
  "min_count": 100,
  "n_rows": 10302900,
  "n_words": 103029,
  "sort_order": [
    "word ASC",
    "sample_priority ASC"
  ],
  "sorted_candidate_path": "C:\\Users\\jisaa\\Workspace\\Tesis\\Exp2\\Results\\contextual_dataset\\contextual_occurrence_candidates_sorted.parquet"
}
FIN — CANDIDATOS ORDENADOS


In [38]:
# ============================================================
# 10.b Diagnóstico de espacio disponible antes de BERT
# ============================================================

expected_rows = int(
    len(eligible_words)
    * OCCURRENCES_PER_WORD
)

raw_embedding_payload_gib = float(
    expected_rows
    * 768
    * 4
    / 1024**3
)

disk_usage = shutil.disk_usage(
    OUT_DIR
)

free_gib = float(
    disk_usage.free
    / 1024**3
)

candidate_size_gib = float(
    CANDIDATES_PATH.stat().st_size
    / 1024**3
)

print(
    "Filas esperadas:",
    expected_rows,
)

print(
    "Payload mínimo solo embeddings float32 [GiB]:",
    round(
        raw_embedding_payload_gib,
        2,
    ),
)

print(
    "Tamaño candidates [GiB]:",
    round(
        candidate_size_gib,
        2,
    ),
)

print(
    "Espacio libre actual [GiB]:",
    round(
        free_gib,
        2,
    ),
)

if free_gib < raw_embedding_payload_gib * 1.15:

    raise RuntimeError(
        "El espacio libre es demasiado cercano al payload mínimo "
        "de embeddings. Libera espacio antes de iniciar BERT."
    )

Filas esperadas: 10302900
Payload mínimo solo embeddings float32 [GiB]: 29.48
Tamaño candidates [GiB]: 4.74
Espacio libre actual [GiB]: 42.37


In [39]:
# ============================================================
# 11. Cargar BERT y capturar únicamente las últimas 4 capas
# ============================================================
#
# El generador anterior usaba output_hidden_states=True.
#
# Aquí se registran hooks solo en las capas 9-12 de BERT-base
# (índices 8,9,10,11), reduciendo VRAM sin cambiar la cantidad
# científica que se promedia.
# ============================================================

if DEVICE.type != "cuda":

    print(
        "ADVERTENCIA: CUDA no está disponible. "
        "La generación completa con BERT en CPU será muy costosa."
    )

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
)

if not tokenizer.is_fast:
    raise RuntimeError(
        "Se necesita un tokenizer fast para usar word_ids()."
    )

model = AutoModel.from_pretrained(
    MODEL_NAME,
    output_hidden_states=False,
)

model.to(DEVICE)
model.eval()

hidden_size = int(
    model.config.hidden_size
)

num_hidden_layers = int(
    model.config.num_hidden_layers
)

if hidden_size != 768:
    raise RuntimeError(
        f"Se esperaba BERT-base 768D y se obtuvo {hidden_size}."
    )

resolved_layer_indices = [
    (
        num_hidden_layers + layer_id
        if layer_id < 0
        else layer_id
    )
    for layer_id in SELECTED_LAYER_IDS
]

if resolved_layer_indices != [
    8,
    9,
    10,
    11,
]:
    raise RuntimeError(
        "Las capas resueltas no corresponden a las últimas "
        f"4 capas de BERT-base: {resolved_layer_indices}"
    )

captured_layer_outputs = {}


def _make_capture_hook(layer_idx):

    def hook(module, inputs, output):

        if isinstance(output, tuple):
            tensor = output[0]
        else:
            tensor = output

        captured_layer_outputs[
            layer_idx
        ] = tensor

    return hook


capture_hooks = []

for layer_idx in resolved_layer_indices:

    handle = (
        model.encoder.layer[
            layer_idx
        ].register_forward_hook(
            _make_capture_hook(
                layer_idx
            )
        )
    )

    capture_hooks.append(handle)

print("Modelo:", MODEL_NAME)
print("hidden_size:", hidden_size)
print("num_hidden_layers:", num_hidden_layers)
print(
    "Capas capturadas [0-based]:",
    resolved_layer_indices,
)
print("DEVICE:", DEVICE)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Modelo: bert-base-uncased
hidden_size: 768
num_hidden_layers: 12
Capas capturadas [0-based]: [8, 9, 10, 11]
DEVICE: cuda


In [43]:
# ============================================================
# 11.b Auditoría PRE-BERT con recorte WordPiece centrado
# ============================================================
#
# Política definitiva de entrada a BERT:
#
# 1. context_window sigue siendo ±40 palabras;
# 2. se tokeniza SIN truncar para localizar todos los WordPieces;
# 3. se conservan TODOS los WordPieces de la palabra objetivo;
# 4. si el input excede MAX_SEQ_LEN=128:
#       - se recorta alrededor del objetivo;
#       - se reparte el presupuesto entre izquierda/derecha;
#       - si un lado no lo usa, el otro aprovecha el remanente;
# 5. [CLS] + contenido + [SEP] <= 128.
#
# No se elimina ninguna ocurrencia.
# ============================================================

TOKENIZER_ALIGNMENT_AUDIT_PATH = (
    OUT_DIR
    / "tokenizer_alignment_audit.json"
)

TOKENIZER_AUDIT_BATCH_SIZE = 1024


N_SPECIAL_TOKENS = int(
    tokenizer.num_special_tokens_to_add(
        pair=False
    )
)

MAX_CONTENT_WORDPIECES = int(
    MAX_SEQ_LEN
    - N_SPECIAL_TOKENS
)


if N_SPECIAL_TOKENS != 2:
    raise RuntimeError(
        "Para bert-base-uncased se esperaban exactamente "
        f"2 tokens especiales, recibidos={N_SPECIAL_TOKENS}."
    )


if MAX_CONTENT_WORDPIECES != 126:
    raise RuntimeError(
        "Con MAX_SEQ_LEN=128 se esperaban "
        f"126 WordPieces de contenido, recibidos="
        f"{MAX_CONTENT_WORDPIECES}."
    )


def compute_target_centered_wordpiece_crop(
    input_ids_no_special,
    word_ids,
    target_word_position,
):
    """
    Calcula un recorte WordPiece centrado en la palabra objetivo.

    Retorna:
    - cropped_ids SIN tokens especiales;
    - posiciones del objetivo en el input final CON [CLS];
    - metadata diagnóstica.

    Nunca corta los WordPieces de la palabra objetivo.
    """

    input_ids_no_special = list(
        input_ids_no_special
    )

    word_ids = list(
        word_ids
    )

    target_word_position = int(
        target_word_position
    )


    target_positions = [
        idx
        for idx, word_id
        in enumerate(
            word_ids
        )
        if word_id
        == target_word_position
    ]


    if not target_positions:

        raise RuntimeError(
            "La palabra objetivo no tiene WordPieces "
            "antes del recorte."
        )


    target_start = int(
        min(
            target_positions
        )
    )

    target_end = int(
        max(
            target_positions
        )
    )


    # Los WordPieces de una palabra deben ser contiguos.
    expected_target_positions = list(
        range(
            target_start,
            target_end + 1,
        )
    )


    if target_positions != expected_target_positions:

        raise RuntimeError(
            "Los WordPieces de la palabra objetivo "
            "no son contiguos."
        )


    target_len = int(
        target_end
        - target_start
        + 1
    )


    if target_len > MAX_CONTENT_WORDPIECES:

        raise RuntimeError(
            "Una única palabra requiere más WordPieces "
            "que todo el presupuesto de BERT: "
            f"{target_len} > {MAX_CONTENT_WORDPIECES}"
        )


    n_raw_wordpieces = int(
        len(
            input_ids_no_special
        )
    )


    # ========================================================
    # Si cabe completo, no recortamos nada.
    # ========================================================

    if (
        n_raw_wordpieces
        <= MAX_CONTENT_WORDPIECES
    ):

        crop_start = 0
        crop_end = n_raw_wordpieces


    # ========================================================
    # Si no cabe, recortar alrededor del target.
    # ========================================================

    else:

        remaining_budget = int(
            MAX_CONTENT_WORDPIECES
            - target_len
        )

        left_available = int(
            target_start
        )

        right_available = int(
            n_raw_wordpieces
            - target_end
            - 1
        )


        # Mitad a cada lado.
        desired_left = int(
            remaining_budget
            // 2
        )

        desired_right = int(
            remaining_budget
            - desired_left
        )


        left_take = min(
            left_available,
            desired_left,
        )

        right_take = min(
            right_available,
            desired_right,
        )


        # Si uno de los lados no puede ocupar su mitad,
        # transferimos el presupuesto sobrante al otro.
        remaining = int(
            remaining_budget
            - left_take
            - right_take
        )


        if remaining > 0:

            extra_left = min(
                left_available
                - left_take,
                remaining,
            )

            left_take += extra_left
            remaining -= extra_left


        if remaining > 0:

            extra_right = min(
                right_available
                - right_take,
                remaining,
            )

            right_take += extra_right
            remaining -= extra_right


        crop_start = int(
            target_start
            - left_take
        )

        crop_end = int(
            target_end
            + right_take
            + 1
        )


    cropped_ids = (
        input_ids_no_special[
            crop_start:crop_end
        ]
    )


    # BERT agrega:
    #
    # [CLS] cropped_ids [SEP]
    #
    # por lo que el offset del contenido es +1.
    wp_start = int(
        1
        + target_start
        - crop_start
    )

    wp_end = int(
        1
        + target_end
        - crop_start
    )


    final_length = int(
        len(
            cropped_ids
        )
        + N_SPECIAL_TOKENS
    )


    if final_length > MAX_SEQ_LEN:

        raise RuntimeError(
            "El recorte centrado excede MAX_SEQ_LEN: "
            f"{final_length} > {MAX_SEQ_LEN}"
        )


    if not (
        1
        <= wp_start
        <= wp_end
        < final_length - 1
    ):

        raise RuntimeError(
            "Posiciones WordPiece objetivo inválidas "
            "después del recorte."
        )


    return {

        "cropped_ids":
            cropped_ids,

        "wp_start":
            wp_start,

        "wp_end":
            wp_end,

        "target_n_wordpieces":
            target_len,

        "raw_n_wordpieces":
            n_raw_wordpieces,

        "cropped_n_wordpieces":
            int(
                len(
                    cropped_ids
                )
            ),

        "crop_start":
            int(
                crop_start
            ),

        "crop_end":
            int(
                crop_end
            ),

        "was_cropped":
            bool(
                n_raw_wordpieces
                > MAX_CONTENT_WORDPIECES
            ),

        "final_length_with_special_tokens":
            final_length,
    }


# ============================================================
# Auditoría completa
# ============================================================

candidate_pf_for_alignment = pq.ParquetFile(
    CANDIDATES_PATH
)


n_alignment_rows_checked = 0

n_target_missing_before_crop = 0

n_target_too_long = 0

n_rows_requiring_crop = 0

n_naive_target_fully_lost = 0

n_naive_target_partially_truncated = 0

n_centered_crop_failures = 0

max_raw_wordpieces = 0

max_target_wordpieces = 0

failure_examples = []


started_alignment = time.perf_counter()


for batch in tqdm(
    candidate_pf_for_alignment.iter_batches(
        batch_size=TOKENIZER_AUDIT_BATCH_SIZE,
        columns=[
            "word",
            "word_pos_window",
            "context_window",
        ],
    ),
    total=math.ceil(
        candidate_pf_for_alignment.metadata.num_rows
        / TOKENIZER_AUDIT_BATCH_SIZE
    ),
    desc="Auditando recorte WordPiece centrado",
):

    batch_df = batch.to_pandas()


    batch_tokens = [
        str(
            value
        ).split()
        for value
        in batch_df[
            "context_window"
        ].tolist()
    ]


    positions = (
        batch_df[
            "word_pos_window"
        ]
        .astype(
            int
        )
        .tolist()
    )


    # IMPORTANTE:
    # no truncamos aquí porque primero debemos localizar
    # correctamente la palabra objetivo.
    enc = tokenizer(
        batch_tokens,
        is_split_into_words=True,
        add_special_tokens=False,
        truncation=False,
        padding=False,
        return_attention_mask=False,
        verbose=False,
    )


    for batch_idx, word_position in enumerate(
        positions
    ):

        ids = enc[
            "input_ids"
        ][
            batch_idx
        ]

        word_ids = enc.word_ids(
            batch_index=batch_idx
        )


        target_positions = [
            idx
            for idx, word_id
            in enumerate(
                word_ids
            )
            if word_id
            == word_position
        ]


        if not target_positions:

            n_target_missing_before_crop += 1

            if len(
                failure_examples
            ) < 20:

                failure_examples.append(
                    {
                        "problem":
                            "target_missing_before_crop",

                        "word":
                            str(
                                batch_df.iloc[
                                    batch_idx
                                ][
                                    "word"
                                ]
                            ),

                        "word_pos_window":
                            int(
                                word_position
                            ),

                        "context_window":
                            str(
                                batch_df.iloc[
                                    batch_idx
                                ][
                                    "context_window"
                                ]
                            ),
                    }
                )

            continue


        target_start = int(
            min(
                target_positions
            )
        )

        target_end = int(
            max(
                target_positions
            )
        )

        target_len = int(
            target_end
            - target_start
            + 1
        )


        max_raw_wordpieces = max(
            max_raw_wordpieces,
            len(
                ids
            ),
        )

        max_target_wordpieces = max(
            max_target_wordpieces,
            target_len,
        )


        if target_len > MAX_CONTENT_WORDPIECES:

            n_target_too_long += 1
            continue


        if len(
            ids
        ) > MAX_CONTENT_WORDPIECES:

            n_rows_requiring_crop += 1


        # ----------------------------------------------------
        # Diagnóstico de qué habría hecho el truncado antiguo.
        # ----------------------------------------------------

        if (
            target_start
            >= MAX_CONTENT_WORDPIECES
        ):

            n_naive_target_fully_lost += 1

        elif (
            target_end
            >= MAX_CONTENT_WORDPIECES
        ):

            n_naive_target_partially_truncated += 1


        # ----------------------------------------------------
        # Nueva política definitiva.
        # ----------------------------------------------------

        try:

            crop_info = (
                compute_target_centered_wordpiece_crop(
                    input_ids_no_special=ids,
                    word_ids=word_ids,
                    target_word_position=word_position,
                )
            )


            if (
                crop_info[
                    "final_length_with_special_tokens"
                ]
                > MAX_SEQ_LEN
            ):

                raise RuntimeError(
                    "final_length > MAX_SEQ_LEN"
                )


            recovered_target_len = int(
                crop_info[
                    "wp_end"
                ]
                - crop_info[
                    "wp_start"
                ]
                + 1
            )


            if recovered_target_len != target_len:

                raise RuntimeError(
                    "El recorte no conservó todos "
                    "los WordPieces del target."
                )


        except Exception as exc:

            n_centered_crop_failures += 1

            if len(
                failure_examples
            ) < 20:

                failure_examples.append(
                    {
                        "problem":
                            repr(
                                exc
                            ),

                        "word":
                            str(
                                batch_df.iloc[
                                    batch_idx
                                ][
                                    "word"
                                ]
                            ),

                        "word_pos_window":
                            int(
                                word_position
                            ),

                        "context_window":
                            str(
                                batch_df.iloc[
                                    batch_idx
                                ][
                                    "context_window"
                                ]
                            ),
                    }
                )


    n_alignment_rows_checked += int(
        len(
            batch_df
        )
    )


alignment_audit = {

    "input_policy":
        (
            "target_centered_wordpiece_crop"
        ),

    "n_rows_checked":
        int(
            n_alignment_rows_checked
        ),

    "max_seq_len":
        int(
            MAX_SEQ_LEN
        ),

    "n_special_tokens":
        int(
            N_SPECIAL_TOKENS
        ),

    "max_content_wordpieces":
        int(
            MAX_CONTENT_WORDPIECES
        ),

    "n_target_missing_before_crop":
        int(
            n_target_missing_before_crop
        ),

    "n_target_too_long":
        int(
            n_target_too_long
        ),

    "n_rows_requiring_wordpiece_crop":
        int(
            n_rows_requiring_crop
        ),

    "n_naive_target_fully_lost":
        int(
            n_naive_target_fully_lost
        ),

    "n_naive_target_partially_truncated":
        int(
            n_naive_target_partially_truncated
        ),

    "n_centered_crop_failures":
        int(
            n_centered_crop_failures
        ),

    "max_raw_wordpieces":
        int(
            max_raw_wordpieces
        ),

    "max_target_wordpieces":
        int(
            max_target_wordpieces
        ),

    "all_rows_alignable":
        bool(
            n_target_missing_before_crop == 0
            and n_target_too_long == 0
            and n_centered_crop_failures == 0
        ),

    "examples":
        failure_examples,

    "elapsed_seconds":
        float(
            time.perf_counter()
            - started_alignment
        ),
}


save_json(
    alignment_audit,
    TOKENIZER_ALIGNMENT_AUDIT_PATH,
)


print_copy_block(
    "AUDITORÍA WORDPIECE CENTRADA PRE-BERT",
    alignment_audit,
)


if not alignment_audit[
    "all_rows_alignable"
]:

    raise RuntimeError(
        "Persisten ocurrencias no alineables incluso "
        "con recorte WordPiece centrado. "
        "No ejecutes BERT."
    )

Auditando recorte WordPiece centrado:   0%|          | 0/10062 [00:00<?, ?it/s]


PARA COPIAR Y PEGAR — AUDITORÍA WORDPIECE CENTRADA PRE-BERT
{
  "all_rows_alignable": true,
  "elapsed_seconds": 1085.3627992999973,
  "examples": [],
  "input_policy": "target_centered_wordpiece_crop",
  "max_content_wordpieces": 126,
  "max_raw_wordpieces": 593,
  "max_seq_len": 128,
  "max_target_wordpieces": 10,
  "n_centered_crop_failures": 0,
  "n_naive_target_fully_lost": 7288,
  "n_naive_target_partially_truncated": 1274,
  "n_rows_checked": 10302900,
  "n_rows_requiring_wordpiece_crop": 566399,
  "n_special_tokens": 2,
  "n_target_missing_before_crop": 0,
  "n_target_too_long": 0
}
FIN — AUDITORÍA WORDPIECE CENTRADA PRE-BERT


In [44]:
# ============================================================
# 12. Schema final occurrence-level
# ============================================================

embedding_type = pa.list_(
    pa.float32(),
    list_size=hidden_size,
)

FINAL_SCHEMA = pa.schema(
    [
        ("occ_id", pa.int64()),
        ("candidate_id", pa.int64()),
        ("word", pa.string()),
        ("is_target", pa.bool_()),
        ("word_role", pa.string()),
        ("true_class", pa.int32()),
        ("class_label", pa.string()),
        ("is_exp2_word", pa.bool_()),
        ("selection_stage", pa.string()),
        ("source_name", pa.string()),
        ("source_split", pa.string()),
        ("source_row_idx", pa.int64()),
        ("source_doc_key", pa.string()),
        ("doc_split", pa.string()),
        ("word_pos_original", pa.int32()),
        ("word_pos_window", pa.int32()),
        ("wp_start", pa.int32()),
        ("wp_end", pa.int32()),
        ("n_tokens_window", pa.int32()),
        ("context_window", pa.string()),
        ("left_context", pa.string()),
        ("right_context", pa.string()),
        ("text", pa.string()),
        ("embedding", embedding_type),
    ]
)

print(FINAL_SCHEMA)

occ_id: int64
candidate_id: int64
word: string
is_target: bool
word_role: string
true_class: int32
class_label: string
is_exp2_word: bool
selection_stage: string
source_name: string
source_split: string
source_row_idx: int64
source_doc_key: string
doc_split: string
word_pos_original: int32
word_pos_window: int32
wp_start: int32
wp_end: int32
n_tokens_window: int32
context_window: string
left_context: string
right_context: string
text: string
embedding: fixed_size_list<item: float>[768]
  child 0, item: float


In [ ]:
# ============================================================
# 13. Extracción BERT incremental al Parquet final
# ============================================================
#
# El candidate parquet ya está ordenado por palabra.
# El Parquet final conserva ese orden.
#
# La ejecución ABORTA ante cualquier ocurrencia no alineable:
# no se permite terminar con 99 observaciones en una palabra.
# ============================================================

FINAL_METADATA_COLUMNS = [
    "candidate_id",
    "word",
    "is_target",
    "word_role",
    "true_class",
    "class_label",
    "is_exp2_word",
    "selection_stage",
    "source_name",
    "source_split",
    "source_row_idx",
    "source_doc_key",
    "doc_split",
    "word_pos_original",
    "word_pos_window",
    "n_tokens_window",
    "context_window",
    "left_context",
    "right_context",
    "text",
]


def rows_embeddings_to_final_table(
    rows,
    embeddings,
):

    X = np.asarray(
        embeddings,
        dtype=np.float32,
    )

    if X.ndim != 2:
        raise RuntimeError(
            f"Embeddings con forma inválida: {X.shape}"
        )

    if X.shape[1] != hidden_size:
        raise RuntimeError(
            f"Dimensión inválida: {X.shape[1]} != {hidden_size}"
        )

    arrays = []

    for field in FINAL_SCHEMA:

        name = field.name

        if name == "embedding":

            arrays.append(
                pa.FixedSizeListArray.from_arrays(
                    pa.array(
                        X.reshape(-1),
                        type=pa.float32(),
                    ),
                    hidden_size,
                )
            )

        else:

            arrays.append(
                pa.array(
                    [
                        row[name]
                        for row in rows
                    ],
                    type=field.type,
                )
            )

    return pa.Table.from_arrays(
        arrays,
        schema=FINAL_SCHEMA,
    )


def extract_batch_embeddings(
    batch_df,
    occ_id_start,
):

    # ========================================================
    # 1. Contextos y posición de la palabra objetivo
    # ========================================================

    batch_tokens = [
        str(value).split()
        for value in batch_df[
            "context_window"
        ].tolist()
    ]

    batch_word_positions = (
        batch_df[
            "word_pos_window"
        ]
        .astype(int)
        .tolist()
    )


    # ========================================================
    # 2. Tokenizar SIN truncar y SIN tokens especiales
    #
    # Esto permite localizar exactamente todos los WordPieces
    # de la palabra objetivo antes de decidir el recorte.
    # ========================================================

    raw_enc = tokenizer(
        batch_tokens,
        is_split_into_words=True,
        add_special_tokens=False,
        truncation=False,
        padding=False,
        return_attention_mask=False,
        verbose=False,
    )


    prepared_input_ids = []

    wp_starts = []
    wp_ends = []

    crop_was_applied = []

    # ========================================================
    # IDs especiales de BERT
    # ========================================================

    CLS_TOKEN_ID = tokenizer.cls_token_id
    SEP_TOKEN_ID = tokenizer.sep_token_id


    if CLS_TOKEN_ID is None:
        raise RuntimeError(
            "El tokenizer no define cls_token_id."
        )


    if SEP_TOKEN_ID is None:
        raise RuntimeError(
            "El tokenizer no define sep_token_id."
        )


    CLS_TOKEN_ID = int(
        CLS_TOKEN_ID
    )

    SEP_TOKEN_ID = int(
        SEP_TOKEN_ID
    )


    # ========================================================
    # 3. Recorte WordPiece centrado en la palabra objetivo
    # ========================================================

    for batch_idx, word_position in enumerate(
        batch_word_positions
    ):

        raw_ids = raw_enc[
            "input_ids"
        ][
            batch_idx
        ]

        raw_word_ids = raw_enc.word_ids(
            batch_index=batch_idx
        )


        crop_info = (
            compute_target_centered_wordpiece_crop(
                input_ids_no_special=raw_ids,
                word_ids=raw_word_ids,
                target_word_position=word_position,
            )
        )


        cropped_ids = crop_info[
            "cropped_ids"
        ]


        # --------------------------------------------------------
        # Agregar explícitamente [CLS] y [SEP].
        #
        # Evitamos build_inputs_with_special_tokens(), ya que la
        # versión instalada de transformers no expone ese método.
        # --------------------------------------------------------

        model_ids = (
            [CLS_TOKEN_ID]
            + [
                int(token_id)
                for token_id
                in cropped_ids
            ]
            + [SEP_TOKEN_ID]
        )


        if len(
            model_ids
        ) > MAX_SEQ_LEN:

            raise RuntimeError(
                "Secuencia BERT excede MAX_SEQ_LEN "
                "después del recorte centrado: "
                f"{len(model_ids)} > {MAX_SEQ_LEN}"
            )


        prepared_input_ids.append(
            model_ids
        )


        wp_starts.append(
            int(
                crop_info[
                    "wp_start"
                ]
            )
        )


        wp_ends.append(
            int(
                crop_info[
                    "wp_end"
                ]
            )
        )


        crop_was_applied.append(
            bool(
                crop_info[
                    "was_cropped"
                ]
            )
        )


    # ========================================================
    # 4. Padding del batch ya recortado
    # ========================================================

    padded = tokenizer.pad(
        {
            "input_ids":
                prepared_input_ids
        },
        padding=True,
        return_tensors="pt",
    )


    input_ids = (
        padded[
            "input_ids"
        ]
        .to(
            DEVICE,
            non_blocking=True,
        )
    )


    attention_mask = (
        padded[
            "attention_mask"
        ]
        .to(
            DEVICE,
            non_blocking=True,
        )
    )


    batch_size = int(
        input_ids.shape[
            0
        ]
    )


    sequence_length = int(
        input_ids.shape[
            1
        ]
    )


    if sequence_length > MAX_SEQ_LEN:

        raise RuntimeError(
            "El batch padded excede MAX_SEQ_LEN: "
            f"{sequence_length} > {MAX_SEQ_LEN}"
        )


    # ========================================================
    # 5. Máscara de WordPieces de la palabra objetivo
    # ========================================================

    target_mask_np = np.zeros(
        (
            batch_size,
            sequence_length,
        ),
        dtype=np.float32,
    )


    for batch_idx, (
        wp_start,
        wp_end,
    ) in enumerate(
        zip(
            wp_starts,
            wp_ends,
        )
    ):

        if not (
            0
            <= wp_start
            <= wp_end
            < sequence_length
        ):

            raise RuntimeError(
                "Posición WordPiece objetivo inválida: "
                f"batch_idx={batch_idx}, "
                f"wp_start={wp_start}, "
                f"wp_end={wp_end}, "
                f"sequence_length={sequence_length}"
            )


        target_mask_np[
            batch_idx,
            wp_start:
            wp_end + 1,
        ] = 1.0


    # ========================================================
    # 6. Forward BERT
    # ========================================================

    captured_layer_outputs.clear()


    with torch.inference_mode():

        _ = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True,
        )


        missing_layers = [
            layer_idx
            for layer_idx
            in resolved_layer_indices
            if layer_idx
            not in captured_layer_outputs
        ]


        if missing_layers:

            raise RuntimeError(
                "No se capturaron todas las capas BERT "
                f"requeridas: {missing_layers}"
            )


        # ----------------------------------------------------
        # Promedio de las últimas 4 capas
        # ----------------------------------------------------

        token_reps = None


        for layer_idx in resolved_layer_indices:

            layer_tensor = (
                captured_layer_outputs[
                    layer_idx
                ]
            )


            if token_reps is None:

                token_reps = layer_tensor

            else:

                token_reps = (
                    token_reps
                    + layer_tensor
                )


        token_reps = (
            token_reps
            / float(
                len(
                    resolved_layer_indices
                )
            )
        )


        # ----------------------------------------------------
        # Máscara target -> GPU
        # ----------------------------------------------------

        target_mask = (
            torch.from_numpy(
                target_mask_np
            )
            .to(
                DEVICE,
                dtype=token_reps.dtype,
                non_blocking=True,
            )
        )


        denominator = target_mask.sum(
            dim=1,
            keepdim=True,
        )


        if torch.any(
            denominator
            <= 0
        ):

            raise RuntimeError(
                "Máscara WordPiece vacía."
            )


        # ----------------------------------------------------
        # Promedio de los WordPieces de la ocurrencia objetivo
        # ----------------------------------------------------

        pooled = (
            token_reps
            * target_mask.unsqueeze(
                -1
            )
        ).sum(
            dim=1
        ) / denominator


        # ----------------------------------------------------
        # Debe permanecer False en nuestro diseño definitivo
        # ----------------------------------------------------

        if L2_NORMALIZE_EMBEDDINGS:

            pooled = (
                torch.nn.functional.normalize(
                    pooled,
                    p=2,
                    dim=1,
                )
            )


        X = (
            pooled
            .detach()
            .to(
                "cpu",
                dtype=torch.float32,
            )
            .numpy()
        )


    # ========================================================
    # 7. Validaciones numéricas
    # ========================================================

    if X.ndim != 2:

        raise RuntimeError(
            f"Forma de embeddings inesperada: {X.shape}"
        )


    if X.shape[
        1
    ] != hidden_size:

        raise RuntimeError(
            "Dimensión embedding incorrecta: "
            f"{X.shape[1]} != {hidden_size}"
        )


    if X.shape[
        0
    ] != len(
        batch_df
    ):

        raise RuntimeError(
            "BERT cambió el número de observaciones "
            "dentro del batch."
        )


    if not np.isfinite(
        X
    ).all():

        raise RuntimeError(
            "BERT produjo NaN o Inf."
        )


    norms = np.linalg.norm(
        X,
        axis=1,
    )


    if np.any(
        norms
        == 0
    ):

        raise RuntimeError(
            "BERT produjo un vector cero."
        )


    # ========================================================
    # 8. Metadata de las filas finales
    # ========================================================

    rows = []


    for local_idx, (
        _,
        source_row,
    ) in enumerate(
        batch_df.iterrows()
    ):

        row = source_row.to_dict()


        row[
            "occ_id"
        ] = int(
            occ_id_start
            + local_idx
        )


        row[
            "wp_start"
        ] = int(
            wp_starts[
                local_idx
            ]
        )


        row[
            "wp_end"
        ] = int(
            wp_ends[
                local_idx
            ]
        )


        # sample_priority pertenece solo al candidate parquet.
        row.pop(
            "sample_priority",
            None,
        )


        rows.append(
            row
        )


    # ========================================================
    # 9. Liberar referencias grandes
    # ========================================================

    del target_mask
    del pooled
    del token_reps

    del input_ids
    del attention_mask

    del padded
    del raw_enc


    captured_layer_outputs.clear()


    # ========================================================
    # 10. RETORNO — esto faltaba en tu función actual
    # ========================================================

    return (
        rows,
        X,
    )


def extract_all_embeddings():

    if OUT_PARQUET_PATH.exists():

        print(
            "El Parquet final ya existe. "
            "Se validará en las celdas siguientes."
        )

        parquet_file = pq.ParquetFile(
            OUT_PARQUET_PATH
        )

        return {
            "reused_existing_final_parquet": True,
            "n_written": int(
                parquet_file.metadata.num_rows
            ),
            "output_path": str(
                OUT_PARQUET_PATH
            ),
        }

    if OUT_PARQUET_PARTIAL_PATH.exists():

        print(
            "Eliminando Parquet parcial de una ejecución anterior:",
            OUT_PARQUET_PARTIAL_PATH,
        )

        OUT_PARQUET_PARTIAL_PATH.unlink()

    candidate_file = pq.ParquetFile(
        CANDIDATES_PATH
    )

    expected_rows = int(
        len(eligible_words)
        * OCCURRENCES_PER_WORD
    )

    if (
        candidate_file.metadata.num_rows
        != expected_rows
    ):
        raise RuntimeError(
            "El candidate parquet no contiene el número esperado "
            f"de filas: {candidate_file.metadata.num_rows} != {expected_rows}"
        )

    writer = pq.ParquetWriter(
        OUT_PARQUET_PARTIAL_PATH,
        schema=FINAL_SCHEMA,
        compression="zstd",
        compression_level=1,
    )

    output_rows_buffer = []
    output_embeddings_buffer = []

    occ_id = 0
    started = time.perf_counter()

    pbar = tqdm(
        total=expected_rows,
        desc="BERT occurrence embeddings",
        unit="occ",
    )

    try:

        for batch in candidate_file.iter_batches(
            batch_size=BATCH_SIZE_OCCURRENCES
        ):

            batch_df = (
                batch.to_pandas()
            )

            (
                batch_rows,
                batch_embeddings,
            ) = extract_batch_embeddings(
                batch_df,
                occ_id,
            )

            if len(batch_rows) != len(batch_df):
                raise RuntimeError(
                    "La extracción BERT cambió el número de filas."
                )

            output_rows_buffer.extend(
                batch_rows
            )

            output_embeddings_buffer.extend(
                batch_embeddings
            )

            occ_id += len(
                batch_rows
            )

            pbar.update(
                len(batch_rows)
            )

            while (
                len(output_rows_buffer)
                >= FINAL_ROW_GROUP_ROWS
            ):

                rows_chunk = (
                    output_rows_buffer[
                        :FINAL_ROW_GROUP_ROWS
                    ]
                )

                embeddings_chunk = (
                    output_embeddings_buffer[
                        :FINAL_ROW_GROUP_ROWS
                    ]
                )

                table = (
                    rows_embeddings_to_final_table(
                        rows_chunk,
                        embeddings_chunk,
                    )
                )

                writer.write_table(
                    table,
                    row_group_size=(
                        FINAL_ROW_GROUP_ROWS
                    ),
                )

                del output_rows_buffer[
                    :FINAL_ROW_GROUP_ROWS
                ]

                del output_embeddings_buffer[
                    :FINAL_ROW_GROUP_ROWS
                ]

        if output_rows_buffer:

            table = (
                rows_embeddings_to_final_table(
                    output_rows_buffer,
                    output_embeddings_buffer,
                )
            )

            writer.write_table(
                table,
                row_group_size=len(
                    output_rows_buffer
                ),
            )

            output_rows_buffer.clear()
            output_embeddings_buffer.clear()

    finally:

        writer.close()
        pbar.close()

    if occ_id != expected_rows:
        raise RuntimeError(
            f"Se escribieron {occ_id} filas, esperadas {expected_rows}."
        )

    partial_pf = pq.ParquetFile(
        OUT_PARQUET_PARTIAL_PATH
    )

    try:

        partial_n_rows = int(
            partial_pf.metadata.num_rows
        )

    finally:

        partial_pf.close()

        del partial_pf


    gc.collect()


    if partial_n_rows != expected_rows:

        raise RuntimeError(
            "El Parquet parcial cerrado no contiene "
            "el número esperado de filas."
        )


    os.replace(
        OUT_PARQUET_PARTIAL_PATH,
        OUT_PARQUET_PATH,
    )

    return {
        "reused_existing_final_parquet": False,
        "n_written": int(occ_id),
        "n_words": int(
            len(eligible_words)
        ),
        "occurrences_per_word": int(
            OCCURRENCES_PER_WORD
        ),
        "embedding_dim": int(
            hidden_size
        ),
        "l2_normalization": bool(
            L2_NORMALIZE_EMBEDDINGS
        ),
        "selected_layers": list(
            SELECTED_LAYER_IDS
        ),
        "batch_size": int(
            BATCH_SIZE_OCCURRENCES
        ),
        "elapsed_seconds": float(
            time.perf_counter()
            - started
        ),
        "output_path": str(
            OUT_PARQUET_PATH
        ),
    }


extraction_report = (
    extract_all_embeddings()
)

print_copy_block(
    "EXTRACCIÓN BERT",
    extraction_report,
)

Eliminando Parquet parcial de una ejecución anterior: C:\Users\jisaa\Workspace\Tesis\Exp2\Results\contextual_dataset\_contextual_occurrences_definitive.partial.parquet


BERT occurrence embeddings:   0%|          | 0/10302900 [00:00<?, ?occ/s]

PermissionError: [WinError 32] El proceso no tiene acceso al archivo porque está siendo utilizado por otro proceso: 'C:\\Users\\jisaa\\Workspace\\Tesis\\Exp2\\Results\\contextual_dataset\\_contextual_occurrences_definitive.partial.parquet' -> 'C:\\Users\\jisaa\\Workspace\\Tesis\\Exp2\\Results\\contextual_dataset\\contextual_occurrences_definitive.parquet'

In [1]:
# ============================================================
# RESCATE DEL PARQUET BERT COMPLETO
# ============================================================
#
# NO vuelve a ejecutar BERT.
# Valida el archivo parcial, libera handles y lo renombra
# como Parquet definitivo.
# ============================================================

from pathlib import Path
import gc
import os
import sys
import time

import pyarrow.parquet as pq


PARTIAL_PATH = Path(
    r"C:\Users\jisaa\Workspace\Tesis\Exp2\Results\contextual_dataset"
    r"\_contextual_occurrences_definitive.partial.parquet"
)

FINAL_PATH = Path(
    r"C:\Users\jisaa\Workspace\Tesis\Exp2\Results\contextual_dataset"
    r"\contextual_occurrences_definitive.parquet"
)

EXPECTED_ROWS = 10_302_900


# ------------------------------------------------------------
# 1. Verificar que el archivo parcial existe
# ------------------------------------------------------------

if not PARTIAL_PATH.exists():

    raise FileNotFoundError(
        "No existe el Parquet parcial:\n"
        f"{PARTIAL_PATH}"
    )


print(
    "Parquet parcial encontrado:",
    PARTIAL_PATH,
)

print(
    "Tamaño [GiB]:",
    round(
        PARTIAL_PATH.stat().st_size
        / 1024**3,
        3,
    ),
)


# ------------------------------------------------------------
# 2. Liberar referencias al traceback anterior
#
# El traceback del PermissionError puede conservar vivo el
# frame local donde estaba partial_pf.
# ------------------------------------------------------------

for attr in [
    "last_traceback",
    "last_value",
    "last_type",
]:

    if hasattr(
        sys,
        attr,
    ):

        try:
            setattr(
                sys,
                attr,
                None,
            )
        except Exception:
            pass


gc.collect()


# ------------------------------------------------------------
# 3. Validar el archivo parcial y CERRARLO explícitamente
# ------------------------------------------------------------

partial_pf = pq.ParquetFile(
    PARTIAL_PATH
)

try:

    n_rows = int(
        partial_pf.metadata.num_rows
    )

    n_row_groups = int(
        partial_pf.metadata.num_row_groups
    )

    schema_arrow = (
        partial_pf.schema_arrow
    )

finally:

    partial_pf.close()

    del partial_pf


gc.collect()


print(
    "Filas encontradas:",
    f"{n_rows:,}",
)

print(
    "Row groups:",
    f"{n_row_groups:,}",
)

print(
    "Columnas:",
    schema_arrow.names,
)


if n_rows != EXPECTED_ROWS:

    raise RuntimeError(
        "El Parquet parcial NO contiene todas las filas. "
        f"{n_rows:,} != {EXPECTED_ROWS:,}"
    )


if "embedding" not in schema_arrow.names:

    raise RuntimeError(
        "El Parquet parcial no contiene la columna embedding."
    )


print(
    "\nOK — el Parquet parcial contiene exactamente "
    f"{EXPECTED_ROWS:,} filas."
)


# ------------------------------------------------------------
# 4. El destino no debería existir todavía
# ------------------------------------------------------------

if FINAL_PATH.exists():

    raise RuntimeError(
        "Ya existe contextual_occurrences_definitive.parquet. "
        "No se sobrescribirá automáticamente."
    )


# ------------------------------------------------------------
# 5. Intentar renombrar
# ------------------------------------------------------------

rename_success = False

last_error = None


for attempt in range(
    1,
    6,
):

    try:

        gc.collect()

        os.replace(
            PARTIAL_PATH,
            FINAL_PATH,
        )

        rename_success = True

        break

    except PermissionError as exc:

        last_error = exc

        print(
            f"Intento {attempt}/5: "
            "el archivo sigue bloqueado."
        )

        gc.collect()

        time.sleep(
            2
        )


if not rename_success:

    raise RuntimeError(
        "El Parquet está completo, pero Windows continúa "
        "manteniendo un handle abierto. "
        "NO vuelvas a ejecutar BERT. "
        "Reinicia el kernel y ejecuta únicamente la celda "
        "de rescate nuevamente."
    ) from last_error


# ------------------------------------------------------------
# 6. Validar el archivo YA renombrado
# ------------------------------------------------------------

final_pf = pq.ParquetFile(
    FINAL_PATH
)

try:

    final_n_rows = int(
        final_pf.metadata.num_rows
    )

    final_n_row_groups = int(
        final_pf.metadata.num_row_groups
    )

finally:

    final_pf.close()

    del final_pf


gc.collect()


if final_n_rows != EXPECTED_ROWS:

    raise RuntimeError(
        "El archivo definitivo tiene un número inesperado "
        f"de filas: {final_n_rows:,}"
    )


print(
    "\n"
    + "=" * 80
)

print(
    "RESCATE COMPLETADO CORRECTAMENTE"
)

print(
    "=" * 80
)

print(
    "Archivo definitivo:",
    FINAL_PATH,
)

print(
    "Filas:",
    f"{final_n_rows:,}",
)

print(
    "Row groups:",
    f"{final_n_row_groups:,}",
)

print(
    "Parquet parcial sigue existiendo:",
    PARTIAL_PATH.exists(),
)

print(
    "Parquet definitivo existe:",
    FINAL_PATH.exists(),
)

Parquet parcial encontrado: C:\Users\jisaa\Workspace\Tesis\Exp2\Results\contextual_dataset\_contextual_occurrences_definitive.partial.parquet
Tamaño [GiB]: 38.446
Filas encontradas: 10,302,900
Row groups: 10,303
Columnas: ['occ_id', 'candidate_id', 'word', 'is_target', 'word_role', 'true_class', 'class_label', 'is_exp2_word', 'selection_stage', 'source_name', 'source_split', 'source_row_idx', 'source_doc_key', 'doc_split', 'word_pos_original', 'word_pos_window', 'wp_start', 'wp_end', 'n_tokens_window', 'context_window', 'left_context', 'right_context', 'text', 'embedding']

OK — el Parquet parcial contiene exactamente 10,302,900 filas.

RESCATE COMPLETADO CORRECTAMENTE
Archivo definitivo: C:\Users\jisaa\Workspace\Tesis\Exp2\Results\contextual_dataset\contextual_occurrences_definitive.parquet
Filas: 10,302,900
Row groups: 10,303
Parquet parcial sigue existiendo: False
Parquet definitivo existe: True


In [2]:
# ============================================================
# 14. Construir índice word -> row group
# ============================================================
#
# Invariantes:
# - archivo ordenado por word;
# - 100 ocurrencias exactas por word;
# - row groups de 1000 filas;
# - cada row group contiene 10 palabras completas.
# ============================================================

final_pf = pq.ParquetFile(
    OUT_PARQUET_PATH
)

expected_rows = int(
    len(eligible_words)
    * OCCURRENCES_PER_WORD
)

if (
    final_pf.metadata.num_rows
    != expected_rows
):
    raise RuntimeError(
        "No se puede construir el índice: "
        "número de filas final incorrecto."
    )

row_group_index_rows = []

for word_index, word in enumerate(
    eligible_words
):

    row_start = int(
        word_index
        * OCCURRENCES_PER_WORD
    )

    row_end = int(
        row_start
        + OCCURRENCES_PER_WORD
    )

    row_group_id = int(
        row_start
        // FINAL_ROW_GROUP_ROWS
    )

    row_group_index_rows.append(
        {
            "word": word,
            "word_index": int(
                word_index
            ),
            "n_occurrences": int(
                OCCURRENCES_PER_WORD
            ),
            "row_start": row_start,
            "row_end_exclusive": row_end,
            "row_group_id": row_group_id,
        }
    )

word_row_group_index_df = pd.DataFrame(
    row_group_index_rows
)

word_row_group_index_df.to_csv(
    ROW_GROUP_INDEX_PATH,
    index=False,
    encoding="utf-8",
)

expected_n_row_groups = int(
    math.ceil(
        expected_rows
        / FINAL_ROW_GROUP_ROWS
    )
)

if (
    final_pf.metadata.num_row_groups
    != expected_n_row_groups
):
    raise RuntimeError(
        "El número de row groups del Parquet no coincide "
        "con el diseño de indexación. "
        f"{final_pf.metadata.num_row_groups} != {expected_n_row_groups}"
    )

print(
    "Índice guardado:",
    ROW_GROUP_INDEX_PATH,
)

print(
    "N row groups:",
    final_pf.metadata.num_row_groups,
)

display(
    word_row_group_index_df.head(20)
)

NameError: name 'OUT_PARQUET_PATH' is not defined

In [4]:
# ============================================================
# Mostrar resultado de la celda 14 después del reinicio
# ============================================================

import json

print(
    "\n"
    + "=" * 90
)

print(
    "PARA COPIAR Y PEGAR — ÍNDICE WORD ROW-GROUP"
)

print(
    "=" * 90
)

print(
    json.dumps(
        index_report,
        ensure_ascii=False,
        indent=2,
        sort_keys=True,
    )
)

print(
    "=" * 90
)

print(
    "FIN — ÍNDICE WORD ROW-GROUP"
)

print(
    "=" * 90
)

display(
    word_row_group_index_df.head(20)
)


PARA COPIAR Y PEGAR — ÍNDICE WORD ROW-GROUP
{
  "first_word": "aaa",
  "index_valid": true,
  "last_row_group_rows": 900,
  "last_word": "zzap",
  "n_row_groups": 10303,
  "n_rows": 10302900,
  "n_words": 103029,
  "nan_absent": true,
  "occurrences_per_word": 100,
  "parquet_path": "C:\\Users\\jisaa\\Workspace\\Tesis\\Exp2\\Results\\contextual_dataset\\contextual_occurrences_definitive.parquet",
  "row_group_index_path": "C:\\Users\\jisaa\\Workspace\\Tesis\\Exp2\\Results\\contextual_dataset\\word_row_group_index.csv",
  "row_group_rows": 1000
}
FIN — ÍNDICE WORD ROW-GROUP


,word,word_index,n_occurrences,row_start,row_end_exclusive,row_group_id
0,aaa,0,100,0,100,0
1,aaaa,1,100,100,200,0
2,aaaaa,2,100,200,300,0
3,aaas,3,100,300,400,0
4,aab,4,100,400,500,0
5,aac,5,100,500,600,0
6,aachen,6,100,600,700,0
7,aacs,7,100,700,800,0
8,aacsb,8,100,800,900,0
9,aacta,9,100,900,1000,0


In [5]:
# ============================================================
# 15. VALIDACIÓN FUERTE FINAL DEL DATASET CONTEXTUAL
#     VERSIÓN AUTOSUFICIENTE POST-REINICIO
# ============================================================

from pathlib import Path
from collections import Counter
import json
import math
import time

import numpy as np
import pandas as pd
import pyarrow.parquet as pq


# ============================================================
# 1. Configuración cerrada
# ============================================================

OUT_DIR = Path(
    r"C:\Users\jisaa\Workspace\Tesis\Exp2\Results\contextual_dataset"
)

OUT_PARQUET_PATH = (
    OUT_DIR
    / "contextual_occurrences_definitive.parquet"
)

FINAL_ELIGIBLE_VOCAB_PATH = (
    OUT_DIR
    / "eligible_vocabulary_min100.csv"
)

ROW_GROUP_INDEX_PATH = (
    OUT_DIR
    / "word_row_group_index.csv"
)

VALIDATION_REPORT_PATH = (
    OUT_DIR
    / "validation_report.json"
)

WORD_COUNTS_PATH = (
    OUT_DIR
    / "word_counts.csv"
)

CATEGORY_COVERAGE_PATH = (
    OUT_DIR
    / "category_words_coverage.csv"
)


OCCURRENCES_PER_WORD = 100

EXPECTED_N_WORDS = 103_029

EXPECTED_N_ROWS = (
    EXPECTED_N_WORDS
    * OCCURRENCES_PER_WORD
)

EXPECTED_EMBEDDING_DIM = 768

EXPECTED_ROW_GROUP_ROWS = 1000


CATEGORY_WORDS = [
    "apple",
    "bicycle",
    "bread",
    "canoe",
    "chair",
    "jellyfish",
    "screwdriver",
    "skunk",
]


# ============================================================
# 2. Verificar archivos
# ============================================================

for required_path in [
    OUT_PARQUET_PATH,
    FINAL_ELIGIBLE_VOCAB_PATH,
    ROW_GROUP_INDEX_PATH,
]:

    if not required_path.exists():

        raise FileNotFoundError(
            f"No existe el archivo requerido:\n{required_path}"
        )


# ============================================================
# 3. Recuperar vocabulario definitivo
# ============================================================

eligible_vocab_df = pd.read_csv(
    FINAL_ELIGIBLE_VOCAB_PATH,
    keep_default_na=False,
    na_filter=False,
)


if "word" not in eligible_vocab_df.columns:

    raise RuntimeError(
        "eligible_vocabulary_min100.csv "
        "no contiene columna 'word'."
    )


eligible_vocab_df["word"] = (
    eligible_vocab_df["word"]
    .astype(str)
    .str.lower()
    .str.strip()
)


eligible_vocab_df = (
    eligible_vocab_df
    .drop_duplicates(
        subset=["word"]
    )
    .sort_values("word")
    .reset_index(drop=True)
)


eligible_words = (
    eligible_vocab_df["word"]
    .tolist()
)

eligible_word_set = set(
    eligible_words
)


if len(eligible_words) != EXPECTED_N_WORDS:

    raise RuntimeError(
        "Número inesperado de palabras elegibles: "
        f"{len(eligible_words):,} "
        f"!= {EXPECTED_N_WORDS:,}"
    )


if "nan" in eligible_word_set:

    raise RuntimeError(
        "'nan' reapareció en el vocabulario definitivo."
    )


missing_category_words = sorted(
    set(CATEGORY_WORDS)
    - eligible_word_set
)


if missing_category_words:

    raise RuntimeError(
        "Faltan CategoryWords en el vocabulario: "
        f"{missing_category_words}"
    )


# ============================================================
# 4. Abrir Parquet y validar schema general
# ============================================================

pf = pq.ParquetFile(
    OUT_PARQUET_PATH
)


try:

    n_rows = int(
        pf.metadata.num_rows
    )

    n_row_groups = int(
        pf.metadata.num_row_groups
    )

    schema_arrow = (
        pf.schema_arrow
    )


    if n_rows != EXPECTED_N_ROWS:

        raise RuntimeError(
            "Número incorrecto de filas: "
            f"{n_rows:,} != {EXPECTED_N_ROWS:,}"
        )


    required_columns = [
        "occ_id",
        "candidate_id",
        "word",
        "is_target",
        "word_role",
        "true_class",
        "class_label",
        "is_exp2_word",
        "selection_stage",
        "source_name",
        "source_split",
        "source_row_idx",
        "source_doc_key",
        "doc_split",
        "word_pos_original",
        "word_pos_window",
        "wp_start",
        "wp_end",
        "n_tokens_window",
        "context_window",
        "left_context",
        "right_context",
        "text",
        "embedding",
    ]


    missing_columns = [
        column
        for column in required_columns
        if column not in schema_arrow.names
    ]


    if missing_columns:

        raise RuntimeError(
            "Faltan columnas en el Parquet: "
            f"{missing_columns}"
        )


    # ========================================================
    # 5. Auditoría streaming de words + orden
    # ========================================================

    word_counts = Counter()

    previous_word = None

    word_order_violations = 0

    n_word_rows_checked = 0


    started_words = time.perf_counter()


    for batch in pf.iter_batches(
        batch_size=200_000,
        columns=[
            "word",
        ],
    ):

        words = (
            batch.column(0)
            .to_pylist()
        )


        word_counts.update(
            words
        )


        for word in words:

            if (
                previous_word is not None
                and word < previous_word
            ):

                word_order_violations += 1

            previous_word = word


        n_word_rows_checked += len(
            words
        )


    word_scan_seconds = float(
        time.perf_counter()
        - started_words
    )


    if n_word_rows_checked != EXPECTED_N_ROWS:

        raise RuntimeError(
            "No se recorrieron todas las filas "
            "durante auditoría de words."
        )


    if word_order_violations != 0:

        raise RuntimeError(
            "El Parquet no está ordenado por word. "
            f"Violaciones={word_order_violations}"
        )


    if len(word_counts) != EXPECTED_N_WORDS:

        raise RuntimeError(
            "Número incorrecto de palabras únicas: "
            f"{len(word_counts):,} "
            f"!= {EXPECTED_N_WORDS:,}"
        )


    if set(word_counts) != eligible_word_set:

        missing_from_parquet = sorted(
            eligible_word_set
            - set(word_counts)
        )

        unexpected_in_parquet = sorted(
            set(word_counts)
            - eligible_word_set
        )

        raise RuntimeError(
            "El vocabulario del Parquet no coincide "
            "con eligible_vocabulary_min100.csv. "
            f"Missing={missing_from_parquet[:20]}, "
            f"Unexpected={unexpected_in_parquet[:20]}"
        )


    bad_word_counts = {
        word:
            count
        for word, count
        in word_counts.items()
        if count != OCCURRENCES_PER_WORD
    }


    if bad_word_counts:

        raise RuntimeError(
            "Existen palabras con conteo distinto de 100. "
            f"Ejemplos={list(bad_word_counts.items())[:20]}"
        )


    # ========================================================
    # 6. Validar ocurrencias de las 8 CategoryWords
    # ========================================================

    category_coverage_df = pd.DataFrame(
        [
            {
                "category_word":
                    word,

                "n_occurrences":
                    int(
                        word_counts[word]
                    ),

                "exactly_100":
                    bool(
                        word_counts[word]
                        == OCCURRENCES_PER_WORD
                    ),
            }
            for word in CATEGORY_WORDS
        ]
    )


    if not category_coverage_df[
        "exactly_100"
    ].all():

        raise RuntimeError(
            "Alguna CategoryWord no tiene exactamente "
            "100 ocurrencias."
        )


    # ========================================================
    # 7. Auditoría NUMÉRICA de todos los embeddings
    # ========================================================

    n_embedding_rows_checked = 0

    n_non_finite_embeddings = 0

    n_zero_embeddings = 0

    embedding_dim = None

    global_norm_min = float(
        "inf"
    )

    global_norm_max = float(
        "-inf"
    )

    norm_sum = 0.0

    norm_sq_sum = 0.0


    started_embeddings = time.perf_counter()


    for batch in pf.iter_batches(
        batch_size=4096,
        columns=[
            "embedding",
        ],
    ):

        embedding_array = (
            batch.column(0)
        )


        X = (
            embedding_array
            .values
            .to_numpy(
                zero_copy_only=False
            )
            .reshape(
                len(batch),
                -1,
            )
            .astype(
                np.float32,
                copy=False,
            )
        )


        if embedding_dim is None:

            embedding_dim = int(
                X.shape[1]
            )


        if X.shape[1] != EXPECTED_EMBEDDING_DIM:

            raise RuntimeError(
                "Dimensión embedding incorrecta: "
                f"{X.shape[1]} "
                f"!= {EXPECTED_EMBEDDING_DIM}"
            )


        finite_mask = np.isfinite(
            X
        ).all(
            axis=1
        )


        n_non_finite_embeddings += int(
            (
                ~finite_mask
            ).sum()
        )


        zero_mask = np.all(
            X == 0,
            axis=1,
        )


        n_zero_embeddings += int(
            zero_mask.sum()
        )


        norms = np.linalg.norm(
            X,
            axis=1,
        )


        global_norm_min = min(
            global_norm_min,
            float(
                norms.min()
            ),
        )

        global_norm_max = max(
            global_norm_max,
            float(
                norms.max()
            ),
        )


        norm_sum += float(
            norms.astype(
                np.float64
            ).sum()
        )


        norm_sq_sum += float(
            np.square(
                norms.astype(
                    np.float64
                )
            ).sum()
        )


        n_embedding_rows_checked += int(
            X.shape[0]
        )


    embedding_scan_seconds = float(
        time.perf_counter()
        - started_embeddings
    )


    if (
        n_embedding_rows_checked
        != EXPECTED_N_ROWS
    ):

        raise RuntimeError(
            "No se auditaron todos los embeddings: "
            f"{n_embedding_rows_checked:,} "
            f"!= {EXPECTED_N_ROWS:,}"
        )


    if n_non_finite_embeddings != 0:

        raise RuntimeError(
            "Se detectaron embeddings con NaN/Inf: "
            f"{n_non_finite_embeddings}"
        )


    if n_zero_embeddings != 0:

        raise RuntimeError(
            "Se detectaron embeddings cero: "
            f"{n_zero_embeddings}"
        )


    norm_mean = (
        norm_sum
        / n_embedding_rows_checked
    )


    norm_variance = (
        norm_sq_sum
        / n_embedding_rows_checked
        - norm_mean**2
    )


    norm_std = math.sqrt(
        max(
            0.0,
            norm_variance,
        )
    )


finally:

    pf.close()

    del pf


# ============================================================
# 8. Validar índice row-group guardado
# ============================================================

row_group_index_df = pd.read_csv(
    ROW_GROUP_INDEX_PATH,
    keep_default_na=False,
    na_filter=False,
)


required_index_columns = [
    "word",
    "word_index",
    "n_occurrences",
    "row_start",
    "row_end_exclusive",
    "row_group_id",
]


missing_index_columns = [
    column
    for column in required_index_columns
    if column not in row_group_index_df.columns
]


if missing_index_columns:

    raise RuntimeError(
        "Faltan columnas en word_row_group_index.csv: "
        f"{missing_index_columns}"
    )


if len(
    row_group_index_df
) != EXPECTED_N_WORDS:

    raise RuntimeError(
        "Número incorrecto de filas en "
        "word_row_group_index.csv."
    )


if (
    row_group_index_df["word"]
    .astype(str)
    .tolist()
    != eligible_words
):

    raise RuntimeError(
        "El orden de words del índice no coincide "
        "con el vocabulario definitivo."
    )


if (
    row_group_index_df[
        "n_occurrences"
    ]
    .astype(int)
    .ne(
        OCCURRENCES_PER_WORD
    )
    .any()
):

    raise RuntimeError(
        "El índice contiene n_occurrences != 100."
    )


# ============================================================
# 9. Guardar conteos y cobertura
# ============================================================

word_counts_df = pd.DataFrame(
    {
        "word":
            eligible_words,

        "n_occurrences":
            [
                int(
                    word_counts[word]
                )
                for word in eligible_words
            ],
    }
)


word_counts_df.to_csv(
    WORD_COUNTS_PATH,
    index=False,
    encoding="utf-8",
)


category_coverage_df.to_csv(
    CATEGORY_COVERAGE_PATH,
    index=False,
    encoding="utf-8",
)


# ============================================================
# 10. Reporte final
# ============================================================

validation_report = {

    "status":
        "VALID",

    "parquet_path":
        str(
            OUT_PARQUET_PATH
        ),

    "parquet_size_gib":
        float(
            OUT_PARQUET_PATH.stat().st_size
            / 1024**3
        ),

    "n_rows":
        int(
            EXPECTED_N_ROWS
        ),

    "n_unique_words":
        int(
            EXPECTED_N_WORDS
        ),

    "occurrences_per_word":
        int(
            OCCURRENCES_PER_WORD
        ),

    "all_words_exactly_100":
        bool(
            not bad_word_counts
        ),

    "all_8_category_words_exactly_100":
        bool(
            category_coverage_df[
                "exactly_100"
            ].all()
        ),

    "category_words":
        CATEGORY_WORDS,

    "nan_absent":
        bool(
            "nan"
            not in eligible_word_set
        ),

    "word_order_violations":
        int(
            word_order_violations
        ),

    "embedding_dim":
        int(
            embedding_dim
        ),

    "embedding_dtype":
        "float32",

    "l2_normalized":
        False,

    "n_embedding_rows_checked":
        int(
            n_embedding_rows_checked
        ),

    "n_non_finite_embeddings":
        int(
            n_non_finite_embeddings
        ),

    "n_zero_embeddings":
        int(
            n_zero_embeddings
        ),

    "embedding_norm_min":
        float(
            global_norm_min
        ),

    "embedding_norm_max":
        float(
            global_norm_max
        ),

    "embedding_norm_mean":
        float(
            norm_mean
        ),

    "embedding_norm_std":
        float(
            norm_std
        ),

    "n_row_groups":
        int(
            n_row_groups
        ),

    "row_group_rows_target":
        int(
            EXPECTED_ROW_GROUP_ROWS
        ),

    "row_group_index_valid":
        True,

    "word_scan_seconds":
        float(
            word_scan_seconds
        ),

    "embedding_scan_seconds":
        float(
            embedding_scan_seconds
        ),
}


with open(
    VALIDATION_REPORT_PATH,
    "w",
    encoding="utf-8",
) as handle:

    json.dump(
        validation_report,
        handle,
        ensure_ascii=False,
        indent=2,
        sort_keys=True,
    )


# ============================================================
# 11. Salida copiable
# ============================================================

print(
    "\n"
    + "=" * 90
)

print(
    "PARA COPIAR Y PEGAR — "
    "VALIDACIÓN FINAL DATASET CONTEXTUAL"
)

print(
    "=" * 90
)

print(
    json.dumps(
        validation_report,
        ensure_ascii=False,
        indent=2,
        sort_keys=True,
    )
)

print(
    "=" * 90
)

print(
    "FIN — VALIDACIÓN FINAL DATASET CONTEXTUAL"
)

print(
    "=" * 90
)


PARA COPIAR Y PEGAR — VALIDACIÓN FINAL DATASET CONTEXTUAL
{
  "all_8_category_words_exactly_100": true,
  "all_words_exactly_100": true,
  "category_words": [
    "apple",
    "bicycle",
    "bread",
    "canoe",
    "chair",
    "jellyfish",
    "screwdriver",
    "skunk"
  ],
  "embedding_dim": 768,
  "embedding_dtype": "float32",
  "embedding_norm_max": 25.098310470581055,
  "embedding_norm_mean": 16.451971886219336,
  "embedding_norm_min": 7.250402450561523,
  "embedding_norm_std": 1.4067481587434703,
  "embedding_scan_seconds": 125.19874329998856,
  "l2_normalized": false,
  "n_embedding_rows_checked": 10302900,
  "n_non_finite_embeddings": 0,
  "n_row_groups": 10303,
  "n_rows": 10302900,
  "n_unique_words": 103029,
  "n_zero_embeddings": 0,
  "nan_absent": true,
  "occurrences_per_word": 100,
  "parquet_path": "C:\\Users\\jisaa\\Workspace\\Tesis\\Exp2\\Results\\contextual_dataset\\contextual_occurrences_definitive.parquet",
  "parquet_size_gib": 38.44596943538636,
  "row_group_

In [8]:
# ============================================================
# 16. METADATOS DEFINITIVOS DEL DATASET CONTEXTUAL
#     VERSIÓN AUTOSUFICIENTE POST-REINICIO
# ============================================================
#
# Esta celda NO modifica embeddings ni vuelve a ejecutar BERT.
#
# Registra exactamente la configuración que produjo:
# contextual_occurrences_definitive.parquet
# ============================================================

from pathlib import Path
import json

import pandas as pd
import pyarrow.parquet as pq


# ============================================================
# 1. Rutas
# ============================================================

PROJECT_ROOT = Path(
    r"C:\Users\jisaa\Workspace\Tesis"
)

OUT_DIR = (
    PROJECT_ROOT
    / "Exp2"
    / "Results"
    / "contextual_dataset"
)


OUT_PARQUET_PATH = (
    OUT_DIR
    / "contextual_occurrences_definitive.parquet"
)

FINAL_ELIGIBLE_VOCAB_PATH = (
    OUT_DIR
    / "eligible_vocabulary_min100.csv"
)

ROW_GROUP_INDEX_PATH = (
    OUT_DIR
    / "word_row_group_index.csv"
)

SELECTION_REPORT_PATH = (
    OUT_DIR
    / "selection_report.json"
)

VALIDATION_REPORT_PATH = (
    OUT_DIR
    / "validation_report.json"
)

TOKENIZER_ALIGNMENT_AUDIT_PATH = (
    OUT_DIR
    / "tokenizer_alignment_audit.json"
)

META_PATH = (
    OUT_DIR
    / "contextual_dataset_meta.json"
)

GENERATION_CONFIG_PATH = (
    OUT_DIR
    / "generation_config.json"
)


# ============================================================
# 2. Constantes metodológicas DEFINITIVAS
# ============================================================

MODEL_NAME = "bert-base-uncased"

EMBEDDING_DIM = 768

SELECTED_LAYER_IDS = [
    -4,
    -3,
    -2,
    -1,
]

LAYER_AGGREGATION = "mean"

TARGET_WORDPIECE_AGGREGATION = "mean"

L2_NORMALIZE_EMBEDDINGS = False


OCCURRENCES_PER_WORD = 100

EXPECTED_N_WORDS = 103_029

EXPECTED_N_ROWS = 10_302_900


WINDOW_TOKENS_EACH_SIDE = 40

MAX_SEQ_LEN = 128

N_SPECIAL_TOKENS = 2

MAX_CONTENT_WORDPIECES = (
    MAX_SEQ_LEN
    - N_SPECIAL_TOKENS
)


FINAL_ROW_GROUP_ROWS = 1000


CONTEXTUAL_DATASET_SCHEMA_VERSION = (
    "ctx_v3_commonpool_equal100_rawbert"
)

CONFIG_NAME = (
    "ctx_v3_bert_base_commonpool_equal100"
)


EXPECTED_SELECTION_SHA256 = (
    "e130061a093ff98c3265cda041a3a86e"
    "77c6204c3509e0aa0a56405af87af087"
)


CATEGORY_WORDS = [
    "apple",
    "bicycle",
    "bread",
    "canoe",
    "chair",
    "jellyfish",
    "screwdriver",
    "skunk",
]


# ============================================================
# 3. Helpers locales
# ============================================================

def load_json_required(path):

    if not path.exists():

        raise FileNotFoundError(
            "No existe el JSON requerido:\n"
            f"{path}"
        )

    with open(
        path,
        "r",
        encoding="utf-8",
    ) as handle:

        return json.load(
            handle
        )


def save_json(
    path,
    payload,
):

    with open(
        path,
        "w",
        encoding="utf-8",
    ) as handle:

        json.dump(
            payload,
            handle,
            ensure_ascii=False,
            indent=2,
            sort_keys=True,
        )


# ============================================================
# 4. Recuperar reportes ya generados
# ============================================================

selection_report = load_json_required(
    SELECTION_REPORT_PATH
)

validation_report = load_json_required(
    VALIDATION_REPORT_PATH
)

alignment_audit = load_json_required(
    TOKENIZER_ALIGNMENT_AUDIT_PATH
)


# ============================================================
# 5. Validar fingerprint de selección
# ============================================================

selection_sha256 = (
    selection_report.get(
        "selection_sha256_final"
    )
    or selection_report.get(
        "selection_sha256"
    )
)


if selection_sha256 is None:

    raise RuntimeError(
        "selection_report.json no contiene "
        "selection_sha256_final ni selection_sha256."
    )


if selection_sha256 != EXPECTED_SELECTION_SHA256:

    raise RuntimeError(
        "Fingerprint de selección inesperado:\n"
        f"{selection_sha256}\n"
        "Esperado:\n"
        f"{EXPECTED_SELECTION_SHA256}"
    )


# ============================================================
# 6. Validar reporte final
# ============================================================

if validation_report.get(
    "status"
) != "VALID":

    raise RuntimeError(
        "validation_report.json no tiene status='VALID'."
    )


if int(
    validation_report[
        "n_rows"
    ]
) != EXPECTED_N_ROWS:

    raise RuntimeError(
        "Número final de filas inesperado."
    )


if int(
    validation_report[
        "n_unique_words"
    ]
) != EXPECTED_N_WORDS:

    raise RuntimeError(
        "Número final de palabras inesperado."
    )


if not bool(
    validation_report[
        "all_words_exactly_100"
    ]
):

    raise RuntimeError(
        "No todas las palabras tienen exactamente "
        "100 ocurrencias."
    )


if not bool(
    validation_report[
        "all_8_category_words_exactly_100"
    ]
):

    raise RuntimeError(
        "Las CategoryWords no satisfacen "
        "la cobertura definitiva."
    )


if int(
    validation_report[
        "n_non_finite_embeddings"
    ]
) != 0:

    raise RuntimeError(
        "El reporte registra embeddings no finitos."
    )


if int(
    validation_report[
        "n_zero_embeddings"
    ]
) != 0:

    raise RuntimeError(
        "El reporte registra embeddings cero."
    )


# ============================================================
# 7. Validar auditoría WordPiece centrada
# ============================================================

if not bool(
    alignment_audit.get(
        "all_rows_alignable",
        False,
    )
):

    raise RuntimeError(
        "La auditoría WordPiece no indica "
        "all_rows_alignable=True."
    )


if int(
    alignment_audit.get(
        "n_centered_crop_failures",
        0,
    )
) != 0:

    raise RuntimeError(
        "La auditoría registra fallos del "
        "recorte WordPiece centrado."
    )


if int(
    alignment_audit.get(
        "n_target_missing_before_crop",
        0,
    )
) != 0:

    raise RuntimeError(
        "La auditoría registra targets ausentes "
        "antes del recorte."
    )


if int(
    alignment_audit.get(
        "n_target_too_long",
        0,
    )
) != 0:

    raise RuntimeError(
        "La auditoría registra targets mayores "
        "que el presupuesto WordPiece."
    )


# ============================================================
# 8. Recuperar vocabulario e índice
# ============================================================

eligible_vocab_df = pd.read_csv(
    FINAL_ELIGIBLE_VOCAB_PATH,
    keep_default_na=False,
    na_filter=False,
)


eligible_vocab_df["word"] = (
    eligible_vocab_df[
        "word"
    ]
    .astype(str)
    .str.lower()
    .str.strip()
)


eligible_vocab_df = (
    eligible_vocab_df
    .drop_duplicates(
        subset=[
            "word",
        ]
    )
    .sort_values(
        "word"
    )
    .reset_index(
        drop=True
    )
)


eligible_words = (
    eligible_vocab_df[
        "word"
    ]
    .tolist()
)


if len(
    eligible_words
) != EXPECTED_N_WORDS:

    raise RuntimeError(
        "eligible_vocabulary_min100.csv "
        "no contiene exactamente 103029 palabras."
    )


if "nan" in eligible_words:

    raise RuntimeError(
        "'nan' reapareció en el vocabulario."
    )


if "null" not in eligible_words:

    raise RuntimeError(
        "El literal válido 'null' desapareció "
        "inesperadamente del vocabulario."
    )


row_group_index_df = pd.read_csv(
    ROW_GROUP_INDEX_PATH,
    keep_default_na=False,
    na_filter=False,
)


if len(
    row_group_index_df
) != EXPECTED_N_WORDS:

    raise RuntimeError(
        "word_row_group_index.csv tiene "
        "un número inesperado de palabras."
    )


# ============================================================
# 9. Recuperar estructura física del Parquet
# ============================================================

pf = pq.ParquetFile(
    OUT_PARQUET_PATH
)


try:

    parquet_n_rows = int(
        pf.metadata.num_rows
    )

    parquet_n_row_groups = int(
        pf.metadata.num_row_groups
    )

finally:

    pf.close()

    del pf


if parquet_n_rows != EXPECTED_N_ROWS:

    raise RuntimeError(
        "El Parquet definitivo no contiene "
        "10.302.900 filas."
    )


# ============================================================
# 10. Configuración de generación
# ============================================================

generation_config = {

    "config_name":
        CONFIG_NAME,

    "schema_version":
        CONTEXTUAL_DATASET_SCHEMA_VERSION,


    # --------------------------------------------------------
    # Modelo contextual
    # --------------------------------------------------------

    "model": {

        "name":
            MODEL_NAME,

        "architecture":
            "BERT-base uncased",

        "embedding_dim":
            EMBEDDING_DIM,

        "selected_layers":
            SELECTED_LAYER_IDS,

        "layer_aggregation":
            LAYER_AGGREGATION,

        "target_wordpiece_aggregation":
            TARGET_WORDPIECE_AGGREGATION,

        "l2_normalized":
            L2_NORMALIZE_EMBEDDINGS,

        "embedding_dtype":
            "float32",
    },


    # --------------------------------------------------------
    # Contexto léxico
    # --------------------------------------------------------

    "context": {

        "lexical_tokens_each_side":
            WINDOW_TOKENS_EACH_SIDE,

        "context_window_definition":
            "target ±40 lexical whitespace tokens",

        "stopwords_retained_inside_context":
            True,

        "context_is_selection_unit":
            True,
    },


    # --------------------------------------------------------
    # Política WordPiece / BERT
    # --------------------------------------------------------

    "bert_input_policy": {

        "policy":
            "target_centered_wordpiece_crop",

        "max_seq_len":
            MAX_SEQ_LEN,

        "n_special_tokens":
            N_SPECIAL_TOKENS,

        "max_content_wordpieces":
            MAX_CONTENT_WORDPIECES,

        "special_tokens":
            [
                "[CLS]",
                "[SEP]",
            ],

        "preserve_all_target_wordpieces":
            True,

        "standard_right_truncation_used":
            False,

        "crop_applied_before_model_forward":
            True,

        "target_positions_recorded_as":
            [
                "wp_start",
                "wp_end",
            ],
    },


    # --------------------------------------------------------
    # Selección occurrence-level
    # --------------------------------------------------------

    "occurrence_selection": {

        "occurrences_per_word":
            OCCURRENCES_PER_WORD,

        "selection_method":
            (
                "blake2b_128_keep_100_smallest_"
                "unique_word_context_hashes"
            ),

        "selection_personalization":
            "ctx100-v1",

        "selection_sha256":
            selection_sha256,

        "same_sampling_policy_for_all_words":
            True,

        "category_words_sampling_privilege":
            False,
    },


    # --------------------------------------------------------
    # Corpus
    # --------------------------------------------------------

    "corpus_pool": {

        "policy":
            "common_pool_same_for_every_word",

        "sources": {

            "wikitext_103_raw": {

                "dataset":
                    "wikitext-103-raw",

                "records_policy":
                    "all available records",
            },

            "wikimedia_wikipedia_20231101_en_common": {

                "dataset":
                    "wikimedia/wikipedia",

                "subset":
                    "20231101.en",

                "records_policy":
                    "first 800000 train records",

                "max_records":
                    800000,
            },
        },

        "source_documents_scanned":
            selection_report.get(
                "source_documents_scanned"
            ),
    },


    # --------------------------------------------------------
    # Vocabulario
    # --------------------------------------------------------

    "vocabulary": {

        "n_words":
            EXPECTED_N_WORDS,

        "minimum_unique_usable_occurrences":
            OCCURRENCES_PER_WORD,

        "automatic_candidate_contract": {

            "regex":
                "[a-z]+",

            "minimum_length":
                3,

            "english_stopwords_excluded_as_candidates":
                True,

            "custom_bad_words_excluded":
                True,

            "pos_filter":
                None,
        },

        "literal_nan_excluded":
            True,

        "literal_null_retained":
            True,

        "conceptual_lexicon_filter_applied":
            False,

        "conceptual_lexicon_filter_note":
            (
                "A downstream conceptual vocabulary may be "
                "constructed for prototype/neighbor analyses "
                "without modifying this master occurrence-level "
                "dataset."
            ),
    },


    # --------------------------------------------------------
    # Dataset final
    # --------------------------------------------------------

    "final_dataset": {

        "n_words":
            EXPECTED_N_WORDS,

        "occurrences_per_word":
            OCCURRENCES_PER_WORD,

        "n_rows":
            EXPECTED_N_ROWS,

        "ordered_by":
            [
                "word ASC",
                "sample_priority ASC",
            ],

        "row_group_target_rows":
            FINAL_ROW_GROUP_ROWS,

        "n_row_groups":
            parquet_n_row_groups,

        "occurrence_level":
            True,
    },


    # --------------------------------------------------------
    # Uso posterior como prototipo
    # --------------------------------------------------------

    "downstream_static_prototype_policy": {

        "prototype_definition":
            (
                "arithmetic mean of the 100 raw contextual "
                "occurrence embeddings for each word"
            ),

        "formula":
            "p_w = (1/100) * sum_i x_raw[w,i]",

        "l2_before_mean":
            False,

        "l2_after_mean":
            False,

        "prototype_already_stored_in_master_dataset":
            False,
    },
}


# ============================================================
# 11. Metadata final resumida
# ============================================================

dataset_meta = {

    "status":
        "DEFINITIVE_VALIDATED",

    "schema_version":
        CONTEXTUAL_DATASET_SCHEMA_VERSION,

    "config_name":
        CONFIG_NAME,

    "parquet_path":
        str(
            OUT_PARQUET_PATH
        ),

    "eligible_vocabulary_path":
        str(
            FINAL_ELIGIBLE_VOCAB_PATH
        ),

    "word_row_group_index_path":
        str(
            ROW_GROUP_INDEX_PATH
        ),

    "validation_report_path":
        str(
            VALIDATION_REPORT_PATH
        ),

    "tokenizer_alignment_audit_path":
        str(
            TOKENIZER_ALIGNMENT_AUDIT_PATH
        ),

    "generation_config_path":
        str(
            GENERATION_CONFIG_PATH
        ),

    "n_words":
        EXPECTED_N_WORDS,

    "occurrences_per_word":
        OCCURRENCES_PER_WORD,

    "n_rows":
        EXPECTED_N_ROWS,

    "embedding_dim":
        EMBEDDING_DIM,

    "embedding_dtype":
        "float32",

    "l2_normalized":
        False,

    "model":
        MODEL_NAME,

    "selected_layers":
        SELECTED_LAYER_IDS,

    "context_tokens_each_side":
        WINDOW_TOKENS_EACH_SIDE,

    "max_seq_len":
        MAX_SEQ_LEN,

    "wordpiece_input_policy":
        "target_centered_wordpiece_crop",

    "max_content_wordpieces":
        MAX_CONTENT_WORDPIECES,

    "preserve_all_target_wordpieces":
        True,

    "selection_sha256":
        selection_sha256,

    "all_words_exactly_100":
        True,

    "all_8_category_words_exactly_100":
        True,

    "all_embeddings_finite":
        True,

    "zero_embeddings":
        0,

    "word_order_violations":
        0,

    "row_group_index_valid":
        True,

    "conceptual_filter_applied":
        False,

    "master_dataset_should_remain_immutable":
        True,
}


# ============================================================
# 12. Guardar metadata
# ============================================================

save_json(
    GENERATION_CONFIG_PATH,
    generation_config,
)

save_json(
    META_PATH,
    dataset_meta,
)


# ============================================================
# 13. Comprobación posterior de escritura
# ============================================================

if not GENERATION_CONFIG_PATH.exists():

    raise RuntimeError(
        "No se escribió generation_config.json."
    )


if not META_PATH.exists():

    raise RuntimeError(
        "No se escribió contextual_dataset_meta.json."
    )


# ============================================================
# 14. Reporte final copiable
# ============================================================

metadata_report = {

    "status":
        "METADATA_FINALIZED",

    "dataset_status":
        dataset_meta[
            "status"
        ],

    "n_words":
        EXPECTED_N_WORDS,

    "n_rows":
        EXPECTED_N_ROWS,

    "occurrences_per_word":
        OCCURRENCES_PER_WORD,

    "embedding_dim":
        EMBEDDING_DIM,

    "l2_normalized":
        False,

    "wordpiece_input_policy":
        "target_centered_wordpiece_crop",

    "max_seq_len":
        MAX_SEQ_LEN,

    "max_content_wordpieces":
        MAX_CONTENT_WORDPIECES,

    "selection_sha256":
        selection_sha256,

    "generation_config_path":
        str(
            GENERATION_CONFIG_PATH
        ),

    "dataset_meta_path":
        str(
            META_PATH
        ),
}


print(
    "\n"
    + "=" * 90
)

print(
    "PARA COPIAR Y PEGAR — "
    "METADATOS DEFINITIVOS DATASET CONTEXTUAL"
)

print(
    "=" * 90
)

print(
    json.dumps(
        metadata_report,
        ensure_ascii=False,
        indent=2,
        sort_keys=True,
    )
)

print(
    "=" * 90
)

print(
    "FIN — METADATOS DEFINITIVOS DATASET CONTEXTUAL"
)

print(
    "=" * 90
)


PARA COPIAR Y PEGAR — METADATOS DEFINITIVOS DATASET CONTEXTUAL
{
  "dataset_meta_path": "C:\\Users\\jisaa\\Workspace\\Tesis\\Exp2\\Results\\contextual_dataset\\contextual_dataset_meta.json",
  "dataset_status": "DEFINITIVE_VALIDATED",
  "embedding_dim": 768,
  "generation_config_path": "C:\\Users\\jisaa\\Workspace\\Tesis\\Exp2\\Results\\contextual_dataset\\generation_config.json",
  "l2_normalized": false,
  "max_content_wordpieces": 126,
  "max_seq_len": 128,
  "n_rows": 10302900,
  "n_words": 103029,
  "occurrences_per_word": 100,
  "selection_sha256": "e130061a093ff98c3265cda041a3a86e77c6204c3509e0aa0a56405af87af087",
  "status": "METADATA_FINALIZED",
  "wordpiece_input_policy": "target_centered_wordpiece_crop"
}
FIN — METADATOS DEFINITIVOS DATASET CONTEXTUAL


In [3]:
# ============================================================
# 14. Construir índice word -> row group
#     VERSIÓN AUTOSUFICIENTE POST-REINICIO
# ============================================================
#
# Invariantes:
# - Parquet final ordenado por word;
# - 100 ocurrencias exactas por word;
# - row groups objetivo de 1000 filas;
# - 10 palabras completas por row group,
#   excepto el último si corresponde.
# ============================================================

from pathlib import Path
import math

import pandas as pd
import pyarrow.parquet as pq


# ============================================================
# 1. Rutas definitivas
# ============================================================

OUT_DIR = Path(
    r"C:\Users\jisaa\Workspace\Tesis\Exp2\Results\contextual_dataset"
)

OUT_PARQUET_PATH = (
    OUT_DIR
    / "contextual_occurrences_definitive.parquet"
)

FINAL_ELIGIBLE_VOCAB_PATH = (
    OUT_DIR
    / "eligible_vocabulary_min100.csv"
)

ROW_GROUP_INDEX_PATH = (
    OUT_DIR
    / "word_row_group_index.csv"
)


# ============================================================
# 2. Constantes cerradas del dataset
# ============================================================

OCCURRENCES_PER_WORD = 100

FINAL_ROW_GROUP_ROWS = 1000

EXPECTED_N_WORDS = 103_029

EXPECTED_N_ROWS = (
    EXPECTED_N_WORDS
    * OCCURRENCES_PER_WORD
)


# ============================================================
# 3. Verificar archivos requeridos
# ============================================================

if not OUT_PARQUET_PATH.exists():

    raise FileNotFoundError(
        "No existe el Parquet contextual definitivo:\n"
        f"{OUT_PARQUET_PATH}"
    )


if not FINAL_ELIGIBLE_VOCAB_PATH.exists():

    raise FileNotFoundError(
        "No existe el vocabulario definitivo:\n"
        f"{FINAL_ELIGIBLE_VOCAB_PATH}"
    )


# ============================================================
# 4. Reconstruir eligible_words desde disco
#
# IMPORTANTE:
# keep_default_na=False preserva correctamente tokens
# literales como 'null'.
# ============================================================

eligible_vocab_df = pd.read_csv(
    FINAL_ELIGIBLE_VOCAB_PATH,
    keep_default_na=False,
    na_filter=False,
)


if "word" not in eligible_vocab_df.columns:

    raise RuntimeError(
        "eligible_vocabulary_min100.csv "
        "no contiene columna 'word'."
    )


eligible_vocab_df["word"] = (
    eligible_vocab_df["word"]
    .astype(str)
    .str.lower()
    .str.strip()
)


eligible_vocab_df = (
    eligible_vocab_df
    .drop_duplicates(
        subset=["word"]
    )
    .sort_values("word")
    .reset_index(drop=True)
)


eligible_words = (
    eligible_vocab_df["word"]
    .tolist()
)


# ============================================================
# 5. Validar vocabulario definitivo
# ============================================================

if len(eligible_words) != EXPECTED_N_WORDS:

    raise RuntimeError(
        "Número inesperado de palabras en "
        "eligible_vocabulary_min100.csv: "
        f"{len(eligible_words):,} "
        f"!= {EXPECTED_N_WORDS:,}"
    )


if "nan" in eligible_words:

    raise RuntimeError(
        "La palabra 'nan' reapareció en el "
        "vocabulario definitivo."
    )


if len(set(eligible_words)) != len(eligible_words):

    raise RuntimeError(
        "El vocabulario contiene duplicados."
    )


print(
    "Vocabulario recuperado:",
    f"{len(eligible_words):,}",
)


# ============================================================
# 6. Abrir Parquet y validar estructura global
# ============================================================

final_pf = pq.ParquetFile(
    OUT_PARQUET_PATH
)


try:

    n_rows_parquet = int(
        final_pf.metadata.num_rows
    )

    n_row_groups_actual = int(
        final_pf.metadata.num_row_groups
    )


    if n_rows_parquet != EXPECTED_N_ROWS:

        raise RuntimeError(
            "Número de filas inesperado en el Parquet: "
            f"{n_rows_parquet:,} "
            f"!= {EXPECTED_N_ROWS:,}"
        )


    # ========================================================
    # 7. Validar tamaños físicos de row groups
    # ========================================================

    row_group_sizes = [
        int(
            final_pf.metadata
            .row_group(row_group_id)
            .num_rows
        )
        for row_group_id
        in range(
            n_row_groups_actual
        )
    ]


    expected_n_row_groups = int(
        math.ceil(
            EXPECTED_N_ROWS
            / FINAL_ROW_GROUP_ROWS
        )
    )


    if n_row_groups_actual != expected_n_row_groups:

        raise RuntimeError(
            "El número de row groups del Parquet "
            "no coincide con el diseño: "
            f"{n_row_groups_actual:,} "
            f"!= {expected_n_row_groups:,}"
        )


    # Todos salvo el último deben tener 1000 filas.
    if any(
        size != FINAL_ROW_GROUP_ROWS
        for size
        in row_group_sizes[:-1]
    ):

        bad = [
            (idx, size)
            for idx, size
            in enumerate(
                row_group_sizes[:-1]
            )
            if size != FINAL_ROW_GROUP_ROWS
        ]

        raise RuntimeError(
            "Hay row groups intermedios con tamaño "
            "distinto de 1000. "
            f"Ejemplos: {bad[:20]}"
        )


    expected_last_row_group_size = int(
        EXPECTED_N_ROWS
        - (
            expected_n_row_groups - 1
        )
        * FINAL_ROW_GROUP_ROWS
    )


    if (
        row_group_sizes[-1]
        != expected_last_row_group_size
    ):

        raise RuntimeError(
            "Tamaño inesperado del último row group: "
            f"{row_group_sizes[-1]} "
            f"!= {expected_last_row_group_size}"
        )


finally:

    # Evitar volver a dejar bloqueado el Parquet en Windows.
    final_pf.close()

    del final_pf


# ============================================================
# 8. Construir índice word -> row group
# ============================================================

row_group_index_rows = []


for word_index, word in enumerate(
    eligible_words
):

    row_start = int(
        word_index
        * OCCURRENCES_PER_WORD
    )

    row_end = int(
        row_start
        + OCCURRENCES_PER_WORD
    )

    row_group_id = int(
        row_start
        // FINAL_ROW_GROUP_ROWS
    )


    row_group_index_rows.append(
        {
            "word":
                word,

            "word_index":
                int(
                    word_index
                ),

            "n_occurrences":
                int(
                    OCCURRENCES_PER_WORD
                ),

            "row_start":
                row_start,

            "row_end_exclusive":
                row_end,

            "row_group_id":
                row_group_id,
        }
    )


word_row_group_index_df = pd.DataFrame(
    row_group_index_rows
)


# ============================================================
# 9. Validaciones del índice
# ============================================================

if len(
    word_row_group_index_df
) != EXPECTED_N_WORDS:

    raise RuntimeError(
        "El índice no contiene exactamente "
        f"{EXPECTED_N_WORDS:,} palabras."
    )


if (
    word_row_group_index_df[
        "n_occurrences"
    ]
    != OCCURRENCES_PER_WORD
).any():

    raise RuntimeError(
        "El índice contiene una palabra con "
        "n_occurrences != 100."
    )


if (
    word_row_group_index_df[
        "row_group_id"
    ].max()
    >= expected_n_row_groups
):

    raise RuntimeError(
        "El índice referencia un row group inexistente."
    )


# ============================================================
# 10. Guardar
# ============================================================

word_row_group_index_df.to_csv(
    ROW_GROUP_INDEX_PATH,
    index=False,
    encoding="utf-8",
)


# ============================================================
# 11. Resumen
# ============================================================

index_report = {

    "parquet_path":
        str(
            OUT_PARQUET_PATH
        ),

    "row_group_index_path":
        str(
            ROW_GROUP_INDEX_PATH
        ),

    "n_words":
        int(
            EXPECTED_N_WORDS
        ),

    "occurrences_per_word":
        int(
            OCCURRENCES_PER_WORD
        ),

    "n_rows":
        int(
            EXPECTED_N_ROWS
        ),

    "row_group_rows":
        int(
            FINAL_ROW_GROUP_ROWS
        ),

    "n_row_groups":
        int(
            expected_n_row_groups
        ),

    "last_row_group_rows":
        int(
            expected_last_row_group_size
        ),

    "first_word":
        str(
            eligible_words[0]
        ),

    "last_word":
        str(
            eligible_words[-1]
        ),

    "nan_absent":
        bool(
            "nan"
            not in eligible_words
        ),

    "index_valid":
        True,
}


print_copy_block(
    "ÍNDICE WORD ROW-GROUP",
    index_report,
)


display(
    word_row_group_index_df.head(
        20
    )
)

Vocabulario recuperado: 103,029


NameError: name 'print_copy_block' is not defined

In [6]:
# ============================================================
# 15. Validación final fuerte del dataset
# ============================================================

final_pf = pq.ParquetFile(
    OUT_PARQUET_PATH
)

logical_columns = (
    final_pf.schema_arrow.names
)

required_columns = (
    FINAL_SCHEMA.names
)

missing_columns = [
    column
    for column in required_columns
    if column not in logical_columns
]

if missing_columns:
    raise RuntimeError(
        f"Faltan columnas finales: {missing_columns}"
    )

n_rows_final = int(
    final_pf.metadata.num_rows
)

expected_rows = int(
    len(eligible_words)
    * OCCURRENCES_PER_WORD
)

if n_rows_final != expected_rows:
    raise RuntimeError(
        f"n_rows={n_rows_final} != expected={expected_rows}"
    )

# ------------------------------------------------------------
# Conteos por palabra, orden y metadata
# ------------------------------------------------------------

word_counts_stream = Counter()
last_word = None
order_violations = 0

category_metadata_rows = []
sample_category_rows = []

metadata_scan_columns = [
    "word",
    "is_target",
    "word_role",
    "true_class",
    "class_label",
    "is_exp2_word",
    "source_name",
    "source_split",
    "doc_split",
    "context_window",
]

for batch in tqdm(
    final_pf.iter_batches(
        batch_size=100_000,
        columns=metadata_scan_columns,
    ),
    desc="Validando metadata final",
):

    batch_df = batch.to_pandas()

    words_batch = (
        batch_df["word"]
        .astype(str)
        .tolist()
    )

    word_counts_stream.update(
        words_batch
    )

    for word in words_batch:

        if (
            last_word is not None
            and word < last_word
        ):
            order_violations += 1

        last_word = word

    category_batch = batch_df[
        batch_df["word"].isin(
            category_set
        )
    ]

    if len(category_batch):

        category_metadata_rows.append(
            category_batch[
                [
                    "word",
                    "is_target",
                    "word_role",
                    "true_class",
                    "class_label",
                    "is_exp2_word",
                ]
            ].copy()
        )

        for category_word in category_words:

            existing_n = sum(
                int(
                    (
                        piece["word"]
                        == category_word
                    ).sum()
                )
                for piece
                in sample_category_rows
            )

            if existing_n >= 3:
                continue

            sample_piece = (
                category_batch[
                    category_batch["word"]
                    == category_word
                ]
                .head(
                    3 - existing_n
                )
                [
                    [
                        "word",
                        "source_name",
                        "source_split",
                        "doc_split",
                        "context_window",
                    ]
                ]
                .copy()
            )

            if len(sample_piece):
                sample_category_rows.append(
                    sample_piece
                )

bad_word_counts = {
    word: count
    for word, count in word_counts_stream.items()
    if count != OCCURRENCES_PER_WORD
}

if bad_word_counts:
    raise RuntimeError(
        "No todas las palabras tienen exactamente 100 ocurrencias. "
        f"Ejemplos={list(bad_word_counts.items())[:20]}"
    )

if len(word_counts_stream) != len(eligible_words):
    raise RuntimeError(
        "Número incorrecto de palabras únicas."
    )

if set(word_counts_stream) != eligible_word_set:
    raise RuntimeError(
        "El vocabulario final no coincide exactamente con "
        "eligible_vocabulary_min100.csv."
    )

if order_violations != 0:
    raise RuntimeError(
        "El Parquet final no está ordenado por word: "
        f"{order_violations} violaciones."
    )

# ------------------------------------------------------------
# CategoryWords
# ------------------------------------------------------------

if category_metadata_rows:

    category_meta_df = pd.concat(
        category_metadata_rows,
        ignore_index=True,
    )

else:

    category_meta_df = pd.DataFrame()

category_coverage_df = pd.DataFrame(
    [
        {
            "category_word": word,
            "n_occurrences": int(
                word_counts_stream[word]
            ),
            "true_class": int(
                WORD_TO_TRUE_CLASS[word]
            ),
            "class_label": str(
                WORD_TO_CLASS_LABEL[word]
            ),
        }
        for word in category_words
    ]
)

if not (
    category_coverage_df["n_occurrences"]
    == OCCURRENCES_PER_WORD
).all():
    raise RuntimeError(
        "Alguna CategoryWord no tiene 100 ocurrencias."
    )

if len(category_meta_df):

    if (
        ~category_meta_df["is_target"]
    ).any():
        raise RuntimeError(
            "Alguna CategoryWord tiene is_target=False."
        )

    if (
        ~category_meta_df["is_exp2_word"]
    ).any():
        raise RuntimeError(
            "Alguna CategoryWord tiene is_exp2_word=False."
        )

# ------------------------------------------------------------
# Cobertura descriptiva de las 24 palabras con metadata Exp2
# ------------------------------------------------------------

exp2_metadata_coverage_df = pd.DataFrame(
    [
        {
            "word": word,
            "present": bool(
                word in word_counts_stream
            ),
            "n_occurrences": int(
                word_counts_stream.get(
                    word,
                    0,
                )
            ),
            "word_role": target_roles.get(
                word,
                "unknown",
            ),
            "true_class": int(
                WORD_TO_TRUE_CLASS[word]
            ),
            "class_label": str(
                WORD_TO_CLASS_LABEL[word]
            ),
        }
        for word in target_words
    ]
)

# ------------------------------------------------------------
# Auditoría numérica completa de embeddings por streaming
# ------------------------------------------------------------

embedding_dim = None
n_embedding_rows_checked = 0
n_non_finite = 0
n_zero = 0

for batch in tqdm(
    final_pf.iter_batches(
        batch_size=4096,
        columns=["embedding"],
    ),
    desc="Validando embeddings finales",
):

    embedding_array = batch.column(0)

    X = (
        embedding_array.values
        .to_numpy(
            zero_copy_only=False
        )
        .reshape(
            len(batch),
            -1,
        )
        .astype(
            np.float32,
            copy=False,
        )
    )

    if embedding_dim is None:
        embedding_dim = int(
            X.shape[1]
        )

    if X.shape[1] != hidden_size:
        raise RuntimeError(
            f"Embedding dim inesperado: {X.shape[1]}"
        )

    n_embedding_rows_checked += int(
        X.shape[0]
    )

    n_non_finite += int(
        (
            ~np.isfinite(X)
        ).any(
            axis=1
        ).sum()
    )

    n_zero += int(
        np.all(
            X == 0,
            axis=1,
        ).sum()
    )

if (
    n_embedding_rows_checked
    != n_rows_final
):
    raise RuntimeError(
        "No se auditaron todas las filas de embeddings."
    )

if n_non_finite != 0:
    raise RuntimeError(
        f"Hay {n_non_finite} embeddings con NaN/Inf."
    )

if n_zero != 0:
    raise RuntimeError(
        f"Hay {n_zero} embeddings cero."
    )

# ------------------------------------------------------------
# Guardar auxiliares
# ------------------------------------------------------------

word_counts_df = pd.DataFrame(
    {
        "word": eligible_words,
        "n_occurrences": [
            int(
                word_counts_stream[word]
            )
            for word in eligible_words
        ],
    }
)

word_counts_df.to_csv(
    WORD_COUNTS_PATH,
    index=False,
    encoding="utf-8",
)

category_coverage_df.to_csv(
    CATEGORY_COVERAGE_PATH,
    index=False,
    encoding="utf-8",
)

exp2_metadata_coverage_df.to_csv(
    EXP2_METADATA_COVERAGE_PATH,
    index=False,
    encoding="utf-8",
)

if sample_category_rows:

    sample_contexts_df = (
        pd.concat(
            sample_category_rows,
            ignore_index=True,
        )
        .drop_duplicates(
            [
                "word",
                "context_window",
            ]
        )
        .groupby(
            "word",
            group_keys=False,
        )
        .head(3)
        .sort_values("word")
        .reset_index(drop=True)
    )

else:

    sample_contexts_df = pd.DataFrame()

sample_contexts_df.to_csv(
    SAMPLE_CONTEXTS_PATH,
    index=False,
    encoding="utf-8",
)

validation_report = {
    "schema_version": (
        CONTEXTUAL_DATASET_SCHEMA_VERSION
    ),
    "parquet_path": str(
        OUT_PARQUET_PATH
    ),
    "parquet_size_gib": float(
        OUT_PARQUET_PATH.stat().st_size
        / 1024**3
    ),
    "n_rows": int(
        n_rows_final
    ),
    "expected_rows": int(
        expected_rows
    ),
    "n_unique_words": int(
        len(word_counts_stream)
    ),
    "occurrences_per_word": int(
        OCCURRENCES_PER_WORD
    ),
    "min_word_count": int(
        min(
            word_counts_stream.values()
        )
    ),
    "max_word_count": int(
        max(
            word_counts_stream.values()
        )
    ),
    "all_words_exactly_100": bool(
        not bad_word_counts
    ),
    "all_8_category_words_exactly_100": bool(
        (
            category_coverage_df["n_occurrences"]
            == OCCURRENCES_PER_WORD
        ).all()
    ),
    "word_order_violations": int(
        order_violations
    ),
    "embedding_dim": int(
        embedding_dim
    ),
    "n_embedding_rows_checked": int(
        n_embedding_rows_checked
    ),
    "n_non_finite_embeddings": int(
        n_non_finite
    ),
    "n_zero_embeddings": int(
        n_zero
    ),
    "l2_normalized": bool(
        L2_NORMALIZE_EMBEDDINGS
    ),
    "n_row_groups": int(
        final_pf.metadata.num_row_groups
    ),
    "row_group_rows_target": int(
        FINAL_ROW_GROUP_ROWS
    ),
    "row_group_index_path": str(
        ROW_GROUP_INDEX_PATH
    ),
}

save_json(
    validation_report,
    VALIDATION_REPORT_PATH,
)

print_copy_block(
    "VALIDACIÓN FINAL DATASET CONTEXTUAL",
    validation_report,
)

NameError: name 'FINAL_SCHEMA' is not defined

In [7]:
# ============================================================
# 16. Metadata científica y configuración final
# ============================================================

generation_config = {
    "config_name": CONFIG_NAME,
    "schema_version": CONTEXTUAL_DATASET_SCHEMA_VERSION,
    "seed": int(SEED),

    "pool_sources": CORPUS_SOURCES,

    "vocabulary_definition": (
        "audited common-pool words with >=100 unique contexts "
        "after final lexical contract"
    ),

    "occurrences_per_word": int(
        OCCURRENCES_PER_WORD
    ),

    "selection_method": (
        "BLAKE2b-128 deterministic priority; "
        "100 smallest unique (word, context_window) hashes per word"
    ),

    "selection_personalization": "ctx100-v1",

    "model_name": MODEL_NAME,

    "embedding_dim": int(
        hidden_size
    ),

    "selected_layer_ids": list(
        SELECTED_LAYER_IDS
    ),

    "resolved_encoder_layer_indices": list(
        resolved_layer_indices
    ),

    "layer_aggregation": "arithmetic_mean",

    "l2_normalize_embeddings": bool(
        L2_NORMALIZE_EMBEDDINGS
    ),

    "max_seq_len": int(
        MAX_SEQ_LEN
    ),

    "window_tokens_each_side": int(
        WINDOW_TOKENS_EACH_SIDE
    ),

    "context_keep_stopwords": bool(
        CONTEXT_KEEP_STOPWORDS
    ),

    "minimum_word_length": int(
        MIN_ROW_WORD_LEN
    ),

    "custom_bad_words": sorted(
        CUSTOM_BAD_WORDS
    ),

    "category_words": list(
        category_words
    ),

    "target_words_metadata_only": list(
        target_words
    ),

    "special_target_sampling": False,
    "special_category_sampling": False,
    "category_words_required_present": True,

    "final_parquet_sorted_by": [
        "word",
        "sample_priority",
    ],

    "final_row_group_rows": int(
        FINAL_ROW_GROUP_ROWS
    ),

    "paths": {
        "output_dir": str(
            OUT_DIR
        ),
        "eligible_vocabulary": str(
            FINAL_ELIGIBLE_VOCAB_PATH
        ),
        "selection_manifest": str(
            SELECTED_HASHES_PATH
        ),
        "candidates_sorted": str(
            CANDIDATES_PATH
        ),
        "parquet": str(
            OUT_PARQUET_PATH
        ),
        "word_row_group_index": str(
            ROW_GROUP_INDEX_PATH
        ),
        "word_counts": str(
            WORD_COUNTS_PATH
        ),
        "category_coverage": str(
            CATEGORY_COVERAGE_PATH
        ),
        "exp2_metadata_coverage": str(
            EXP2_METADATA_COVERAGE_PATH
        ),
        "validation_report": str(
            VALIDATION_REPORT_PATH
        ),
    },
}

config_canonical_json = json.dumps(
    json_safe(
        generation_config
    ),
    ensure_ascii=False,
    sort_keys=True,
    separators=(
        ",",
        ":",
    ),
)

generation_config[
    "config_sha256"
] = hashlib.sha256(
    config_canonical_json.encode(
        "utf-8"
    )
).hexdigest()

save_json(
    generation_config,
    GENERATION_CONFIG_PATH,
)

META = {
    **generation_config,
    "vocabulary_summary": vocabulary_summary,
    "selection_report": selection_report,
    "candidate_materialization_report": (
        candidate_materialization_report
    ),
    "candidate_sort_report": (
        candidate_sort_report
    ),
    "extraction_report": extraction_report,
    "validation_report": validation_report,
}

save_json(
    META,
    META_PATH,
)

print(
    "Metadata guardada:",
    META_PATH,
)

print(
    "Config guardada:",
    GENERATION_CONFIG_PATH,
)

NameError: name 'CONFIG_NAME' is not defined

In [ ]:
# ============================================================
# 17. Loader optimizado para la extensión contextual
# ============================================================
#
# Este loader NO escanea el Parquet completo.
#
# Usa word_row_group_index.csv para leer únicamente los row groups
# que contienen las palabras solicitadas.
#
# Con row groups de 10 palabras:
# pedir ~160 palabras implica leer como máximo ~160.000 filas
# antes del filtro, no ~10 millones.
# ============================================================

def load_contextual_words(
    words,
    *,
    parquet_path=OUT_PARQUET_PATH,
    row_group_index_path=ROW_GROUP_INDEX_PATH,
    include_embeddings=True,
):

    words = sorted(
        set(
            str(word).lower()
            for word in words
        )
    )

    if not words:
        raise ValueError(
            "words no puede estar vacío."
        )

    index_df = pd.read_csv(
        row_group_index_path
    )

    missing = sorted(
        set(words)
        - set(
            index_df["word"]
        )
    )

    if missing:
        raise ValueError(
            "Palabras no presentes en el dataset contextual: "
            f"{missing[:50]}"
        )

    selected_index = (
        index_df[
            index_df["word"].isin(
                words
            )
        ]
    )

    row_group_ids = sorted(
        selected_index[
            "row_group_id"
        ]
        .astype(int)
        .unique()
        .tolist()
    )

    parquet_file = pq.ParquetFile(
        parquet_path
    )

    columns = [
        column
        for column in FINAL_SCHEMA.names
        if (
            include_embeddings
            or column != "embedding"
        )
    ]

    table = parquet_file.read_row_groups(
        row_group_ids,
        columns=columns,
    )

    df = table.to_pandas()

    df = (
        df[
            df["word"].isin(
                words
            )
        ]
        .sort_values(
            [
                "word",
                "occ_id",
            ]
        )
        .reset_index(drop=True)
    )

    counts = (
        df["word"]
        .value_counts()
        .reindex(
            words,
            fill_value=0,
        )
    )

    if not (
        counts == OCCURRENCES_PER_WORD
    ).all():
        raise RuntimeError(
            "El loader no recuperó exactamente 100 "
            "ocurrencias por palabra."
        )

    return df


# Prueba mínima con las 8 CategoryWords.
category_sample_df = load_contextual_words(
    category_words,
    include_embeddings=False,
)

print(
    "Shape prueba CategoryWords:",
    category_sample_df.shape,
)

display(
    category_sample_df[
        [
            "word",
            "source_name",
            "doc_split",
            "context_window",
        ]
    ].head(16)
)

In [ ]:
# ============================================================
# 18. Resumen final copiable
# ============================================================

final_summary = {
    "status": (
        "dataset_contextual_definitivo_generado_y_validado"
    ),
    "output_dir": str(
        OUT_DIR
    ),
    "parquet_path": str(
        OUT_PARQUET_PATH
    ),
    "n_words": int(
        len(eligible_words)
    ),
    "occurrences_per_word": int(
        OCCURRENCES_PER_WORD
    ),
    "n_rows": int(
        validation_report["n_rows"]
    ),
    "embedding_dim": int(
        validation_report["embedding_dim"]
    ),
    "embedding_definition": (
        "raw mean of last 4 BERT-base-uncased "
        "transformer layers per occurrence"
    ),
    "l2_normalized": False,
    "all_words_exactly_100": bool(
        validation_report[
            "all_words_exactly_100"
        ]
    ),
    "all_8_category_words_exactly_100": bool(
        validation_report[
            "all_8_category_words_exactly_100"
        ]
    ),
    "parquet_sorted_by_word": bool(
        validation_report[
            "word_order_violations"
        ]
        == 0
    ),
    "n_row_groups": int(
        validation_report[
            "n_row_groups"
        ]
    ),
    "row_group_index_path": str(
        ROW_GROUP_INDEX_PATH
    ),
    "selection_manifest_path": str(
        SELECTED_HASHES_PATH
    ),
    "selection_sha256": (
        selection_report.get(
            "selection_sha256"
        )
    ),
    "config_sha256": (
        generation_config[
            "config_sha256"
        ]
    ),
}

print_copy_block(
    "RESUMEN FINAL GENERACIÓN CONTEXTUAL",
    final_summary,
)

## Bloques que conviene conservar / compartir

Durante la ejecución, los resultados más informativos son:

1. `PARA COPIAR Y PEGAR — VOCABULARIO DEFINITIVO`
2. `PARA COPIAR Y PEGAR — SELECCIÓN HASH TOP100`
3. `PARA COPIAR Y PEGAR — CANDIDATOS MATERIALIZADOS`
4. `PARA COPIAR Y PEGAR — CANDIDATOS ORDENADOS`
5. `PARA COPIAR Y PEGAR — EXTRACCIÓN BERT`
6. `PARA COPIAR Y PEGAR — VALIDACIÓN FINAL DATASET CONTEXTUAL`
7. `PARA COPIAR Y PEGAR — RESUMEN FINAL GENERACIÓN CONTEXTUAL`

Las etapas previas a BERT quedan persistidas (`eligible_vocabulary`, `selected_context_hashes` y `candidates_sorted`) para no repetir selección si el kernel se reinicia antes de la extracción contextual.

In [ ]:
# ============================================================
# 19. Preparar regeneración autónoma del input contextual de Exp2
# ============================================================
#
# Esta sección permite reconstruir directamente el input usado por
# Experimento_2_Hopfield_vs_HFY_UNIFICADO.ipynb a partir del
# dataset occurrence-level definitivo generado por este notebook.
#
# NO depende de Exp2_prototiposV1/V2 ni de notebooks de auditoría.
#
# Si el kernel fue reiniciado, esta celda puede ejecutarse de forma
# autónoma siempre que contextual_occurrences_definitive.parquet y
# eligible_vocabulary_min100.csv ya existan.
# ============================================================

import hashlib
import importlib.util
import json
import pickle
import subprocess
import sys
import time

from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from tqdm.auto import tqdm


# ------------------------------------------------------------
# 1. Rutas fuente y destino
# ------------------------------------------------------------

PROJECT_ROOT_EXP2_REBUILD = Path(
    r"C:\Users\jisaa\Workspace\Tesis"
)

SOURCE_CONTEXTUAL_DIR = (
    PROJECT_ROOT_EXP2_REBUILD
    / "Exp2"
    / "Results"
    / "contextual_dataset"
)

SOURCE_CONTEXTUAL_PARQUET_PATH = (
    SOURCE_CONTEXTUAL_DIR
    / "contextual_occurrences_definitive.parquet"
)

SOURCE_ELIGIBLE_VOCAB_PATH = (
    SOURCE_CONTEXTUAL_DIR
    / "eligible_vocabulary_min100.csv"
)

EXP2_CONTEXTUAL_RESULTS_ROOT = (
    PROJECT_ROOT_EXP2_REBUILD
    / "Exp2"
    / "Results"
    / "ResultsExp2_prototipos"
    / "ctx_v3_equal100"
)

EXP2_CONTEXTUAL_INPUT_ROOT = (
    EXP2_CONTEXTUAL_RESULTS_ROOT
    / "03_exp2_input"
)

EXP2_CONTEXTUAL_INPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

EXP2_CONTEXTUAL_PKL_PATH = (
    EXP2_CONTEXTUAL_INPUT_ROOT
    / "contextual_exp2_embeddings_rawmean100_wordnet_v1.pkl"
)

EXP2_CONTEXTUAL_PARQUET_PATH = (
    EXP2_CONTEXTUAL_INPUT_ROOT
    / "contextual_exp2_embeddings_rawmean100_wordnet_v1.parquet"
)

EXP2_CONTEXTUAL_VOCAB_PATH = (
    EXP2_CONTEXTUAL_INPUT_ROOT
    / "contextual_exp2_vocabulary_wordnet_v1.csv"
)

EXP2_CONTEXTUAL_FILTER_AUDIT_PATH = (
    EXP2_CONTEXTUAL_INPUT_ROOT
    / "contextual_exp2_vocabulary_wordnet_v1_audit.csv"
)

EXP2_CONTEXTUAL_MANIFEST_PATH = (
    EXP2_CONTEXTUAL_INPUT_ROOT
    / "contextual_exp2_input_manifest.json"
)


# ------------------------------------------------------------
# 2. Dependencia WordNet
# ------------------------------------------------------------

if importlib.util.find_spec("nltk") is None:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "nltk",
        ]
    )

import nltk
from nltk.corpus import wordnet as wn

try:
    wn.ensure_loaded()

except LookupError:

    download_ok = nltk.download(
        "wordnet",
        quiet=False,
    )

    if not download_ok:
        raise RuntimeError(
            "No fue posible descargar WordNet."
        )

    wn.ensure_loaded()


# ------------------------------------------------------------
# 3. Contrato congelado del input contextual de Exp2
# ------------------------------------------------------------

MODEL_NAME_EXP2_REBUILD = (
    "bert-base-uncased"
)

SOURCE_CONTEXTUAL_SCHEMA_VERSION = (
    "ctx_v3_commonpool_equal100_rawbert"
)

CONTEXTUAL_EXP2_EMBEDDING_NAME = (
    "bert_contextual_prototype_rawmean100_wordnet_v1"
)

CONCEPTUAL_FILTER_METHOD = (
    "wordnet_synset_or_morphy_v1"
)

EXPECTED_MASTER_N_WORDS = 103_029
EXPECTED_OCCURRENCES_PER_WORD = 100
EXPECTED_EMBEDDING_DIM = 768

EXPECTED_EXP2_N_WORDS = 50_455

EXPECTED_MASTER_SELECTION_SHA256 = (
    "e130061a093ff98c3265cda041a3a86e"
    "77c6204c3509e0aa0a56405af87af087"
)

EXPECTED_CONTEXTUAL_EMBEDDING_SHA256 = (
    "0590ffe496e1c0da00b399e4773d715f"
    "835b17ce2c0b0004ff19fcd3c7269593"
)

EXPECTED_CONTEXTUAL_VOCABULARY_SHA256 = (
    "95d87d7ce442dd36699ab0ff7bdd8bd1"
    "2c10e5460f642f6ba4a514369e088edc"
)

CATEGORY_WORDS_EXP2 = [
    "apple",
    "bicycle",
    "bread",
    "canoe",
    "chair",
    "jellyfish",
    "screwdriver",
    "skunk",
]


# ------------------------------------------------------------
# 4. Helpers locales aislados
# ------------------------------------------------------------
#
# Se usan nombres propios para no depender de redefiniciones de
# save_json()/sha256_file() en otras celdas del notebook.
# ------------------------------------------------------------

def _exp2_write_json(
    path,
    payload,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    with open(
        path,
        "w",
        encoding="utf-8",
    ) as handle:

        json.dump(
            payload,
            handle,
            ensure_ascii=False,
            indent=2,
            sort_keys=True,
        )

    return path


def _exp2_sha256_file(
    path,
):
    hasher = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as handle:

        while True:

            block = handle.read(
                1024 * 1024
            )

            if not block:
                break

            hasher.update(
                block
            )

    return hasher.hexdigest()


# ------------------------------------------------------------
# 5. Validar fuente occurrence-level
# ------------------------------------------------------------

if not SOURCE_CONTEXTUAL_PARQUET_PATH.exists():
    raise FileNotFoundError(
        "No existe el dataset occurrence-level definitivo:\n"
        f"{SOURCE_CONTEXTUAL_PARQUET_PATH}"
    )

if not SOURCE_ELIGIBLE_VOCAB_PATH.exists():
    raise FileNotFoundError(
        "No existe el vocabulario maestro definitivo:\n"
        f"{SOURCE_ELIGIBLE_VOCAB_PATH}"
    )

eligible_vocab_exp2_df = pd.read_csv(
    SOURCE_ELIGIBLE_VOCAB_PATH,
    keep_default_na=False,
    na_filter=False,
)

eligible_vocab_exp2_df["word"] = (
    eligible_vocab_exp2_df["word"]
    .astype(str)
    .str.lower()
    .str.strip()
)

eligible_words_exp2_source = (
    eligible_vocab_exp2_df["word"]
    .drop_duplicates()
    .sort_values()
    .tolist()
)

if len(
    eligible_words_exp2_source
) != EXPECTED_MASTER_N_WORDS:
    raise RuntimeError(
        "El vocabulario maestro no contiene exactamente "
        f"{EXPECTED_MASTER_N_WORDS:,} palabras."
    )

if "nan" in eligible_words_exp2_source:
    raise RuntimeError(
        "El literal espurio 'nan' reapareció en el vocabulario."
    )

master_pf = pq.ParquetFile(
    SOURCE_CONTEXTUAL_PARQUET_PATH
)

try:

    master_n_rows = int(
        master_pf.metadata.num_rows
    )

finally:

    master_pf.close()
    del master_pf

expected_master_n_rows = (
    EXPECTED_MASTER_N_WORDS
    * EXPECTED_OCCURRENCES_PER_WORD
)

if master_n_rows != expected_master_n_rows:
    raise RuntimeError(
        "El Parquet occurrence-level no tiene el número esperado "
        f"de filas: {master_n_rows:,} != "
        f"{expected_master_n_rows:,}."
    )

print(
    "Fuente occurrence-level validada:",
    SOURCE_CONTEXTUAL_PARQUET_PATH,
)

print(
    "Vocabulario maestro:",
    len(
        eligible_words_exp2_source
    ),
)

print(
    "Destino input Exp2:",
    EXP2_CONTEXTUAL_INPUT_ROOT,
)

In [ ]:
# ============================================================
# 20. Reconstruir los 103.029 prototipos RAW mean100
# ============================================================
#
# Definición cerrada:
#
#     p_w = (1/100) * sum_i x_raw[w, i]
#
# NO se aplica:
# - L2 a las ocurrencias;
# - L2 al prototipo;
# - clustering;
# - pseudo-sentidos.
#
# La aritmética replica la usada en Exp2_prototiposV2:
# float32 del Parquet -> float64 -> media float64.
# ============================================================

prototype_started = time.perf_counter()

parquet_file_exp2 = pq.ParquetFile(
    SOURCE_CONTEXTUAL_PARQUET_PATH
)

PROTOTYPE_ROW_GROUP_CHUNK = 25

prototype_blocks_exp2 = []
prototype_words_exp2 = []

rows_scanned_exp2 = 0

try:

    n_row_groups_exp2 = int(
        parquet_file_exp2.metadata.num_row_groups
    )

    for rg_start in tqdm(
        range(
            0,
            n_row_groups_exp2,
            PROTOTYPE_ROW_GROUP_CHUNK,
        ),
        total=(
            n_row_groups_exp2
            + PROTOTYPE_ROW_GROUP_CHUNK
            - 1
        )
        // PROTOTYPE_ROW_GROUP_CHUNK,
        desc="Reconstruyendo prototipos RAW mean100",
        unit="chunk",
    ):

        rg_ids = list(
            range(
                rg_start,
                min(
                    rg_start
                    + PROTOTYPE_ROW_GROUP_CHUNK,
                    n_row_groups_exp2,
                ),
            )
        )

        table = parquet_file_exp2.read_row_groups(
            rg_ids,
            columns=[
                "word",
                "embedding",
            ],
        )

        n_rows_chunk = int(
            table.num_rows
        )

        if (
            n_rows_chunk
            % EXPECTED_OCCURRENCES_PER_WORD
            != 0
        ):
            raise RuntimeError(
                "Un chunk de row groups no contiene un múltiplo "
                "de 100 filas."
            )

        words_chunk = (
            table["word"]
            .to_pylist()
        )

        embedding_array = (
            table["embedding"]
            .combine_chunks()
        )

        X_chunk = (
            embedding_array
            .values
            .to_numpy(
                zero_copy_only=False
            )
            .reshape(
                n_rows_chunk,
                EXPECTED_EMBEDDING_DIM,
            )
            .astype(
                np.float64,
                copy=False,
            )
        )

        n_words_chunk = (
            n_rows_chunk
            // EXPECTED_OCCURRENCES_PER_WORD
        )

        X_words = X_chunk.reshape(
            n_words_chunk,
            EXPECTED_OCCURRENCES_PER_WORD,
            EXPECTED_EMBEDDING_DIM,
        )

        chunk_word_names = []

        for block_idx in range(
            n_words_chunk
        ):

            row_start = (
                block_idx
                * EXPECTED_OCCURRENCES_PER_WORD
            )

            row_end = (
                row_start
                + EXPECTED_OCCURRENCES_PER_WORD
            )

            word_block = words_chunk[
                row_start:
                row_end
            ]

            canonical_word = str(
                word_block[0]
            )

            if any(
                str(word)
                != canonical_word
                for word in word_block
            ):
                raise RuntimeError(
                    "Un bloque de 100 filas contiene más de una palabra. "
                    f"RG start={rg_start}, bloque={block_idx}."
                )

            chunk_word_names.append(
                canonical_word
            )

        chunk_prototypes = (
            X_words.mean(
                axis=1,
                dtype=np.float64,
            )
        )

        prototype_blocks_exp2.append(
            chunk_prototypes
        )

        prototype_words_exp2.extend(
            chunk_word_names
        )

        rows_scanned_exp2 += (
            n_rows_chunk
        )

        del table
        del embedding_array
        del X_chunk
        del X_words
        del chunk_prototypes

finally:

    parquet_file_exp2.close()
    del parquet_file_exp2


if rows_scanned_exp2 != (
    EXPECTED_MASTER_N_WORDS
    * EXPECTED_OCCURRENCES_PER_WORD
):
    raise RuntimeError(
        "No se recorrieron todas las filas del Parquet."
    )

prototype_matrix_exp2 = np.vstack(
    prototype_blocks_exp2
)

del prototype_blocks_exp2

if prototype_matrix_exp2.shape != (
    EXPECTED_MASTER_N_WORDS,
    EXPECTED_EMBEDDING_DIM,
):
    raise RuntimeError(
        "Forma inesperada de la matriz maestra de prototipos: "
        f"{prototype_matrix_exp2.shape}"
    )

if (
    prototype_words_exp2
    != eligible_words_exp2_source
):
    raise RuntimeError(
        "El orden de palabras reconstruido desde el Parquet "
        "no coincide exactamente con el vocabulario maestro."
    )

if not np.isfinite(
    prototype_matrix_exp2
).all():
    raise RuntimeError(
        "Los prototipos RAW contienen NaN o Inf."
    )

prototype_norms_master = np.linalg.norm(
    prototype_matrix_exp2,
    axis=1,
)

if np.any(
    prototype_norms_master <= 0.0
):
    raise RuntimeError(
        "Existe al menos un prototipo maestro de norma cero."
    )

prototype_columns_exp2 = [
    f"dim_{idx:03d}"
    for idx in range(
        EXPECTED_EMBEDDING_DIM
    )
]

contextual_prototypes_all_df = pd.DataFrame(
    prototype_matrix_exp2,
    index=pd.Index(
        prototype_words_exp2,
        name="word",
    ),
    columns=prototype_columns_exp2,
)

if not contextual_prototypes_all_df.index.is_monotonic_increasing:
    raise RuntimeError(
        "Los prototipos maestros no están ordenados "
        "lexicográficamente."
    )

print(
    "Prototipos maestros reconstruidos:",
    contextual_prototypes_all_df.shape,
)

print(
    "Tiempo [s]:",
    round(
        time.perf_counter()
        - prototype_started,
        2,
    ),
)

In [ ]:
# ============================================================
# 21. Reconstruir V_conceptual WordNet v1
# ============================================================
#
# Regla EXACTA cerrada:
#
# conservar w si:
#
#   1) wn.synsets(w) no es vacío
#
#      O
#
#   2) wn.morphy(w, pos) produce alguna forma base para
#      NOUN/VERB/ADJ/ADV y esa forma base tiene algún synset.
#
# Resultado histórico esperado:
#   103.029 -> 50.455 palabras.
# ============================================================

WORDNET_POS_EXP2 = [
    wn.NOUN,
    wn.VERB,
    wn.ADJ,
    wn.ADV,
]

conceptual_rows_exp2 = []

for word in tqdm(
    eligible_words_exp2_source,
    desc="Reconstruyendo V_conceptual WordNet v1",
    unit="word",
):

    direct_synsets = wn.synsets(
        word
    )

    morphy_lemmas = sorted(
        {
            lemma
            for pos in WORDNET_POS_EXP2
            for lemma in [
                wn.morphy(
                    word,
                    pos=pos,
                )
            ]
            if lemma
        }
    )

    morphy_has_synset = any(
        bool(
            wn.synsets(
                lemma
            )
        )
        for lemma in morphy_lemmas
    )

    keep_conceptual = bool(
        direct_synsets
        or morphy_has_synset
    )

    conceptual_rows_exp2.append(
        {
            "word":
                word,

            "keep_conceptual":
                keep_conceptual,

            "direct_wordnet_synset":
                bool(
                    direct_synsets
                ),

            "morphy_has_synset":
                bool(
                    morphy_has_synset
                ),

            "morphy_lemmas":
                " | ".join(
                    morphy_lemmas
                ),

            "reason":
                (
                    "wordnet_direct_or_morphy"
                    if keep_conceptual
                    else "not_recognized_by_wordnet"
                ),
        }
    )


conceptual_vocabulary_exp2_df = pd.DataFrame(
    conceptual_rows_exp2
)

if len(
    conceptual_vocabulary_exp2_df
) != EXPECTED_MASTER_N_WORDS:
    raise RuntimeError(
        "La auditoría WordNet no cubrió las 103.029 palabras."
    )

if conceptual_vocabulary_exp2_df[
    "word"
].duplicated().any():
    raise RuntimeError(
        "La auditoría WordNet contiene palabras duplicadas."
    )

conceptual_kept_exp2_df = (
    conceptual_vocabulary_exp2_df.loc[
        conceptual_vocabulary_exp2_df[
            "keep_conceptual"
        ]
    ]
    .copy()
    .reset_index(
        drop=True
    )
)

conceptual_words_exp2 = (
    conceptual_kept_exp2_df[
        "word"
    ]
    .tolist()
)

if len(
    conceptual_words_exp2
) != EXPECTED_EXP2_N_WORDS:
    raise RuntimeError(
        "El filtro WordNet produjo un número diferente al "
        f"dataset congelado: {len(conceptual_words_exp2):,} != "
        f"{EXPECTED_EXP2_N_WORDS:,}. "
        "No se persistirá un input distinto."
    )

missing_category_words_exp2 = [
    word
    for word in CATEGORY_WORDS_EXP2
    if word not in set(
        conceptual_words_exp2
    )
]

if missing_category_words_exp2:
    raise RuntimeError(
        "El filtro WordNet eliminó CategoryWords: "
        f"{missing_category_words_exp2}"
    )

contextual_exp2_embeddings_df = (
    contextual_prototypes_all_df.loc[
        conceptual_words_exp2
    ]
    .copy(deep=True)
)

contextual_exp2_embeddings_df.index = pd.Index(
    [
        str(word)
        for word
        in contextual_exp2_embeddings_df.index
    ],
    name="word",
)

contextual_exp2_embeddings_df = (
    contextual_exp2_embeddings_df
    .sort_index()
)

X_exp2 = (
    contextual_exp2_embeddings_df
    .to_numpy(
        dtype=np.float64,
        copy=False,
    )
)

if contextual_exp2_embeddings_df.shape != (
    EXPECTED_EXP2_N_WORDS,
    EXPECTED_EMBEDDING_DIM,
):
    raise RuntimeError(
        "Shape final inesperado: "
        f"{contextual_exp2_embeddings_df.shape}"
    )

if not np.isfinite(
    X_exp2
).all():
    raise RuntimeError(
        "El embedding contextual Exp2 contiene NaN/Inf."
    )

exp2_norms = np.linalg.norm(
    X_exp2,
    axis=1,
)

if np.any(
    exp2_norms <= 0.0
):
    raise RuntimeError(
        "El embedding contextual Exp2 contiene vectores cero."
    )


# ------------------------------------------------------------
# Fingerprint del vocabulario
# ------------------------------------------------------------

vocabulary_payload_exp2 = json.dumps(
    [
        str(word)
        for word
        in contextual_exp2_embeddings_df.index
    ],
    ensure_ascii=False,
    separators=(
        ",",
        ":",
    ),
)

observed_vocabulary_sha256 = hashlib.sha256(
    vocabulary_payload_exp2.encode(
        "utf-8"
    )
).hexdigest()

if (
    observed_vocabulary_sha256
    != EXPECTED_CONTEXTUAL_VOCABULARY_SHA256
):
    raise RuntimeError(
        "V_conceptual no coincide con el vocabulario congelado.\n"
        f"Observado: {observed_vocabulary_sha256}\n"
        f"Esperado:  {EXPECTED_CONTEXTUAL_VOCABULARY_SHA256}\n"
        "No se persistirá un dataset distinto."
    )


# ------------------------------------------------------------
# Fingerprint científico del embedding
# ------------------------------------------------------------

embedding_hasher_exp2 = hashlib.sha256()

for word in contextual_exp2_embeddings_df.index:

    embedding_hasher_exp2.update(
        str(word).encode(
            "utf-8"
        )
    )

    embedding_hasher_exp2.update(
        b"\0"
    )

X_exp2_le = np.ascontiguousarray(
    X_exp2,
    dtype="<f8",
)

embedding_hasher_exp2.update(
    X_exp2_le.tobytes(
        order="C"
    )
)

observed_embedding_sha256 = (
    embedding_hasher_exp2.hexdigest()
)

if (
    observed_embedding_sha256
    != EXPECTED_CONTEXTUAL_EMBEDDING_SHA256
):
    raise RuntimeError(
        "Los prototipos reconstruidos no coinciden con el "
        "embedding contextual congelado.\n"
        f"Observado: {observed_embedding_sha256}\n"
        f"Esperado:  {EXPECTED_CONTEXTUAL_EMBEDDING_SHA256}\n"
        "No se persistirá un dataset distinto."
    )

print(
    "V_conceptual reconstruido:",
    contextual_exp2_embeddings_df.shape,
)

print(
    "Vocabulary SHA256:",
    observed_vocabulary_sha256,
)

print(
    "Embedding SHA256:",
    observed_embedding_sha256,
)

In [ ]:
# ============================================================
# 22. Persistir y validar el input definitivo usado por Exp2
# ============================================================
#
# Esta celda escribe exactamente las rutas consumidas por:
#
#   Experimento_2_Hopfield_vs_HFY_UNIFICADO.ipynb
#
# El módulo exp2_pipeline_integrated.py recibe posteriormente este
# DataFrame desde el notebook; no necesita conocer cómo fue generado.
# ============================================================


# ------------------------------------------------------------
# 1. Persistir embedding y vocabulario
# ------------------------------------------------------------

with open(
    EXP2_CONTEXTUAL_PKL_PATH,
    "wb",
) as handle:

    pickle.dump(
        contextual_exp2_embeddings_df,
        handle,
        protocol=pickle.HIGHEST_PROTOCOL,
    )


contextual_exp2_embeddings_df.to_parquet(
    EXP2_CONTEXTUAL_PARQUET_PATH,
    index=True,
)


pd.DataFrame(
    {
        "word":
            contextual_exp2_embeddings_df.index.tolist()
    }
).to_csv(
    EXP2_CONTEXTUAL_VOCAB_PATH,
    index=False,
    encoding="utf-8",
)


conceptual_vocabulary_exp2_df.to_csv(
    EXP2_CONTEXTUAL_FILTER_AUDIT_PATH,
    index=False,
    encoding="utf-8",
)


# ------------------------------------------------------------
# 2. Manifest metodológico compatible con el notebook unificado
# ------------------------------------------------------------

exp2_input_manifest = {

    "status":
        "EXP2_CONTEXTUAL_INPUT_FINALIZED",

    "embedding_name":
        CONTEXTUAL_EXP2_EMBEDDING_NAME,

    "representation": {

        "source_model":
            MODEL_NAME_EXP2_REBUILD,

        "embedding_dim":
            EXPECTED_EMBEDDING_DIM,

        "occurrences_per_word":
            EXPECTED_OCCURRENCES_PER_WORD,

        "prototype_definition":
            "p_w = (1/100) * sum_i x_raw[w,i]",

        "l2_before_mean":
            False,

        "l2_after_mean":
            False,

        "clustering":
            False,

        "pseudo_senses":
            False,
    },

    "master_dataset": {

        "source_parquet":
            str(
                SOURCE_CONTEXTUAL_PARQUET_PATH
            ),

        "n_words":
            EXPECTED_MASTER_N_WORDS,

        "n_rows":
            (
                EXPECTED_MASTER_N_WORDS
                * EXPECTED_OCCURRENCES_PER_WORD
            ),

        "selection_sha256":
            EXPECTED_MASTER_SELECTION_SHA256,

        "schema_version":
            SOURCE_CONTEXTUAL_SCHEMA_VERSION,
    },

    "conceptual_vocabulary": {

        "method":
            CONCEPTUAL_FILTER_METHOD,

        "approved":
            True,

        "n_words":
            int(
                contextual_exp2_embeddings_df.shape[0]
            ),

        "n_rejected":
            int(
                EXPECTED_MASTER_N_WORDS
                - contextual_exp2_embeddings_df.shape[0]
            ),

        "vocabulary_sha256":
            observed_vocabulary_sha256,

        "master_vocabulary_modified":
            False,

        "master_prototypes_modified":
            False,
    },

    "category_words": {

        "words":
            CATEGORY_WORDS_EXP2,

        "n":
            len(
                CATEGORY_WORDS_EXP2
            ),

        "all_present":
            bool(
                all(
                    word
                    in contextual_exp2_embeddings_df.index
                    for word
                    in CATEGORY_WORDS_EXP2
                )
            ),
    },

    "embedding_identity": {

        "scientific_sha256":
            observed_embedding_sha256,

        "n_words":
            int(
                contextual_exp2_embeddings_df.shape[0]
            ),

        "embedding_dim":
            int(
                contextual_exp2_embeddings_df.shape[1]
            ),

        "dtype_for_fingerprint":
            "float64_little_endian",

        "norm_min":
            float(
                exp2_norms.min()
            ),

        "norm_mean":
            float(
                exp2_norms.mean()
            ),

        "norm_median":
            float(
                np.median(
                    exp2_norms
                )
            ),

        "norm_max":
            float(
                exp2_norms.max()
            ),

        "all_finite":
            True,

        "n_zero_norm":
            0,
    },

    "provenance": {

        "generated_directly_by":
            "contextual_def_generation.ipynb",

        "requires_other_notebooks":
            False,

        "source_occurrence_dataset":
            str(
                SOURCE_CONTEXTUAL_PARQUET_PATH
            ),

        "prototype_arithmetic":
            (
                "float32 occurrence embeddings read from parquet; "
                "cast to float64; arithmetic mean over exactly "
                "100 occurrences"
            ),

        "conceptual_filter":
            CONCEPTUAL_FILTER_METHOD,
    },

    "artifacts": {

        "pickle":
            str(
                EXP2_CONTEXTUAL_PKL_PATH
            ),

        "pickle_sha256":
            _exp2_sha256_file(
                EXP2_CONTEXTUAL_PKL_PATH
            ),

        "parquet":
            str(
                EXP2_CONTEXTUAL_PARQUET_PATH
            ),

        "parquet_sha256":
            _exp2_sha256_file(
                EXP2_CONTEXTUAL_PARQUET_PATH
            ),

        "vocabulary_csv":
            str(
                EXP2_CONTEXTUAL_VOCAB_PATH
            ),

        "filter_audit_csv":
            str(
                EXP2_CONTEXTUAL_FILTER_AUDIT_PATH
            ),

        "manifest":
            str(
                EXP2_CONTEXTUAL_MANIFEST_PATH
            ),
    },
}

_exp2_write_json(
    EXP2_CONTEXTUAL_MANIFEST_PATH,
    exp2_input_manifest,
)


# ------------------------------------------------------------
# 3. Round-trip del pickle
# ------------------------------------------------------------

with open(
    EXP2_CONTEXTUAL_PKL_PATH,
    "rb",
) as handle:

    roundtrip_exp2_df = pickle.load(
        handle
    )

if not isinstance(
    roundtrip_exp2_df,
    pd.DataFrame,
):
    raise RuntimeError(
        "El pickle regenerado no contiene un pd.DataFrame."
    )

if roundtrip_exp2_df.shape != (
    EXPECTED_EXP2_N_WORDS,
    EXPECTED_EMBEDDING_DIM,
):
    raise RuntimeError(
        "El pickle regenerado cambió de shape."
    )

if (
    roundtrip_exp2_df.index.tolist()
    != contextual_exp2_embeddings_df.index.tolist()
):
    raise RuntimeError(
        "El pickle regenerado cambió el vocabulario u orden."
    )

if not np.array_equal(
    roundtrip_exp2_df.to_numpy(
        dtype=np.float64,
        copy=False,
    ),
    X_exp2,
):
    raise RuntimeError(
        "El pickle regenerado cambió valores numéricos."
    )

del roundtrip_exp2_df


# ------------------------------------------------------------
# 4. Comprobación FINAL contra el contrato congelado
# ------------------------------------------------------------

with open(
    EXP2_CONTEXTUAL_MANIFEST_PATH,
    "r",
    encoding="utf-8",
) as handle:

    manifest_roundtrip = json.load(
        handle
    )

if manifest_roundtrip.get(
    "status"
) != "EXP2_CONTEXTUAL_INPUT_FINALIZED":
    raise RuntimeError(
        "El manifest regenerado no quedó finalizado."
    )

if (
    manifest_roundtrip
    .get(
        "embedding_identity",
        {},
    )
    .get(
        "scientific_sha256"
    )
    != EXPECTED_CONTEXTUAL_EMBEDDING_SHA256
):
    raise RuntimeError(
        "El manifest contiene un fingerprint científico inesperado."
    )

if (
    manifest_roundtrip
    .get(
        "conceptual_vocabulary",
        {},
    )
    .get(
        "vocabulary_sha256"
    )
    != EXPECTED_CONTEXTUAL_VOCABULARY_SHA256
):
    raise RuntimeError(
        "El manifest contiene un fingerprint de vocabulario inesperado."
    )


regeneration_summary = {

    "status":
        "READY_FOR_EXP2",

    "input_pickle":
        str(
            EXP2_CONTEXTUAL_PKL_PATH
        ),

    "input_manifest":
        str(
            EXP2_CONTEXTUAL_MANIFEST_PATH
        ),

    "shape":
        list(
            contextual_exp2_embeddings_df.shape
        ),

    "n_master_words":
        EXPECTED_MASTER_N_WORDS,

    "n_conceptual_words":
        EXPECTED_EXP2_N_WORDS,

    "occurrences_per_word":
        EXPECTED_OCCURRENCES_PER_WORD,

    "conceptual_filter":
        CONCEPTUAL_FILTER_METHOD,

    "l2_before_mean":
        False,

    "l2_after_mean":
        False,

    "embedding_scientific_sha256":
        observed_embedding_sha256,

    "conceptual_vocabulary_sha256":
        observed_vocabulary_sha256,
}

print(
    json.dumps(
        regeneration_summary,
        ensure_ascii=False,
        indent=2,
        sort_keys=True,
    )
)

